# MDM DJBC — Full Pipeline Notebook (Kelompok 5)

**Anggota Kelompok 5:** Yola Dafwita Chandra, Farkhan Wisnu Wardhono, Vivin Nur Aziza, Beryl Cholif Arrahman Rahardjo, Dananjaya Ricky Setiaji

Notebook ini menggabungkan seluruh script `Source/step01_*.py` s.d. `Source/step11_*.py` menjadi satu
pipeline end-to-end yang bisa dijalankan berurutan dari atas ke bawah:

- **Tahap 0** — Setup, Data Referensi, Generator Identifier & Simulasi Data
- **Tahap 1** — Data Profiling
- **Tahap 2** — Data Cleansing & Standardization
- **Tahap 3** — Duplicate Detection & Matching
- **Tahap 4** — Golden Record & Survivorship
- **Tahap 5** — Data Quality Monitoring
- **Tahap 6** — YData Profiling Dashboard
- **Bonus** — Laporan Akhir & Bahan Presentasi (Final Report Generator)

**Catatan:**
- Jalankan notebook ini dari root direktori project (folder yang berisi `data/`, `reports/`, `docs/`) agar
  path relatif (`data/raw/...`, `reports/...`, dst.) terbaca dengan benar.
- Setiap cell mengikuti struktur file aslinya di `Source/`. Import antar-modul (`from Source.stepXX import ...`)
  dihapus karena seluruh fungsi/variabel yang dibutuhkan sudah ada di namespace global notebook ini setelah
  cell sebelumnya dijalankan.


## Tahap 0 — Setup, Data Referensi, Generator & Simulasi Data

Persiapan lingkungan (instalasi & import library), data referensi DJBC (`docs/01_data_dictionary.md`),
fungsi generator identifier (NIB, NPWP, dll.), dan mesin simulasi data OSS & CEISA berdasarkan
`docs/01_data_dictionary.md` & `docs/02_business_rules.md`.

In [ ]:
# ============================================================
# CELL 1: INSTALL LIBRARY TAMBAHAN
# Jalankan cell ini pertama kali sebelum cell lainnya
# Estimasi waktu: 30-60 detik
# ============================================================

# Install missingno untuk visualisasi missing values
import subprocess
subprocess.check_call(['pip', 'install', 'missingno', '-q'])

# Install faker untuk generate data simulasi
subprocess.check_call(['pip', 'install', 'faker', '-q'])

# Install ydata-profiling untuk laporan profiling Tahap 1 & Tahap 6
subprocess.check_call(['pip', 'install', 'ydata-profiling', '-q'])

# Konfirmasi instalasi berhasil
print('✅ Instalasi library selesai!')
print('Library yang tersedia:')
print('  - pandas       : manipulasi dan analisis data')
print('  - numpy        : komputasi numerik')
print('  - faker        : generate data simulasi realistis')
print('  - matplotlib   : visualisasi dasar')
print('  - seaborn      : visualisasi statistik')
print('  - missingno    : visualisasi missing values')



In [ ]:
# ============================================================
# CELL 2: IMPORT SEMUA LIBRARY
# ============================================================

import pandas as pd               # manipulasi dataframe
import numpy as np                # komputasi numerik
import re                         # regular expression untuk validasi format
import random                     # random number generator
import warnings
warnings.filterwarnings('ignore') # sembunyikan warning yang tidak penting

# Faker untuk generate data simulasi
from faker import Faker
from faker.providers import person, address, company, internet, phone_number

# Visualisasi
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno

# Setting tampilan
pd.set_option('display.max_columns', None)     # tampilkan semua kolom
pd.set_option('display.max_rows', 50)          # max 50 baris ditampilkan
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 desimal
pd.set_option('display.width', 120)

# Setting style visualisasi
sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family']    = 'sans-serif'

# Set seed agar data yang dihasilkan konsisten
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('✅ Semua library berhasil diimport!')
print(f'   Pandas versi  : {pd.__version__}')
print(f'   NumPy versi   : {np.__version__}')



### [LAB 0.1] Data Referensi DJBC

Kode-kode referensi standar DJBC sesuai `docs/01_data_dictionary.md`.

In [ ]:
# [LAB 0.1] DATA REFERENSI DJBC
# Berisi kode-kode referensi standar Direktorat Jenderal Bea dan Cukai (DJBC)
# sesuai dengan docs/01_data_dictionary.md

# Referensi Provinsi Indonesia (34 provinsi)
PROVINSI = {
    '11': 'Aceh',               '12': 'Sumatera Utara',
    '13': 'Sumatera Barat',      '14': 'Riau',
    '15': 'Jambi',               '16': 'Sumatera Selatan',
    '17': 'Bengkulu',            '18': 'Lampung',
    '19': 'Kep. Bangka Belitung','21': 'Kep. Riau',
    '31': 'DKI Jakarta',         '32': 'Jawa Barat',
    '33': 'Jawa Tengah',         '34': 'DI Yogyakarta',
    '35': 'Jawa Timur',          '36': 'Banten',
    '51': 'Bali',                '52': 'Nusa Tenggara Barat',
    '53': 'Nusa Tenggara Timur', '61': 'Kalimantan Barat',
    '62': 'Kalimantan Tengah',   '63': 'Kalimantan Selatan',
    '64': 'Kalimantan Timur',    '65': 'Kalimantan Utara',
    '71': 'Sulawesi Utara',      '72': 'Sulawesi Tengah',
    '73': 'Sulawesi Selatan',    '74': 'Sulawesi Tenggara',
    '75': 'Gorontalo',           '76': 'Sulawesi Barat',
    '81': 'Maluku',              '82': 'Maluku Utara',
    '91': 'Papua Barat',         '94': 'Papua',
}

# Daftar Kantor Pelayanan Bea Cukai (KPPBC)
# KODE_KANTOR yang digunakan dalam OSS & CEISA
KPPBC_LIST = [
    # Kantor Utama / Besar (sudah ada di list asli)
    {'kode': '010100', 'nama': 'KPU Bea dan Cukai Tipe A Tanjung Priok', 'wilayah': '31'},
    {'kode': '040300', 'nama': 'KPPBC TMP B Soekarno-Hatta',           'wilayah': '31'},
    {'kode': '020300', 'nama': 'KPPBC TMP Belawan',                   'wilayah': '12'},
    {'kode': '070100', 'nama': 'KPPBC TMP Tanjung Perak',             'wilayah': '35'},
    {'kode': '050100', 'nama': 'KPPBC TMP Tanjung Emas',              'wilayah': '33'},
    {'kode': '140100', 'nama': 'KPPBC TMP B Makassar',                'wilayah': '73'},
    {'kode': '060100', 'nama': 'KPPBC TMP A Pasuruan',                'wilayah': '35'},
    {'kode': '090100', 'nama': 'KPPBC TMP B Ngurah Rai',              'wilayah': '51'},

    # Tambahan utama & representatif dari berbagai wilayah
    {'kode': '010700', 'nama': 'KPPBC Tipe Madya Pabean Belawan',     'wilayah': '12'},
    {'kode': '010800', 'nama': 'KPPBC Tipe Madya Pabean B Medan',     'wilayah': '12'},
    {'kode': '011200', 'nama': 'KPPBC Tipe Madya Pabean C Kuala Tanjung', 'wilayah': '12'},
    {'kode': '020400', 'nama': 'KPU Bea dan Cukai Tipe B Batam',      'wilayah': '21'},
    {'kode': '020100', 'nama': 'KPPBC Tipe Madya Pabean B Tanjung Balai Karimun', 'wilayah': '21'},
    {'kode': '030100', 'nama': 'KPPBC Tipe Madya Pabean B Palembang', 'wilayah': '16'},
    {'kode': '030700', 'nama': 'KPPBC Tipe Madya Pabean B Bandar Lampung', 'wilayah': '18'},
    {'kode': '050400', 'nama': 'KPPBC Tipe Madya Pabean Merak',       'wilayah': '36'},
    {'kode': '050900', 'nama': 'KPPBC Tipe Madya Pabean A Bekasi',    'wilayah': '32'},
    {'kode': '070500', 'nama': 'KPPBC TMP Juanda',                    'wilayah': '35'},
    {'kode': '080100', 'nama': 'KPPBC TMP Ngurah Rai',                'wilayah': '51'},
    {'kode': '100300', 'nama': 'KPPBC Balikpapan',                    'wilayah': '64'},
    {'kode': '110100', 'nama': 'KPPBC Makassar',                      'wilayah': '73'},
    {'kode': '120300', 'nama': 'KPPBC Sorong',                        'wilayah': '91'},
    {'kode': '040400', 'nama': 'KPPBC Tipe Madya Pabean A Jakarta',   'wilayah': '31'},
    {'kode': '060300', 'nama': 'KPPBC Tipe Madya Cukai Kudus',        'wilayah': '33'},
    {'kode': '071300', 'nama': 'KPPBC Pasuruan',                      'wilayah': '35'},
    {'kode': '090400', 'nama': 'KPPBC Pontianak',                     'wilayah': '61'},
]

# --- POOLS UNTUK SIMULASI (ENUMS) ---

# Status NIB (Sesuai docs/01_data_dictionary.md §1.16)
STATUS_NIB_POOL = ['AKTIF', 'DIBEKUKAN', 'DICABUT']

# Status Badan Hukum (Sesuai docs/01_data_dictionary.md §1.6)
STATUS_BADAN_HUKUM_POOL = ['Berbadan Hukum', 'Belum Berbadan Hukum']

# Status Perseroan (Sesuai docs/01_data_dictionary.md §1.7)
STATUS_PERSEROAN_POOL = ['Aktif', 'Tidak Aktif', 'Dibekukan']

# Jenis API (Angka Pengenal Importir) (Sesuai docs/01_data_dictionary.md §1.14)
JENIS_API_POOL = ['API-U', 'API-P']

# Kategori Pelaku Usaha di CEISA (Sesuai docs/01_data_dictionary.md §2.10)
KATEGORI_CEISA_POOL = ['IMPORTIR', 'EKSPORTIR', 'KEDUA-DUANYA']

# Jenis Badan Usaha (Sesuai docs/01_data_dictionary.md §1.5)
JENIS_PERSEROAN_POOL = ['PT', 'CV', 'Firma', 'Perum', 'UD']

# Flag Fasilitas (Y/N)
FLAG_POOL = ['Y', 'N']

print(f'✅ Lab 0.1: Referensi DJBC Siap (Align dengan Data Dictionary)')
print(f'   - {len(KPPBC_LIST)} Kantor KPPBC terdaftar.')
print(f'   - {len(STATUS_NIB_POOL)} Status NIB didefinisikan.')


### [LAB 0.2] Generator Identifier DJBC

Fungsi pembangkit (_generator_) untuk identifier NIB, NPWP, dan lainnya.

In [ ]:
# [LAB 0.2] GENERATOR IDENTIFIER DJBC
import random
from datetime import datetime, timedelta

def generate_nib_valid():
    """Generate NIB 13 digit numerik sesuai standar OSS"""
    return ''.join([str(random.randint(0, 9)) for _ in range(13)])

def generate_nib_invalid():
    """Generate NIB bermasalah untuk simulasi anomali"""
    error_types = [
        lambda: ''.join([str(random.randint(0, 9)) for _ in range(12)]), # Kurang digit
        lambda: ''.join([str(random.randint(0, 9)) for _ in range(13)]) + 'X', # Ada huruf
        lambda: '0000000000000' # Dummy/Kosong
    ]
    return random.choice(error_types)()

def generate_npwp_valid():
    """Generate NPWP format baku: XX.XXX.XXX.X-XXX.XXX (15 digit)"""
    d = [str(random.randint(0,99)).zfill(2), str(random.randint(0,999)).zfill(3), 
         str(random.randint(0,999)).zfill(3), str(random.randint(0,9)),
         str(random.randint(0,999)).zfill(3), str(random.randint(0,999)).zfill(3)]
    return f"{d[0]}.{d[1]}.{d[2]}.{d[3]}-{d[4]}.{d[5]}"

def generate_npwp_invalid():
    """Generate NPWP kotor (tanpa separator/salah format)"""
    raw_15 = ''.join([str(random.randint(0, 9)) for _ in range(15)])
    error_types = [
        lambda: raw_15, # Tanpa titik/strip (sering di CEISA)
        lambda: f"{raw_15[:9]}", # Digit kurang
        lambda: raw_15.replace('0', 'O').replace('1', 'I'), # Typo karakter mirip
        lambda: f"{raw_15[:2]} {raw_15[2:5]} {raw_15[5:8]}" # Pake spasi
    ]
    return random.choice(error_types)()

def generate_api_valid():
    """Generate Nomor API (10 digit numerik)"""
    return ''.join([str(random.randint(0, 9)) for _ in range(10)])

def generate_niper_valid():
    """Generate Nomor NIPER (10 digit numerik)"""
    return ''.join([str(random.randint(0, 9)) for _ in range(10)])

def generate_date_random(start_year=2020, end_year=2025):
    """Generate tanggal acak dalam format string YYYY-MM-DD"""
    start_date = datetime(start_year, 1, 1)
    end_date = datetime(end_year, 12, 31)
    time_between_dates = end_date - start_date
    days_between_dates = time_between_dates.days
    random_days = random.randrange(days_between_dates)
    return (start_date + timedelta(days=random_days)).strftime('%Y-%m-%d')

def generate_date_recent(max_days_ago=90):
    """Generate tanggal acak dalam N hari terakhir dari hari ini (format YYYY-MM-DD)"""
    days_ago = random.randint(0, max_days_ago)
    return (datetime.now() - timedelta(days=days_ago)).strftime('%Y-%m-%d')


### [LAB 1] Mesin Simulasi Data DJBC

Berdasarkan: `docs/01_data_dictionary.md` & `docs/02_business_rules.md`.

Fungsi utama `run_full_simulation()` membangkitkan data simulasi OSS (NIB) & CEISA, lalu mengekspor
`data/raw/oss_nib_data.csv` dan `data/raw/ceisa_data.csv`.

In [ ]:
# [LAB 1] DJBC DATA SIMULATION ENGINE
# Berdasarkan: docs/01_data_dictionary.md & docs/02_business_rules.md

import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime

# Inisialisasi
fake = Faker('id_ID')
Faker.seed(42)
random.seed(42)

N_MASTER = 5000  # Total perusahaan unik

def run_full_simulation():
    print(f"🚀 Memulai simulasi {N_MASTER} perusahaan (DIRTY MODE)... ")

    # 1. BANGKITKAN MASTER ENTITIES (OSS as Truth) - 16 kolom sesuai data dictionary
    #    KODE_KANTOR ikut dibawa sebagai helper untuk CEISA, di-drop sebelum export OSS
    master_list = []
    for i in range(N_MASTER):
        # 15% NIB Invalid di Master (Validity NIB)
        if random.random() < 0.15:
            nib = generate_nib_invalid()
        else:
            nib = generate_nib_valid()

        npwp = generate_npwp_valid()
        nama = fake.company().upper()
        kppbc = random.choice(KPPBC_LIST)

        flag_impor = random.choice(['Y', 'N'])
        jenis_api = random.choice(JENIS_API_POOL) if flag_impor == 'Y' else ""

        master_list.append({
            'NIB': nib,
            'NPWP_PERSEROAN': npwp,
            'NAMA_PERSEROAN': nama,
            'NAMA_SINGKATAN': nama.split()[0][:5],
            'JENIS_PERSEROAN': random.choice(JENIS_PERSEROAN_POOL),
            'STATUS_BADAN_HUKUM': random.choice(STATUS_BADAN_HUKUM_POOL),
            'STATUS_PERSEROAN': random.choice(STATUS_PERSEROAN_POOL),
            'ALAMAT_PERSEROAN': fake.street_address().upper(),
            'KELURAHAN_PERSEROAN': fake.city().upper(),
            'PERSEROAN_DAERAH_ID': kppbc['wilayah'],
            'KODE_POS_PERSEROAN': fake.postcode(),
            'FLAG_IMPOR': flag_impor,
            'FLAG_EKSPOR': random.choice(['Y', 'N']),
            'JENIS_API': jenis_api,
            'TGL_PERUBAHAN_NIB': generate_date_random(2023, 2025),
            'STATUS_NIB': random.choices(STATUS_NIB_POOL, weights=[80, 10, 10])[0],
            'KODE_KANTOR': kppbc['kode'],
        })

    df_oss = pd.DataFrame(master_list)

    # --- INJEKSI KEKOTORAN GLOBAL (HEAVY ATTACK) ---
    # 1. Completeness: hanya field opsional yang dikosongkan ("Missing Optional Field").
    #    Field wajib (NIB, NPWP, NAMA, STATUS_NIB) dijamin 100% terisi sesuai business_rules §1.
    for col in ['KELURAHAN_PERSEROAN', 'KODE_POS_PERSEROAN', 'NAMA_SINGKATAN']:
        idx_null = df_oss.sample(frac=0.15).index
        df_oss.loc[idx_null, col] = np.nan

    # 2. Stale Data: sebagian record TGL_PERUBAHAN_NIB sangat lama -> target IS_STALE
    idx_stale_date = df_oss.sample(frac=0.03).index
    df_oss.loc[idx_stale_date, 'TGL_PERUBAHAN_NIB'] = [generate_date_random(2014, 2015) for _ in range(len(idx_stale_date))]

    # 3. Uniqueness: tambah 10% record duplikat persis ("Duplicate Entry")
    df_dups = df_oss.sample(frac=0.10)
    df_oss = pd.concat([df_oss, df_dups], ignore_index=True)

    # 2. CREATE CEISA DATASET (Subset & Heavy Anomaly) - 16 kolom sesuai data dictionary
    df_ceisa = df_oss.sample(frac=0.9).copy()

    # Transformasi kolom: OSS -> CEISA
    df_ceisa = df_ceisa.rename(columns={
        'NPWP_PERSEROAN': 'NPWP',
        'NAMA_PERSEROAN': 'NAMA_PERUSAHAAN',
        'ALAMAT_PERSEROAN': 'ALAMAT_PERUSAHAAN',
        'KELURAHAN_PERSEROAN': 'KELURAHAN',
        'PERSEROAN_DAERAH_ID': 'DAERAH_ID',
        'KODE_POS_PERSEROAN': 'KODE_POS',
        'TGL_PERUBAHAN_NIB': 'TGL_TERBIT_NIB'
    })

    df_ceisa['ID_PERUSAHAAN'] = [f"C{str(i).zfill(6)}" for i in range(len(df_ceisa))]
    df_ceisa['NOMOR_TELPON'] = [fake.phone_number() for _ in range(len(df_ceisa))]
    df_ceisa['KATEGORI'] = random.choices(KATEGORI_CEISA_POOL, k=len(df_ceisa))
    df_ceisa['NIPER'] = [generate_niper_valid() if k != 'IMPORTIR' else "" for k in df_ceisa['KATEGORI']]
    df_ceisa['NOMOR_API'] = [generate_api_valid() if k != 'EKSPORTIR' else "" for k in df_ceisa['KATEGORI']]
    # TGL_SYNC_OSS: tanggal sync terakhir dari OSS - dasar HIGH_SYNC_LAG & data mart dedup
    df_ceisa['TGL_SYNC_OSS'] = [generate_date_recent(90) for _ in range(len(df_ceisa))]

    ceisa_cols = ['ID_PERUSAHAAN', 'NIB', 'NPWP', 'NAMA_PERUSAHAAN', 'ALAMAT_PERUSAHAAN',
                  'KELURAHAN', 'DAERAH_ID', 'KODE_POS', 'NOMOR_TELPON', 'KATEGORI',
                  'NIPER', 'NOMOR_API', 'TGL_TERBIT_NIB', 'STATUS_NIB', 'KODE_KANTOR', 'TGL_SYNC_OSS']
    df_ceisa = df_ceisa[ceisa_cols]

    # 3. INJEKSI ANOMALI BERAT (tahap0_simulation_faker.md §4)
    print("⚠️ Menyuntikkan anomali data tingkat tinggi...")

    # Anomali: Format NPWP - NPWP kotor masif di CEISA (25%)
    idx_npwp = df_ceisa.sample(frac=0.25).index
    df_ceisa.loc[idx_npwp, 'NPWP'] = [generate_npwp_invalid() for _ in range(len(idx_npwp))]

    # Anomali: Typo Nama - fuzzy name di CEISA (15%)
    idx_fuzzy = df_ceisa.sample(frac=0.15).index
    df_ceisa.loc[idx_fuzzy, 'NAMA_PERUSAHAAN'] = df_ceisa.loc[idx_fuzzy, 'NAMA_PERUSAHAAN'].apply(lambda x: str(x).replace("PT ", "") + " (CABANG)")

    # Anomali: Logical Conflict - NIPER (CEISA) terisi padahal FLAG_EKSPOR (OSS) = 'N'
    idx_logic = df_ceisa.sample(frac=0.15).index
    df_ceisa.loc[idx_logic, 'NIPER'] = "4803163678"  # paksa terisi
    df_oss.loc[df_oss.index.isin(idx_logic), 'FLAG_EKSPOR'] = 'N'

    # Anomali: Konflik Status - STATUS_NIB OSS != CEISA -> target IS_OUT_OF_SYNC
    idx_conflict = df_ceisa.sample(n=200).index
    df_ceisa.loc[idx_conflict, 'STATUS_NIB'] = 'AKTIF'
    df_oss.loc[df_oss.index.isin(idx_conflict), 'STATUS_NIB'] = 'DICABUT'

    # Anomali: Data Mart Snapshot Duplicate - NIB sama, snapshot lama dgn NAMA & TGL_SYNC_OSS beda
    idx_snapshot = df_ceisa.sample(frac=0.05).index
    df_snapshot_old = df_ceisa.loc[idx_snapshot].copy()
    df_snapshot_old['NAMA_PERUSAHAAN'] = df_snapshot_old['NAMA_PERUSAHAAN'] + " (OLD)"
    df_snapshot_old['TGL_SYNC_OSS'] = [generate_date_random(2023, 2024) for _ in range(len(df_snapshot_old))]
    df_ceisa = pd.concat([df_ceisa, df_snapshot_old], ignore_index=True)

    # 4. EXPORT DATA
    df_oss = df_oss.drop(columns=['KODE_KANTOR'])  # KODE_KANTOR khusus CEISA sesuai data dictionary
    df_oss.to_csv('data/raw/oss_nib_data.csv', index=False)
    df_ceisa.to_csv('data/raw/ceisa_data.csv', index=False)

    print(f"✅ Simulasi Berhasil (DIRTY DATA READY)!")
    print(f"   - OSS: {len(df_oss)} records")
    print(f"   - CEISA: {len(df_ceisa)} records")

if __name__ == "__main__":
    run_full_simulation()


## Tahap 1 — Data Profiling

Berdasarkan: `docs/tahapan/tahap1_data_profiling.md` & `docs/02_business_rules.md` §1, §4.

Profiling lengkap untuk OSS & CEISA (kondisi sebelum cleansing/MDM):
1. Dataset overview (shape, dtype, null, unique)
2. Statistik deskriptif (numerik & kategorik)
3. Missing value analysis (tabel + severity + visualisasi)
4. Duplicate analysis (full duplicate, duplicate by NIB, duplicate by NAMA)
5. DQ scorecard awal (4 dimensi DMBOK)

In [ ]:
# [TAHAP 1] DATA PROFILING
# Berdasarkan: docs/tahapan/tahap1_data_profiling.md & docs/02_business_rules.md §1, §4
#
# Profiling lengkap untuk OSS & CEISA (kondisi sebelum cleansing/MDM):
#   1. Dataset overview (shape, dtype, null, unique)
#   2. Statistik deskriptif (numerik & kategorik)
#   3. Missing value analysis (tabel + severity + visualisasi)
#   4. Duplicate analysis (full duplicate, duplicate by NIB, duplicate by NAMA)
#   5. Format validation (NIB, NPWP, KODE_POS, STATUS_NIB, dst.)
#   6. Baseline Data Quality Score (4 dimensi DMBOK)
#   7. Laporan ydata_profiling -> reports/profiling_before.html

import matplotlib
matplotlib.use('Agg')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import missingno as msno
import re
from datetime import datetime
from ydata_profiling import ProfileReport


NIB_PATTERN = re.compile(r'^\d{13}$')
NPWP_PATTERN = re.compile(r'^\d{2}\.\d{3}\.\d{3}\.\d{1}-\d{3}\.\d{3}$')
KODE_POS_PATTERN = re.compile(r'^\d{5}$')


def _to_digit_str(series):
    """KODE_POS terbaca float64 (mis. 1330.0) - konversi ke string digit tanpa '.0'
    agar leading zero yang hilang akibat tipe numerik tetap terdeteksi sebagai
    format tidak valid."""
    def conv(x):
        if pd.isna(x):
            return None
        if isinstance(x, float) and x.is_integer():
            return str(int(x))
        return str(x)
    return series.apply(conv)


# ───────────────────────── 1. DATASET OVERVIEW ─────────────────────────
def dataset_overview(df, name):
    print(f"\n{'='*65}\n 1. DATASET OVERVIEW — {name}\n{'='*65}")
    print(f"Jumlah baris : {len(df):,}")
    print(f"Jumlah kolom : {df.shape[1]}")

    overview = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'n_null': df.isnull().sum(),
        'pct_null': (df.isnull().sum() / len(df) * 100).round(2),
        'n_unique': df.nunique(),
    })
    print("\nRingkasan kolom:")
    print(overview.to_string())
    return overview


# ───────────────────────── 2. STATISTIK DESKRIPTIF ─────────────────────────
def descriptive_stats(df, name):
    print(f"\n{'='*65}\n 2. STATISTIK DESKRIPTIF — {name}\n{'='*65}")

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        print("\n[Kolom Numerik]")
        print(df[num_cols].describe().to_string())

    print("\n[Kolom Kategorik/Enum - Top 5 Value Counts]")
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in cat_cols:
        if df[col].nunique() <= 20:
            print(f"\n{col}:")
            print(df[col].value_counts(dropna=False).head(5).to_string())


# ───────────────────────── 3. MISSING VALUE ANALYSIS ─────────────────────────
def _severity(pct):
    if pct == 0:
        return 'OK'
    if pct <= 5:
        return 'LOW'
    if pct <= 15:
        return 'MEDIUM'
    if pct <= 30:
        return 'HIGH'
    return 'CRITICAL'


def missing_value_analysis(df, name):
    print(f"\n{'='*65}\n 3. MISSING VALUE ANALYSIS — {name}\n{'='*65}")
    n = len(df)
    n_missing = df.isnull().sum()
    pct_missing = (n_missing / n * 100).round(2)

    table = pd.DataFrame({'n_missing': n_missing, 'pct_missing': pct_missing})
    table['severity'] = table['pct_missing'].apply(_severity)
    table = table[table['n_missing'] > 0].sort_values('pct_missing', ascending=False)

    if table.empty:
        print("Tidak ada missing value pada kolom apapun.")
    else:
        print(table.to_string())

    # Visualisasi: bar chart % missing + missingno matrix
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    pct_missing.sort_values(ascending=False).plot.bar(ax=axes[0], color='#F44336')
    axes[0].set_title(f'% Missing per Kolom — {name}')
    axes[0].set_ylabel('% Missing')
    axes[0].axhline(0, color='black', linewidth=0.5)

    msno.matrix(df, ax=axes[1], sparkline=False)
    axes[1].set_title(f'Missing Value Matrix — {name}')

    plt.tight_layout()
    fname = f'reports/missing_value_{name.lower()}.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'\nChart disimpan: {fname}')
    return table


# ───────────────────────── 4. DUPLICATE ANALYSIS ─────────────────────────
def duplicate_analysis(df, name, nama_col):
    print(f"\n{'='*65}\n 4. DUPLICATE ANALYSIS — {name}\n{'='*65}")
    n = len(df)

    full_dup = df.duplicated(keep=False)
    print(f"Full duplicate (semua kolom identik) : {full_dup.sum():,} baris")

    nib_dup = df['NIB'].duplicated(keep=False)
    print(f"Duplicate by NIB                     : {nib_dup.sum():,} baris "
          f"({nib_dup.sum() / n * 100:.2f}%)")

    if name == 'OSS':
        if nib_dup.sum() > 0:
            print("  -> ANOMALI: NIB seharusnya unik di OSS (lihat business rules dimensi 'Unik').")
        else:
            print("  -> OK: NIB unik di OSS, sesuai harapan (sumber legalitas/master).")
    else:
        # CEISA bersifat data mart - duplikat NIB expected, dipisah jadi:
        # - Data Mart Snapshot Duplicate: NIB sama, baris berbeda (snapshot historis)
        # - Duplicate Entry: baris benar-benar identik (anomali)
        snapshot_dup = nib_dup & ~full_dup
        print(f"  -> Data Mart Snapshot Duplicate (NIB sama, isi beda) : {snapshot_dup.sum():,} baris "
              f"(EXPECTED, lihat business rules anomali #7)")
        print(f"  -> Duplicate Entry (baris identik)                   : {full_dup.sum():,} baris "
              f"(ANOMALI, lihat business rules anomali #6)")

    nama_dup = df[nama_col].duplicated(keep=False)
    print(f"Duplicate by {nama_col:<22}: {nama_dup.sum():,} baris ({nama_dup.sum() / n * 100:.2f}%)")

    return {
        'full_duplicate': int(full_dup.sum()),
        'duplicate_by_nib': int(nib_dup.sum()),
        f'duplicate_by_{nama_col}': int(nama_dup.sum()),
    }


# ───────────────────────── 5. FORMAT VALIDATION ─────────────────────────
def format_validation(df, name):
    print(f"\n{'='*65}\n 5. FORMAT VALIDATION — {name}\n{'='*65}")
    n = len(df)
    checks = {}

    nib_valid = df['NIB'].astype(str).str.match(NIB_PATTERN).sum()
    checks['NIB (13 digit)'] = (int(nib_valid), n)

    npwp_col = 'NPWP_PERSEROAN' if name == 'OSS' else 'NPWP'
    npwp_valid = df[npwp_col].astype(str).str.match(NPWP_PATTERN).sum()
    checks[f'{npwp_col} (format XX.XXX.XXX.X-XXX.XXX)'] = (int(npwp_valid), n)

    kp_col = 'KODE_POS_PERSEROAN' if name == 'OSS' else 'KODE_POS'
    kp_series = _to_digit_str(df[kp_col])
    kp_notna = int(kp_series.notna().sum())
    kp_valid = int(kp_series.dropna().str.match(KODE_POS_PATTERN).sum())
    checks[f'{kp_col} (5 digit, dari yg terisi)'] = (kp_valid, kp_notna)

    status_valid = df['STATUS_NIB'].isin(STATUS_NIB_POOL).sum()
    checks['STATUS_NIB (enum AKTIF/DIBEKUKAN/DICABUT)'] = (int(status_valid), n)

    if name == 'OSS':
        jp_valid = df['JENIS_PERSEROAN'].isin(JENIS_PERSEROAN_POOL).sum()
        checks['JENIS_PERSEROAN (enum valid)'] = (int(jp_valid), n)
    else:
        kat_valid = df['KATEGORI'].isin(KATEGORI_CEISA_POOL).sum()
        checks['KATEGORI (enum valid)'] = (int(kat_valid), n)

    for label, (valid, total) in checks.items():
        pct = valid / total * 100 if total else 0.0
        print(f"  {label:<42}: {valid:>5,}/{total:<5,} valid ({pct:6.2f}%)")

    return checks


# ───────────────────────── 6. BASELINE DQ SCORE (4 DIMENSI DMBOK) ─────────────────────────
def calculate_dq_metrics(df, dataset_name="OSS"):
    """
    Menghitung skor kualitas data berdasarkan 4 dimensi DMBOK
    (Kelengkapan, Validitas, Unik, Ketepatan Waktu) sesuai docs/02_business_rules.md §1.
    """
    n_rows = len(df)
    results = {}

    # 1. KELENGKAPAN (Completeness) - field wajib: NIB, NPWP, NAMA, STATUS_NIB
    mand_cols = ['NIB', 'STATUS_NIB']
    mand_cols += ['NPWP_PERSEROAN', 'NAMA_PERSEROAN'] if dataset_name == "OSS" else ['NPWP', 'NAMA_PERUSAHAAN']
    comp_score = (1 - df[mand_cols].isnull().any(axis=1).sum() / n_rows) * 100
    results['Kelengkapan'] = comp_score

    # 2. VALIDITAS (Validity) - format NIB (13 digit) & NPWP (XX.XXX.XXX.X-XXX.XXX)
    v_nib = df['NIB'].astype(str).str.match(NIB_PATTERN).mean() * 100
    npwp_col = 'NPWP_PERSEROAN' if dataset_name == "OSS" else 'NPWP'
    v_npwp = df[npwp_col].astype(str).str.match(NPWP_PATTERN).mean() * 100
    results['Validitas'] = (v_nib + v_npwp) / 2

    # 3. UNIK (Uniqueness) - duplikat NIB
    uniq_score = (df['NIB'].nunique() / n_rows) * 100
    results['Unik'] = uniq_score

    # 4. KETEPATAN WAKTU (Timeliness)
    #    OSS  -> IS_STALE: TGL_PERUBAHAN_NIB > 1 tahun dari sekarang
    #    CEISA -> HIGH_SYNC_LAG: TGL_SYNC_OSS > 30 hari dari sekarang
    if dataset_name == "OSS":
        tgl = pd.to_datetime(df['TGL_PERUBAHAN_NIB'], errors='coerce')
        is_stale = (datetime.now() - tgl).dt.days > 365
        timeliness_score = (1 - is_stale.sum() / n_rows) * 100
    else:
        tgl_sync = pd.to_datetime(df['TGL_SYNC_OSS'], errors='coerce')
        high_sync_lag = (datetime.now() - tgl_sync).dt.days > 30
        timeliness_score = (1 - high_sync_lag.sum() / n_rows) * 100
    results['Ketepatan Waktu'] = timeliness_score

    return results


def generate_professional_scorecard(metrics, title_suffix="CEISA"):
    """Visualisasi Dual-Chart: Hexagon Radar & Horizontal Bar."""
    dim_names = list(metrics.keys())
    dim_scores = list(metrics.values())
    total_dq_score = np.mean(dim_scores)

    TARGET_SCORE = 80
    WARNING_SCORE = 70

    if total_dq_score >= TARGET_SCORE:
        grade, grade_color = 'A (Excellent)', '#4CAF50'
    elif total_dq_score >= WARNING_SCORE:
        grade, grade_color = 'B (Good)', '#8BC34A'
    elif total_dq_score >= 60:
        grade, grade_color = 'C (Fair)', '#FF9800'
    else:
        grade, grade_color = 'D (Poor)', '#F44336'

    fig = plt.figure(figsize=(16, 8))
    fig.suptitle(f'Data Quality Scorecard — {title_suffix} (Before MDM)', fontsize=16, fontweight='bold', y=1.05)

    # ── Chart 1: Radar Chart (Hexagon) ──
    angles = np.linspace(0, 2 * np.pi, len(dim_names), endpoint=False).tolist()
    dim_scores_polar = dim_scores + [dim_scores[0]]
    angles += [angles[0]]

    ax1 = plt.subplot(121, polar=True)
    ax1.set_theta_offset(np.pi / 2)
    ax1.set_theta_direction(-1)

    ax1.plot(angles, dim_scores_polar, 'o-', linewidth=3, color='#2196F3', markersize=8)
    ax1.fill(angles, dim_scores_polar, alpha=0.3, color='#2196F3')
    ax1.plot(angles, [TARGET_SCORE] * len(angles), '--', color='red', alpha=0.6, label=f'Target {TARGET_SCORE}%')

    ax1.set_xticks(angles[:-1])
    ax1.set_xticklabels(dim_names, fontsize=10, fontweight='bold')
    ax1.set_ylim(0, 100)
    ax1.set_yticks([20, 40, 60, 80, 100])
    ax1.set_yticklabels(['20', '40', '60', '80', '100%'], fontsize=8)
    ax1.set_title('DQ Score Dimensions (DMBOK)', fontweight='bold', pad=30)
    ax1.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1), fontsize=9)

    for angle, score in zip(angles[:-1], dim_scores):
        ax1.annotate(f'{score:.1f}%', xy=(angle, score), fontsize=9, ha='center',
                      xytext=(0, 10), textcoords='offset points', color='white',
                      fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', fc='#1565C0', alpha=0.8))

    # ── Chart 2: Horizontal Bar Chart ──
    ax2 = plt.subplot(122)
    y_pos = np.arange(len(dim_names))
    colors2 = [grade_color if s >= TARGET_SCORE else '#FF9800' if s >= WARNING_SCORE else '#F44336' for s in dim_scores]

    bars2 = ax2.barh(y_pos, dim_scores, color=colors2, edgecolor='white', height=0.6)
    ax2.axvline(x=TARGET_SCORE, color='red', linestyle='--', linewidth=2, label=f'Target {TARGET_SCORE}%')

    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(dim_names, fontsize=10, fontweight='bold')
    ax2.set_xlim(0, 105)
    ax2.set_xlabel('Score (%)')
    ax2.set_title(f'DQ Score vs Target\n(Avg Score: {total_dq_score:.1f}/100 | Grade: {grade})', fontweight='bold')

    for bar, score in zip(bars2, dim_scores):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f'{score:.1f}%', va='center', fontweight='bold', fontsize=10)

    plt.tight_layout()
    filename = f'reports/dq_scorecard_{title_suffix.lower().replace(" ", "_")}.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Chart disimpan: {filename}')


# ───────────────────────── 7. YDATA PROFILING REPORT ─────────────────────────
# Korelasi kategorik (phi_k/cramers) & interaksi numerik dimatikan karena kolom
# identitas (NIB/NPWP/NAMA/ALAMAT) berkardinalitas sangat tinggi (~hampir unik
# per baris) sehingga perhitungan contingency table-nya sangat lambat & tidak
# informatif untuk profiling ini.
YDATA_CORRELATIONS = {
    "auto": {"calculate": False},
    "pearson": {"calculate": True},
    "spearman": {"calculate": False},
    "kendall": {"calculate": False},
    "phi_k": {"calculate": False},
    "cramers": {"calculate": False},
}


def generate_ydata_report(df, name, output_path):
    print(f"\nMembuat ydata_profiling report untuk {name} -> {output_path} ...")
    profile = ProfileReport(
        df,
        title=f'Data Profiling Report — {name} (Before MDM)',
        explorative=True,
        correlations=YDATA_CORRELATIONS,
        interactions={"continuous": False},
    )
    profile.to_file(output_path)
    print(f"Report disimpan: {output_path}")


def build_profiling_index(summary, output_path='reports/profiling_before.html'):
    """Halaman index ringkas yang menggabungkan hasil profiling OSS & CEISA
    (overview, missing value, duplikat, format validation, baseline DQ score)
    serta link ke laporan ydata_profiling lengkap masing-masing dataset."""

    def fmt_checks(checks):
        rows = ''.join(
            f"<tr><td>{label}</td><td>{valid:,}</td><td>{total:,}</td><td>{valid / total * 100 if total else 0:.2f}%</td></tr>"
            for label, (valid, total) in checks.items()
        )
        return f"<table><tr><th>Check</th><th>Valid</th><th>Total</th><th>%</th></tr>{rows}</table>"

    def fmt_missing(table):
        if table.empty:
            return "<p>Tidak ada missing value.</p>"
        rows = ''.join(
            f"<tr><td>{idx}</td><td>{row.n_missing:,}</td><td>{row.pct_missing:.2f}%</td><td>{row.severity}</td></tr>"
            for idx, row in table.iterrows()
        )
        return f"<table><tr><th>Kolom</th><th>Jumlah Missing</th><th>% Missing</th><th>Severity</th></tr>{rows}</table>"

    def fmt_dq(metrics):
        rows = ''.join(f"<tr><td>{k}</td><td>{v:.2f}%</td></tr>" for k, v in metrics.items())
        avg = np.mean(list(metrics.values()))
        return f"<table><tr><th>Dimensi DMBOK</th><th>Score</th></tr>{rows}<tr><th>Overall</th><th>{avg:.2f}%</th></tr></table>"

    html = f"""<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="utf-8">
<title>Data Profiling Report - Before MDM (Tahap 1)</title>
<style>
  body {{ font-family: Arial, Helvetica, sans-serif; margin: 40px; color: #222; }}
  h1 {{ color: #1565C0; }}
  h2 {{ border-bottom: 2px solid #1565C0; padding-bottom: 4px; margin-top: 40px; }}
  table {{ border-collapse: collapse; margin: 10px 0 20px 0; width: 100%; max-width: 800px; }}
  th, td {{ border: 1px solid #ccc; padding: 6px 10px; text-align: left; font-size: 14px; }}
  th {{ background: #1565C0; color: white; }}
  tr:nth-child(even) {{ background: #f5f5f5; }}
  .summary {{ background: #E3F2FD; padding: 12px 16px; border-radius: 6px; }}
  a.btn {{ display: inline-block; margin: 8px 12px 8px 0; padding: 8px 16px; background: #1565C0;
           color: white; text-decoration: none; border-radius: 4px; }}
</style>
</head>
<body>
<h1>Data Profiling Report — Before MDM (Tahap 1)</h1>
<p class="summary">
  Mini Project Kelompok 5 (DJBC) — Master Data Importir &amp; Eksportir Nasional.<br>
  Profiling dilakukan terhadap data <b>OSS</b> ({summary['n_oss']:,} baris) dan
  <b>CEISA</b> ({summary['n_ceisa']:,} baris) sebelum proses cleansing &amp; Golden Record.
</p>

<h2>Laporan ydata_profiling Lengkap</h2>
<a class="btn" href="profiling_before_oss.html">OSS — Full Profiling Report</a>
<a class="btn" href="profiling_before_ceisa.html">CEISA — Full Profiling Report</a>

<h2>1. Dataset Overview</h2>
<table>
<tr><th>Dataset</th><th>Jumlah Baris</th><th>Jumlah Kolom</th></tr>
<tr><td>OSS</td><td>{summary['n_oss']:,}</td><td>{summary['n_cols_oss']}</td></tr>
<tr><td>CEISA</td><td>{summary['n_ceisa']:,}</td><td>{summary['n_cols_ceisa']}</td></tr>
</table>

<h2>2. Missing Value Analysis</h2>
<h3>OSS</h3>
{fmt_missing(summary['missing_oss'])}
<h3>CEISA</h3>
{fmt_missing(summary['missing_ceisa'])}

<h2>3. Duplicate Analysis</h2>
<table>
<tr><th>Metrik</th><th>OSS</th><th>CEISA</th></tr>
<tr><td>Full duplicate (baris identik)</td><td>{summary['dup_oss']['full_duplicate']:,}</td><td>{summary['dup_ceisa']['full_duplicate']:,}</td></tr>
<tr><td>Duplicate by NIB</td><td>{summary['dup_oss']['duplicate_by_nib']:,}</td><td>{summary['dup_ceisa']['duplicate_by_nib']:,}</td></tr>
</table>
<p><i>Catatan: Duplicate by NIB di OSS adalah anomali (NIB harus unik), sedangkan di CEISA sebagian besar
adalah Data Mart Snapshot Duplicate (expected) — lihat <code>docs/02_business_rules.md</code> Section 5.</i></p>

<h2>4. Format Validation</h2>
<h3>OSS</h3>
{fmt_checks(summary['fmt_oss'])}
<h3>CEISA</h3>
{fmt_checks(summary['fmt_ceisa'])}

<h2>5. Baseline Data Quality Score (4 Dimensi DMBOK)</h2>
<h3>OSS</h3>
{fmt_dq(summary['dq_oss'])}
<h3>CEISA</h3>
{fmt_dq(summary['dq_ceisa'])}

<p><i>Generated by Source/step05_profiling.py</i></p>
</body>
</html>
"""
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)
    print(f"\nProfiling index disimpan: {output_path}")


if __name__ == "__main__":
    df_oss = pd.read_csv('data/raw/oss_nib_data.csv')
    df_ceisa = pd.read_csv('data/raw/ceisa_data.csv')

    # 1. Dataset overview
    dataset_overview(df_oss, 'OSS')
    dataset_overview(df_ceisa, 'CEISA')

    # 2. Statistik deskriptif
    descriptive_stats(df_oss, 'OSS')
    descriptive_stats(df_ceisa, 'CEISA')

    # 3. Missing value analysis
    missing_oss = missing_value_analysis(df_oss, 'OSS')
    missing_ceisa = missing_value_analysis(df_ceisa, 'CEISA')

    # 4. Duplicate analysis
    dup_oss = duplicate_analysis(df_oss, 'OSS', 'NAMA_PERSEROAN')
    dup_ceisa = duplicate_analysis(df_ceisa, 'CEISA', 'NAMA_PERUSAHAAN')

    # 5. Format validation
    fmt_oss = format_validation(df_oss, 'OSS')
    fmt_ceisa = format_validation(df_ceisa, 'CEISA')

    # 6. Baseline Data Quality Score
    print(f"\n{'='*65}\n 6. BASELINE DATA QUALITY SCORE (4 DIMENSI DMBOK)\n{'='*65}")
    oss_metrics = calculate_dq_metrics(df_oss, "OSS")
    ceisa_metrics = calculate_dq_metrics(df_ceisa, "CEISA")
    generate_professional_scorecard(oss_metrics, "OSS Master")
    generate_professional_scorecard(ceisa_metrics, "CEISA Operational")

    print("\n" + "=" * 45)
    print("   FINAL BASELINE DMBOK SUMMARY (4-DIM)")
    print("=" * 45)
    for dim in oss_metrics:
        print(f"  {dim:<18}: OSS {oss_metrics[dim]:6.2f}%  |  CEISA {ceisa_metrics[dim]:6.2f}%")
    print("-" * 45)
    print(f"  {'Overall':<18}: OSS {np.mean(list(oss_metrics.values())):6.2f}%  |  "
          f"CEISA {np.mean(list(ceisa_metrics.values())):6.2f}%")
    print("=" * 45)

    # 7. ydata_profiling reports
    print(f"\n{'='*65}\n 7. YDATA PROFILING REPORT\n{'='*65}")
    generate_ydata_report(df_oss, 'OSS', 'reports/profiling_before_oss.html')
    generate_ydata_report(df_ceisa, 'CEISA', 'reports/profiling_before_ceisa.html')

    summary = {
        'n_oss': len(df_oss), 'n_cols_oss': df_oss.shape[1],
        'n_ceisa': len(df_ceisa), 'n_cols_ceisa': df_ceisa.shape[1],
        'missing_oss': missing_oss, 'missing_ceisa': missing_ceisa,
        'dup_oss': dup_oss, 'dup_ceisa': dup_ceisa,
        'fmt_oss': fmt_oss, 'fmt_ceisa': fmt_ceisa,
        'dq_oss': oss_metrics, 'dq_ceisa': ceisa_metrics,
    }
    build_profiling_index(summary, 'reports/profiling_before.html')

    print("\n✅ Tahap 1 - Data Profiling selesai.")


## Tahap 2 — Data Cleansing & Standardization

Berdasarkan: `docs/tahapan/tahap2_cleansing_standardization.md` & `docs/02_business_rules.md`.

Pipeline:
1. AuditTrail - catat setiap operasi cleansing (apa, berapa baris terdampak)
2. Standardisasi NAMA (uppercase, normalisasi prefix PT/CV/UD/Firma/Perum)
3. Standardisasi ALAMAT (title case + normalisasi singkatan Jl./No./RT/RW/Kec./Kel.)
4. Standardisasi identifier: NIB (13 digit), NPWP (XX.XXX.XXX.X-XXX.XXX), KODE_POS

Output: `data/processed/oss_cleaned.csv`, `data/processed/ceisa_cleaned.csv`,
`data/processed/dataset_clean.csv`, `reports/audit_trail.csv`.

In [ ]:
# [TAHAP 2] DATA CLEANSING & STANDARDIZATION
# Berdasarkan: docs/tahapan/tahap2_cleansing_standardization.md & docs/02_business_rules.md
#
# Pipeline:
#   1. AuditTrail - catat setiap operasi cleansing (apa, berapa baris terdampak)
#   2. Standardisasi NAMA (uppercase, normalisasi prefix PT/CV/UD/Firma/Perum)
#   3. Standardisasi ALAMAT (title case + normalisasi singkatan Jl./No./RT/RW/Kec./Kel.)
#   4. Standardisasi identifier: NIB (13 digit), NPWP (XX.XXX.XXX.X-XXX.XXX),
#      KODE_POS (5 digit), NOMOR_TELPON CEISA (+62...)
#   5. CEISA Data Mart Dedup (1 baris per NIB, snapshot TGL_SYNC_OSS terbaru)
#   6. Missing value handling (wajib: flag saja, opsional: dibiarkan null)
#   7. Validasi referensial PERSEROAN_DAERAH_ID/DAERAH_ID terhadap PROVINSI
#   8. Quality Gate sebelum export
#   9. Export oss_cleaned.csv, ceisa_cleaned.csv, dataset_clean.csv, audit_trail.csv
#  10. Perbandingan skor DQ before vs after (4 dimensi DMBOK)

import pandas as pd
import re
from datetime import datetime


DUMMY_NIB = '0' * 13  # NIB dummy/kosong - sama dengan Source/step07_matching.py

MAND_COLS_OSS = ['NIB', 'NPWP_PERSEROAN', 'NAMA_PERSEROAN', 'STATUS_NIB']
MAND_COLS_CEISA = ['NIB', 'NPWP', 'NAMA_PERUSAHAAN', 'STATUS_NIB']


# ───────────────────────── 1. AUDIT TRAIL ─────────────────────────
class AuditTrail:
    def __init__(self):
        self.entries = []

    def log(self, dataset, operation, field, n_affected, total, description):
        pct = round(n_affected / total * 100, 2) if total else 0.0
        self.entries.append({
            'dataset': dataset,
            'timestamp': datetime.now().isoformat(),
            'operation': operation,
            'field': field,
            'n_affected': int(n_affected),
            'pct_affected': pct,
            'description': description,
        })

    def to_dataframe(self):
        return pd.DataFrame(self.entries)


# ───────────────────────── 2. STANDARDISASI NAMA ─────────────────────────
def standardize_nama(nama):
    """Uppercase, rapikan spasi, normalisasi prefix PT/CV/UD/Firma/Perum."""
    if pd.isna(nama):
        return nama
    s = re.sub(r'\s+', ' ', str(nama).strip().upper())
    prefix_map = {
        r'^P\.?\s*T\.?\s+': 'PT ',
        r'^C\.?\s*V\.?\s+': 'CV ',
        r'^U\.?\s*D\.?\s+': 'UD ',
        r'^FIRMA\.?\s+': 'FIRMA ',
        r'^PERUM\.?\s+': 'PERUM ',
    }
    for pattern, repl in prefix_map.items():
        s = re.sub(pattern, repl, s)
    return s


# ───────────────────────── 3. STANDARDISASI ALAMAT ─────────────────────────
_ALAMAT_ABBR = {
    r'\bJl\b\.?': 'Jl.',
    r'\bGg\b\.?': 'Gg.',
    r'\bNo\b\.?': 'No.',
    r'\bRt\b\.?': 'RT',
    r'\bRw\b\.?': 'RW',
    r'\bKec\b\.?': 'Kec.',
    r'\bKel\b\.?': 'Kel.',
    r'\bDr\b\.?': 'Dr.',
}


def standardize_alamat(alamat):
    """Title case + normalisasi singkatan (Jl., No., RT/RW, Kec., Kel.)."""
    if pd.isna(alamat):
        return alamat
    s = re.sub(r'\s+', ' ', str(alamat).strip()).title()
    for pattern, repl in _ALAMAT_ABBR.items():
        s = re.sub(pattern, repl, s)
    return s


# ───────────────────────── 4. STANDARDISASI IDENTIFIER ─────────────────────────
def standardize_nib(nib):
    """Hanya digit, pad/trim ke 13 digit."""
    if pd.isna(nib):
        return nib
    digits = re.sub(r'\D', '', str(nib))
    if len(digits) > 13:
        digits = digits[:13]
    elif len(digits) < 13:
        digits = digits.zfill(13)
    return digits


def standardize_npwp(npwp):
    """Format ke XX.XXX.XXX.X-XXX.XXX jika 15 digit; selain itu tetap 'kotor'
    (akan ditandai IS_NPWP_INVALID, sesuai anomali 'Inconsistent NPWP')."""
    if pd.isna(npwp):
        return npwp
    digits = re.sub(r'\D', '', str(npwp))
    if len(digits) == 15:
        return f"{digits[0:2]}.{digits[2:5]}.{digits[5:8]}.{digits[8]}-{digits[9:12]}.{digits[12:15]}"
    return digits


def standardize_kode_pos(kode_pos):
    """Pastikan KODE_POS 5 digit; selain itu dikosongkan (field opsional).
    Catatan: KODE_POS terbaca float64 sehingga leading zero (mis. '08123'
    -> 8123.0) sudah hilang sejak data mentah - nilai seperti ini tidak
    bisa dipastikan 5 digit sehingga ikut dikosongkan."""
    if pd.isna(kode_pos):
        return None
    val = kode_pos
    if isinstance(val, float) and val.is_integer():
        val = int(val)
    digits = re.sub(r'\D', '', str(val))
    if len(digits) == 5:
        return int(digits)
    return None


def standardize_telpon(telp):
    """Normalisasi nomor telepon CEISA ke format +62... (hapus separator,
    kode negara/awalan 0 ganda, dan leading zero pada nomor lokal)."""
    if pd.isna(telp):
        return telp
    digits = re.sub(r'\D', '', str(telp))
    if digits.startswith('62'):
        digits = digits[2:]
    digits = digits.lstrip('0')
    return '+62' + digits


def _kode_pos_changed(before_series, after_series):
    """Hitung baris yang berubah (dikosongkan / nilai berbeda) akibat
    standardisasi KODE_POS."""
    n_changed = 0
    for b, a in zip(before_series, after_series):
        b_blank, a_blank = pd.isna(b), pd.isna(a)
        if b_blank != a_blank:
            n_changed += 1
        elif not b_blank and not a_blank and int(b) != int(a):
            n_changed += 1
    return n_changed


# ───────────────────────── 5. CEISA DATA MART DEDUP ─────────────────────────
def dedup_ceisa(df, audit):
    """1 baris per NIB, sisakan snapshot dengan TGL_SYNC_OSS terbaru.
    NIB dummy ('0'*13) merepresentasikan banyak entitas berbeda yang belum
    punya NIB valid (lihat Source/step07_matching.py: DUMMY_NIB dikecualikan
    dari exact-match), sehingga untuk baris dummy dedup dilakukan per
    ID_PERUSAHAAN (bukan per NIB) agar snapshot duplikat tetap tereliminasi
    tanpa menghapus entitas berbeda."""
    n_before = len(df)
    df = df.copy()
    df['_TGL_SYNC_SORT'] = pd.to_datetime(df['TGL_SYNC_OSS'], errors='coerce')

    is_dummy = df['NIB'] == DUMMY_NIB
    df_real = df[~is_dummy].sort_values('_TGL_SYNC_SORT', ascending=False)
    df_dummy = df[is_dummy].sort_values('_TGL_SYNC_SORT', ascending=False)

    n_multi_nib = (df_real['NIB'].value_counts() > 1).sum()
    df_real_dedup = df_real.drop_duplicates(subset='NIB', keep='first')

    n_multi_id_dummy = (df_dummy['ID_PERUSAHAAN'].value_counts() > 1).sum()
    df_dummy_dedup = df_dummy.drop_duplicates(subset='ID_PERUSAHAAN', keep='first')

    result = pd.concat([df_real_dedup, df_dummy_dedup], ignore_index=True).drop(columns='_TGL_SYNC_SORT')
    n_removed = n_before - len(result)

    audit.log('CEISA', 'DEDUP_SNAPSHOT', 'NIB', n_removed, n_before,
              f"{n_multi_nib} NIB (non-dummy) punya >1 snapshot data mart - disisakan baris "
              f"TGL_SYNC_OSS terbaru; {n_multi_id_dummy} ID_PERUSAHAAN ber-NIB dummy "
              f"({DUMMY_NIB}) juga di-dedup agar ID_PERUSAHAAN tetap unik")
    return result


# ───────────────────────── 8. QUALITY GATE ─────────────────────────
def quality_gate(df, dataset_name, mand_cols):
    print(f"\n--- Quality Gate: {dataset_name} ---")
    issues = []

    nib_invalid = (~df['NIB'].astype(str).str.match(NIB_PATTERN)).sum()
    if nib_invalid:
        issues.append(f"{nib_invalid} NIB tidak 13 digit setelah standardisasi")

    n_incomplete = df[mand_cols].isnull().any(axis=1).sum()
    if n_incomplete:
        issues.append(f"{n_incomplete} baris field wajib masih kosong")

    if dataset_name == 'CEISA':
        non_dummy = df[df['NIB'] != DUMMY_NIB]
        n_dup = non_dummy['NIB'].duplicated().sum()
        if n_dup:
            issues.append(f"{n_dup} NIB (non-dummy) masih duplikat setelah dedup")

    if issues:
        print("STATUS: FAIL")
        for issue in issues:
            print(f"  - {issue}")
    else:
        print("STATUS: PASS - tidak ada isu kritis terdeteksi")

    return len(issues) == 0


# ───────────────────────── 10. PERBANDINGAN DQ SCORE ─────────────────────────
def print_dq_comparison(metrics_before, metrics_after, name):
    print(f"\n--- Perbandingan Skor DQ (4 Dimensi DMBOK) — {name} ---")
    print(f"  {'Dimensi':<20}{'Before':>10}{'After':>10}{'Delta':>10}")
    for dim in metrics_before:
        before, after = metrics_before[dim], metrics_after[dim]
        print(f"  {dim:<20}{before:>9.2f}%{after:>9.2f}%{after - before:>+9.2f}%")
    avg_before = sum(metrics_before.values()) / len(metrics_before)
    avg_after = sum(metrics_after.values()) / len(metrics_after)
    print(f"  {'TOTAL DQ SCORE':<20}{avg_before:>9.2f}%{avg_after:>9.2f}%{avg_after - avg_before:>+9.2f}%")


# ───────────────────────── PIPELINE PER DATASET ─────────────────────────
def clean_oss(df_raw, audit):
    df = df_raw.copy()
    n = len(df)

    before = df['NAMA_PERSEROAN'].copy()
    df['NAMA_PERSEROAN'] = df['NAMA_PERSEROAN'].apply(standardize_nama)
    audit.log('OSS', 'STANDARDIZE_NAME', 'NAMA_PERSEROAN', (df['NAMA_PERSEROAN'] != before).sum(), n,
              "Uppercase, rapikan spasi, normalisasi prefix PT/CV/Firma/Perum/UD")

    before = df['ALAMAT_PERSEROAN'].copy()
    df['ALAMAT_PERSEROAN'] = df['ALAMAT_PERSEROAN'].apply(standardize_alamat)
    audit.log('OSS', 'STANDARDIZE_ADDRESS', 'ALAMAT_PERSEROAN', (df['ALAMAT_PERSEROAN'] != before).sum(), n,
              "Title case, normalisasi singkatan (Jl., No., RT/RW, Kec., Kel.)")

    before = df['NIB'].copy()
    df['NIB'] = df['NIB'].apply(standardize_nib)
    audit.log('OSS', 'STANDARDIZE_NIB', 'NIB', (df['NIB'] != before).sum(), n,
              "Hanya digit, pad/trim ke 13 digit")

    before = df['NPWP_PERSEROAN'].copy()
    df['NPWP_PERSEROAN'] = df['NPWP_PERSEROAN'].apply(standardize_npwp)
    audit.log('OSS', 'STANDARDIZE_NPWP', 'NPWP_PERSEROAN', (df['NPWP_PERSEROAN'] != before).sum(), n,
              "Format ke XX.XXX.XXX.X-XXX.XXX (jika 15 digit)")

    df['IS_NPWP_INVALID'] = ~df['NPWP_PERSEROAN'].astype(str).str.match(NPWP_PATTERN)
    audit.log('OSS', 'FLAG_VALIDITY', 'NPWP_PERSEROAN', df['IS_NPWP_INVALID'].sum(), n,
              "Format NPWP_PERSEROAN tidak sesuai XX.XXX.XXX.X-XXX.XXX - ditandai IS_NPWP_INVALID")

    before = df['KODE_POS_PERSEROAN'].copy()
    df['KODE_POS_PERSEROAN'] = df['KODE_POS_PERSEROAN'].apply(standardize_kode_pos)
    audit.log('OSS', 'STANDARDIZE_KODEPOS', 'KODE_POS_PERSEROAN', _kode_pos_changed(before, df['KODE_POS_PERSEROAN']), n,
              "Pastikan 5 digit, selain itu dikosongkan (field opsional)")

    df['IS_INCOMPLETE'] = df[MAND_COLS_OSS].isnull().any(axis=1)
    audit.log('OSS', 'FLAG_INCOMPLETE', '+'.join(MAND_COLS_OSS), df['IS_INCOMPLETE'].sum(), n,
              "Field wajib kosong - ditandai untuk review, tidak diisi paksa")

    daerah_str = df['PERSEROAN_DAERAH_ID'].apply(lambda x: f'{int(x):02d}' if pd.notna(x) else None)
    df['IS_DAERAH_VALID'] = daerah_str.isin(PROVINSI.keys())
    audit.log('OSS', 'VALIDATE_REF', 'PERSEROAN_DAERAH_ID', (~df['IS_DAERAH_VALID']).sum(), n,
              "Kode wilayah tidak ditemukan di 34 kode referensi PROVINSI")

    tgl = pd.to_datetime(df['TGL_PERUBAHAN_NIB'], errors='coerce')
    df['IS_STALE'] = (datetime.now() - tgl).dt.days > 365
    audit.log('OSS', 'FLAG_TIMELINESS', 'TGL_PERUBAHAN_NIB', df['IS_STALE'].sum(), n,
              "IS_STALE = True jika TGL_PERUBAHAN_NIB > 1 tahun dari sekarang")

    df['IS_CLEANED'] = True
    df['CLEANSING_TIMESTAMP'] = datetime.now().isoformat()
    return df


def clean_ceisa(df_raw, audit):
    df = df_raw.copy()
    n = len(df)

    before = df['NAMA_PERUSAHAAN'].copy()
    df['NAMA_PERUSAHAAN'] = df['NAMA_PERUSAHAAN'].apply(standardize_nama)
    audit.log('CEISA', 'STANDARDIZE_NAME', 'NAMA_PERUSAHAAN', (df['NAMA_PERUSAHAAN'] != before).sum(), n,
              "Uppercase, rapikan spasi, normalisasi prefix PT/CV/Firma/Perum/UD")

    before = df['ALAMAT_PERUSAHAAN'].copy()
    df['ALAMAT_PERUSAHAAN'] = df['ALAMAT_PERUSAHAAN'].apply(standardize_alamat)
    audit.log('CEISA', 'STANDARDIZE_ADDRESS', 'ALAMAT_PERUSAHAAN', (df['ALAMAT_PERUSAHAAN'] != before).sum(), n,
              "Title case, normalisasi singkatan (Jl., No., RT/RW, Kec., Kel.)")

    before = df['NIB'].copy()
    df['NIB'] = df['NIB'].apply(standardize_nib)
    audit.log('CEISA', 'STANDARDIZE_NIB', 'NIB', (df['NIB'] != before).sum(), n,
              "Hanya digit, pad/trim ke 13 digit")

    before = df['NPWP'].copy()
    df['NPWP'] = df['NPWP'].apply(standardize_npwp)
    n_npwp_changed = (df['NPWP'] != before).sum()
    n_still_dirty = (~df['NPWP'].astype(str).str.match(NPWP_PATTERN)).sum()
    audit.log('CEISA', 'STANDARDIZE_NPWP', 'NPWP', n_npwp_changed, n,
              f"Format ke XX.XXX.XXX.X-XXX.XXX (jika 15 digit); {n_still_dirty} masih tidak baku "
              f"(anomali \"Inconsistent NPWP\" dipertahankan sesuai docs/02_business_rules.md §1)")

    df['IS_NPWP_INVALID'] = ~df['NPWP'].astype(str).str.match(NPWP_PATTERN)
    audit.log('CEISA', 'FLAG_VALIDITY', 'NPWP', df['IS_NPWP_INVALID'].sum(), n,
              "Format NPWP tidak sesuai XX.XXX.XXX.X-XXX.XXX - ditandai IS_NPWP_INVALID")

    before = df['KODE_POS'].copy()
    df['KODE_POS'] = df['KODE_POS'].apply(standardize_kode_pos)
    audit.log('CEISA', 'STANDARDIZE_KODEPOS', 'KODE_POS', _kode_pos_changed(before, df['KODE_POS']), n,
              "Pastikan 5 digit, selain itu dikosongkan (field opsional)")

    before = df['NOMOR_TELPON'].copy()
    df['NOMOR_TELPON'] = df['NOMOR_TELPON'].apply(standardize_telpon)
    audit.log('CEISA', 'STANDARDIZE_PHONE', 'NOMOR_TELPON', (df['NOMOR_TELPON'] != before).sum(), n,
              "Normalisasi ke format +62...")

    df = dedup_ceisa(df, audit)
    n = len(df)

    df['IS_INCOMPLETE'] = df[MAND_COLS_CEISA].isnull().any(axis=1)
    audit.log('CEISA', 'FLAG_INCOMPLETE', '+'.join(MAND_COLS_CEISA), df['IS_INCOMPLETE'].sum(), n,
              "Field wajib kosong - ditandai untuk review, tidak diisi paksa")

    daerah_str = df['DAERAH_ID'].apply(lambda x: f'{int(x):02d}' if pd.notna(x) else None)
    df['IS_DAERAH_VALID'] = daerah_str.isin(PROVINSI.keys())
    audit.log('CEISA', 'VALIDATE_REF', 'DAERAH_ID', (~df['IS_DAERAH_VALID']).sum(), n,
              "Kode wilayah tidak ditemukan di 34 kode referensi PROVINSI")

    tgl_sync = pd.to_datetime(df['TGL_SYNC_OSS'], errors='coerce')
    df['HIGH_SYNC_LAG'] = (datetime.now() - tgl_sync).dt.days > 30
    audit.log('CEISA', 'FLAG_TIMELINESS', 'TGL_SYNC_OSS', df['HIGH_SYNC_LAG'].sum(), n,
              "HIGH_SYNC_LAG = True jika TGL_SYNC_OSS > 30 hari dari sekarang")

    df['IS_CLEANED'] = True
    df['CLEANSING_TIMESTAMP'] = datetime.now().isoformat()
    return df


if __name__ == "__main__":
    print(f"\n{'='*65}\n TAHAP 2 - DATA CLEANSING & STANDARDIZATION\n{'='*65}")

    df_oss_raw = pd.read_csv('data/raw/oss_nib_data.csv', dtype={'NIB': str, 'NPWP_PERSEROAN': str})
    df_ceisa_raw = pd.read_csv('data/raw/ceisa_data.csv', dtype={'NIB': str, 'NPWP': str})

    audit = AuditTrail()

    print("\n[1/4] Cleansing OSS ...")
    metrics_oss_before = calculate_dq_metrics(df_oss_raw, dataset_name="OSS")
    df_oss = clean_oss(df_oss_raw, audit)
    metrics_oss_after = calculate_dq_metrics(df_oss, dataset_name="OSS")
    print(f"  OSS: {len(df_oss_raw):,} -> {len(df_oss):,} baris")
    quality_gate(df_oss, 'OSS', MAND_COLS_OSS)
    print_dq_comparison(metrics_oss_before, metrics_oss_after, 'OSS')

    print("\n[2/4] Cleansing CEISA ...")
    metrics_ceisa_before = calculate_dq_metrics(df_ceisa_raw, dataset_name="CEISA")
    df_ceisa = clean_ceisa(df_ceisa_raw, audit)
    metrics_ceisa_after = calculate_dq_metrics(df_ceisa, dataset_name="CEISA")
    print(f"  CEISA: {len(df_ceisa_raw):,} -> {len(df_ceisa):,} baris (dedup data mart)")
    quality_gate(df_ceisa, 'CEISA', MAND_COLS_CEISA)
    print_dq_comparison(metrics_ceisa_before, metrics_ceisa_after, 'CEISA')

    print("\n[3/4] Membentuk dataset_clean.csv (union OSS + CEISA) ...")
    df_clean = pd.concat(
        [df_oss.assign(SOURCE='OSS'), df_ceisa.assign(SOURCE='CEISA')],
        ignore_index=True, sort=False,
    )
    print(f"  dataset_clean.csv: {len(df_clean):,} baris, {df_clean.shape[1]} kolom")

    print("\n[4/4] Menyimpan output ...")
    df_oss.to_csv('data/processed/oss_cleaned.csv', index=False)
    df_ceisa.to_csv('data/processed/ceisa_cleaned.csv', index=False)
    df_clean.to_csv('data/processed/dataset_clean.csv', index=False)
    audit.to_dataframe().to_csv('reports/audit_trail.csv', index=False)
    print("  - data/processed/oss_cleaned.csv")
    print("  - data/processed/ceisa_cleaned.csv")
    print("  - data/processed/dataset_clean.csv")
    print("  - reports/audit_trail.csv")

    print("\n✅ Tahap 2 - Data Cleansing & Standardization selesai.")


## Tahap 3 — Duplicate Detection & Matching

Berdasarkan: `docs/tahapan/tahap3_duplicate_matching.md` & `docs/02_business_rules.md` §2.

Matching utama dilakukan ANTAR DUA DATASET (OSS vs CEISA), bukan dalam satu dataset:
1. Exact match NIB (Prioritas 1)
2. Exact match NPWP (Prioritas 2, untuk sisa yang belum match -> tangkap "NIB Typo")
3. Fuzzy match nama/alamat (Prioritas 3, untuk sisa yang belum match)
4. Duplicate clustering internal per sumber (OSS: Duplicate Entry by NIB,
   CEISA: nama/alamat sangat mirip dengan NIB berbeda)

Output: `reports/candidate_pairs.csv`, `reports/duplicate_cluster.csv`.

In [ ]:
# [TAHAP 3] DUPLICATE DETECTION & MATCHING
# Berdasarkan: docs/tahapan/tahap3_duplicate_matching.md & docs/02_business_rules.md §2
#
# Matching utama dilakukan ANTAR DUA DATASET (OSS vs CEISA), bukan dalam satu dataset:
#   1. Exact match NIB      (Prioritas 1)
#   2. Exact match NPWP     (Prioritas 2, untuk sisa yang belum match -> tangkap "NIB Typo")
#   3. Fuzzy match nama/alamat (Prioritas 3, untuk sisa yang belum match)
#   4. Duplicate clustering internal per sumber (OSS: Duplicate Entry by NIB,
#      CEISA: nama/alamat sangat mirip dengan NIB berbeda)

import pandas as pd
import numpy as np
import re
import recordlinkage
from fuzzywuzzy import fuzz
import jellyfish
import networkx as nx
import matplotlib.pyplot as plt

# ============================================================
# KONFIGURASI THRESHOLD & BOBOT (Langkah 6-7)
# ============================================================

UPPER_THRESHOLD = 0.85   # composite_score >= ini      -> FUZZY_MATCH (kandidat valid)
LOWER_THRESHOLD = 0.70   # composite_score <  ini       -> NON_MATCH (dibuang)
                          # di antara keduanya           -> FUZZY_REVIEW (perlu review manual)

# Bobot composite score: NPWP (jika sebagian mirip) + Nama + Alamat, total = 1.0
WEIGHTS = {'npwp': 0.20, 'nama': 0.50, 'alamat': 0.30}

DUMMY_NIB = '0' * 13  # NIB dummy/kosong hasil generate_nib_invalid() - dikecualikan dari exact match



In [ ]:
# ============================================================
# LANGKAH 2: PERSIAPAN MATCHING KEYS
# ============================================================

def normalize_for_matching(text):
    """Normalisasi agresif untuk matching: uppercase, normalisasi prefix badan usaha, hapus simbol."""
    if pd.isna(text):
        return ''
    s = str(text).upper()
    for prefix in ['PT', 'CV', 'UD', 'FIRMA', 'PERUM']:
        s = re.sub(rf'\b{prefix}\.?\b', prefix, s)
    s = re.sub(r'[^A-Z0-9\s]', '', s)
    s = re.sub(r'\s+', ' ', s)
    return s.strip()


def extract_digits(value):
    """Ekstrak hanya digit dari sebuah nilai (NIB/NPWP) - tanpa peduli separator."""
    if pd.isna(value):
        return ''
    return re.sub(r'\D', '', str(value))


def prepare_matching_keys(df_oss, df_ceisa):
    """Tambahkan kolom nib_digits, npwp_digits, nama_key, alamat_key, region_key."""
    df_oss = df_oss.copy()
    df_ceisa = df_ceisa.copy()

    df_oss['nib_digits'] = df_oss['NIB'].apply(extract_digits)
    df_oss['npwp_digits'] = df_oss['NPWP_PERSEROAN'].apply(extract_digits)
    df_oss['nama_key'] = df_oss['NAMA_PERSEROAN'].apply(normalize_for_matching)
    df_oss['alamat_key'] = df_oss['ALAMAT_PERSEROAN'].apply(normalize_for_matching)
    df_oss['region_key'] = df_oss['PERSEROAN_DAERAH_ID'].astype(str)

    df_ceisa['nib_digits'] = df_ceisa['NIB'].apply(extract_digits)
    df_ceisa['npwp_digits'] = df_ceisa['NPWP'].apply(extract_digits)
    df_ceisa['nama_key'] = df_ceisa['NAMA_PERUSAHAAN'].apply(normalize_for_matching)
    df_ceisa['alamat_key'] = df_ceisa['ALAMAT_PERUSAHAAN'].apply(normalize_for_matching)
    df_ceisa['region_key'] = df_ceisa['DAERAH_ID'].astype(str)

    return df_oss, df_ceisa



In [ ]:
# ============================================================
# LANGKAH 3-4: EXACT MATCHING (NIB -> NPWP)
# ============================================================

def exact_match_nib(df_oss, df_ceisa):
    """Prioritas 1: join OSS-CEISA pada nib_digits (kecuali NIB dummy '0000000000000')."""
    left = df_oss[df_oss['nib_digits'] != DUMMY_NIB]
    right = df_ceisa[df_ceisa['nib_digits'] != DUMMY_NIB]

    merged = left.merge(right, on='nib_digits', suffixes=('_OSS', '_CEISA'))
    merged = merged[merged['nib_digits'] != '']

    return pd.DataFrame({
        'NIB_OSS': merged['NIB_OSS'],
        'ID_PERUSAHAAN_CEISA': merged['ID_PERUSAHAAN'],
        'NIB_CEISA': merged['NIB_CEISA'],
        'match_type': 'EXACT_NIB',
        'similarity_score': 1.0,
    })


def exact_match_npwp(df_oss, df_ceisa):
    """Prioritas 2: untuk sisa yang belum match NIB, join pada npwp_digits.
    Menangkap kasus anomali 'NIB Typo' - NIB beda tapi NPWP sama."""
    merged = df_oss.merge(df_ceisa, on='npwp_digits', suffixes=('_OSS', '_CEISA'))
    merged = merged[merged['npwp_digits'] != '']

    return pd.DataFrame({
        'NIB_OSS': merged['NIB_OSS'],
        'ID_PERUSAHAAN_CEISA': merged['ID_PERUSAHAAN'],
        'NIB_CEISA': merged['NIB_CEISA'],
        'match_type': 'EXACT_NPWP',
        'similarity_score': 1.0,
    })



In [ ]:
# ============================================================
# LANGKAH 5-6: FUZZY MATCHING & COMPOSITE SCORE
# ============================================================

def fuzzy_match(remaining_oss, remaining_ceisa):
    """Prioritas 3: blocking per region_key (recordlinkage), lalu hitung composite
    similarity score (NPWP + Nama + Alamat) untuk setiap kandidat pasang."""
    cols = ['NIB_OSS', 'ID_PERUSAHAAN_CEISA', 'NIB_CEISA', 'nama_oss', 'nama_ceisa',
            'sim_npwp', 'sim_nama', 'sim_alamat', 'composite_score']

    if remaining_oss.empty or remaining_ceisa.empty:
        return pd.DataFrame(columns=cols)

    oss_idx = remaining_oss.reset_index(drop=True)
    ceisa_idx = remaining_ceisa.set_index('ID_PERUSAHAAN')

    indexer = recordlinkage.Index()
    indexer.block(left_on='region_key', right_on='region_key')
    candidate_links = indexer.index(oss_idx, ceisa_idx)

    results = []
    for pos, id_perusahaan in candidate_links:
        r1 = oss_idx.loc[pos]
        r2 = ceisa_idx.loc[id_perusahaan]

        # Similarity nama: Jaro-Winkler vs Token Set Ratio, ambil yang terbaik
        sim_nama = max(
            jellyfish.jaro_winkler_similarity(r1['nama_key'], r2['nama_key']),
            fuzz.token_set_ratio(r1['nama_key'], r2['nama_key']) / 100,
        )

        # Similarity alamat: Token Set Ratio
        sim_alamat = fuzz.token_set_ratio(r1['alamat_key'], r2['alamat_key']) / 100

        # Similarity NPWP: 1.0 jika digit identik, partial ratio jika sebagian mirip
        d1, d2 = r1['npwp_digits'], r2['npwp_digits']
        if d1 and d2:
            sim_npwp = 1.0 if d1 == d2 else fuzz.ratio(d1, d2) / 100
        else:
            sim_npwp = 0.0

        composite = (
            sim_npwp * WEIGHTS['npwp']
            + sim_nama * WEIGHTS['nama']
            + sim_alamat * WEIGHTS['alamat']
        )

        results.append({
            'NIB_OSS': r1['NIB'],
            'ID_PERUSAHAAN_CEISA': id_perusahaan,
            'NIB_CEISA': r2['NIB'],
            'nama_oss': r1['NAMA_PERSEROAN'],
            'nama_ceisa': r2['NAMA_PERUSAHAAN'],
            'sim_npwp': round(sim_npwp, 4),
            'sim_nama': round(sim_nama, 4),
            'sim_alamat': round(sim_alamat, 4),
            'composite_score': round(composite, 4),
        })

    return pd.DataFrame(results, columns=cols)


def classify_pair(score):
    if score >= UPPER_THRESHOLD:
        return 'FUZZY_MATCH'
    elif score >= LOWER_THRESHOLD:
        return 'FUZZY_REVIEW'
    else:
        return 'NON_MATCH'



In [ ]:
# ============================================================
# LANGKAH 7: VISUALISASI DISTRIBUSI & THRESHOLD ANALYSIS
# ============================================================

def plot_score_distribution(df_fuzzy, output_path='reports/composite_score_distribution.png'):
    if df_fuzzy.empty:
        print('   (tidak ada kandidat fuzzy untuk divisualisasikan)')
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Distribusi Composite Similarity Score (OSS vs CEISA)', fontsize=13, fontweight='bold')

    # Histogram
    ax1 = axes[0]
    ax1.hist(df_fuzzy['composite_score'], bins=30, color='#2196F3', edgecolor='white')
    ax1.axvline(LOWER_THRESHOLD, color='orange', linestyle='--', linewidth=2, label=f'Lower {LOWER_THRESHOLD}')
    ax1.axvline(UPPER_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Upper {UPPER_THRESHOLD}')
    ax1.set_xlabel('Composite Score')
    ax1.set_ylabel('Frekuensi')
    ax1.set_title('Histogram Composite Score')
    ax1.legend()

    # CDF
    ax2 = axes[1]
    scores_sorted = np.sort(df_fuzzy['composite_score'])
    cdf = np.arange(1, len(scores_sorted) + 1) / len(scores_sorted)
    ax2.plot(scores_sorted, cdf, color='#4CAF50', linewidth=2)
    ax2.axvline(LOWER_THRESHOLD, color='orange', linestyle='--', linewidth=1.5, label=f'Lower {LOWER_THRESHOLD}')
    ax2.axvline(UPPER_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Upper {UPPER_THRESHOLD}')
    ax2.set_xlabel('Composite Score')
    ax2.set_ylabel('CDF')
    ax2.set_title('Cumulative Distribution')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'   Chart disimpan: {output_path}')


def threshold_analysis(df_fuzzy):
    print('\n   Analisis dampak threshold:')
    print(f'   {"Klasifikasi":<14} {"Jumlah":>8}  {"Persen":>8}')
    print('   ' + '-' * 35)
    if df_fuzzy.empty:
        print('   (tidak ada kandidat fuzzy)')
        return
    counts = df_fuzzy['match_type'].value_counts()
    for klas in ['FUZZY_MATCH', 'FUZZY_REVIEW', 'NON_MATCH']:
        n = counts.get(klas, 0)
        pct = n / len(df_fuzzy) * 100
        print(f'   {klas:<14} {n:>8,}  {pct:>7.1f}%')



In [ ]:
# ============================================================
# LANGKAH 8: MANUAL SPOT-CHECK
# ============================================================

def spot_check(df_fuzzy, n_sample=3):
    print('\n   Manual spot-check per zona:')
    if df_fuzzy.empty:
        print('   (tidak ada kandidat fuzzy)')
        return

    for zona in ['FUZZY_MATCH', 'FUZZY_REVIEW', 'NON_MATCH']:
        sample = df_fuzzy[df_fuzzy['match_type'] == zona].head(n_sample)
        n_total = (df_fuzzy['match_type'] == zona).sum()
        print(f'\n   --- Zona {zona} ({n_total:,} pairs) ---')
        for _, row in sample.iterrows():
            print(f"     [{row['composite_score']:.4f}] OSS: {row['nama_oss']!r}  <->  CEISA: {row['nama_ceisa']!r} "
                  f"(sim_npwp={row['sim_npwp']:.2f}, sim_nama={row['sim_nama']:.2f}, sim_alamat={row['sim_alamat']:.2f})")



In [ ]:
# ============================================================
# LANGKAH 9: DUPLICATE CLUSTERING (INTERNAL PER SUMBER)
# ============================================================

def cluster_oss_duplicates(df_oss):
    """Cluster record OSS dengan nib_digits sama (anomali 'Duplicate Entry')."""
    G = nx.Graph()
    G.add_nodes_from(df_oss.index)

    groups = df_oss[df_oss['nib_digits'] != DUMMY_NIB].groupby('nib_digits').groups
    for nib_digits, idxs in groups.items():
        idxs = list(idxs)
        if len(idxs) < 2:
            continue
        for i in range(len(idxs) - 1):
            G.add_edge(idxs[i], idxs[i + 1])

    components = [c for c in nx.connected_components(G) if len(c) > 1]

    rows = []
    for cid, comp in enumerate(components, 1):
        for idx in comp:
            rows.append({
                'cluster_id': f'OSS_{cid}',
                'source': 'OSS',
                'record_id': idx,
                'NIB': df_oss.loc[idx, 'NIB'],
                'nama': df_oss.loc[idx, 'NAMA_PERSEROAN'],
            })
    return pd.DataFrame(rows, columns=['cluster_id', 'source', 'record_id', 'NIB', 'nama'])


def cluster_ceisa_similar(df_ceisa):
    """Cluster record CEISA dengan nama/alamat sangat mirip TAPI NIB berbeda
    (dedup by NIB sudah selesai di Tahap 2 - fokus ke kandidat lain)."""
    idx_df = df_ceisa.set_index('ID_PERUSAHAAN')

    indexer = recordlinkage.Index()
    indexer.sortedneighbourhood('nama_key', window=5)
    candidate_links = indexer.index(idx_df)

    G = nx.Graph()
    G.add_nodes_from(idx_df.index)

    for id1, id2 in candidate_links:
        r1 = idx_df.loc[id1]
        r2 = idx_df.loc[id2]
        if r1['nib_digits'] == r2['nib_digits']:
            continue  # NIB sama -> sudah ditangani Tahap 2 (Data Mart Dedup)

        sim_nama = fuzz.token_set_ratio(r1['nama_key'], r2['nama_key']) / 100
        sim_alamat = fuzz.token_set_ratio(r1['alamat_key'], r2['alamat_key']) / 100
        composite = sim_nama * 0.6 + sim_alamat * 0.4

        if composite >= UPPER_THRESHOLD:
            G.add_edge(id1, id2)

    components = [c for c in nx.connected_components(G) if len(c) > 1]

    rows = []
    for cid, comp in enumerate(components, 1):
        for record_id in comp:
            rows.append({
                'cluster_id': f'CEISA_{cid}',
                'source': 'CEISA',
                'record_id': record_id,
                'NIB': idx_df.loc[record_id, 'NIB'],
                'nama': idx_df.loc[record_id, 'NAMA_PERUSAHAAN'],
            })
    return pd.DataFrame(rows, columns=['cluster_id', 'source', 'record_id', 'NIB', 'nama'])



In [ ]:
# ============================================================
# MAIN PIPELINE (Langkah 1-10)
# ============================================================

if __name__ == "__main__":
    print('=' * 60)
    print('TAHAP 3: DUPLICATE DETECTION & MATCHING')
    print('=' * 60)

    # Langkah 1: Load data
    # NIB dibaca sebagai string supaya leading zero (mis. NIB dummy "0000000000000") tidak hilang
    df_oss = pd.read_csv('data/processed/oss_cleaned.csv', dtype={'NIB': str, 'NPWP_PERSEROAN': str})
    df_ceisa = pd.read_csv('data/processed/ceisa_cleaned.csv', dtype={'NIB': str, 'NPWP': str})
    print(f'\n1. Load data: OSS={len(df_oss):,} baris, CEISA={len(df_ceisa):,} baris')

    # Langkah 2: Persiapan matching keys
    df_oss, df_ceisa = prepare_matching_keys(df_oss, df_ceisa)
    print('2. Matching keys (nib_digits, npwp_digits, nama_key, alamat_key, region_key) dibuat')

    # Langkah 3: Exact match - NIB
    exact_nib_pairs = exact_match_nib(df_oss, df_ceisa)
    matched_oss_nib = set(exact_nib_pairs['NIB_OSS'])
    matched_ceisa_id = set(exact_nib_pairs['ID_PERUSAHAAN_CEISA'])
    print(f'3. Exact match NIB    : {len(exact_nib_pairs):,} pairs')

    remaining_oss = df_oss[~df_oss['NIB'].isin(matched_oss_nib)].copy()
    remaining_ceisa = df_ceisa[~df_ceisa['ID_PERUSAHAAN'].isin(matched_ceisa_id)].copy()

    # Langkah 4: Exact match - NPWP (untuk sisa)
    exact_npwp_pairs = exact_match_npwp(remaining_oss, remaining_ceisa)
    matched_oss_npwp = set(exact_npwp_pairs['NIB_OSS'])
    matched_ceisa_npwp = set(exact_npwp_pairs['ID_PERUSAHAAN_CEISA'])
    print(f'4. Exact match NPWP   : {len(exact_npwp_pairs):,} pairs (anomali NIB Typo)')

    remaining_oss = remaining_oss[~remaining_oss['NIB'].isin(matched_oss_npwp)].copy()
    remaining_ceisa = remaining_ceisa[~remaining_ceisa['ID_PERUSAHAAN'].isin(matched_ceisa_npwp)].copy()
    print(f'   Sisa belum match    : OSS={len(remaining_oss):,}, CEISA={len(remaining_ceisa):,}')

    # Langkah 5: Fuzzy matching
    print('\n5. Fuzzy matching (blocking per region_key)...')
    df_fuzzy = fuzzy_match(remaining_oss, remaining_ceisa)
    print(f'   Kandidat pairs fuzzy: {len(df_fuzzy):,}')

    # Langkah 6: Composite score + klasifikasi
    if not df_fuzzy.empty:
        df_fuzzy['match_type'] = df_fuzzy['composite_score'].apply(classify_pair)
        df_fuzzy = df_fuzzy.sort_values('composite_score', ascending=False).reset_index(drop=True)
    print('6. Composite score & klasifikasi (FUZZY_MATCH/FUZZY_REVIEW/NON_MATCH) selesai')

    # Langkah 7: Visualisasi & threshold analysis
    print('\n7. Visualisasi distribusi composite score...')
    plot_score_distribution(df_fuzzy)
    threshold_analysis(df_fuzzy)

    # Langkah 8: Manual spot-check
    spot_check(df_fuzzy)

    # Langkah 9: Duplicate clustering internal
    print('\n9. Duplicate clustering internal...')
    oss_clusters = cluster_oss_duplicates(df_oss)
    ceisa_clusters = cluster_ceisa_similar(df_ceisa)
    df_clusters = pd.concat([oss_clusters, ceisa_clusters], ignore_index=True)
    print(f'   OSS   : {oss_clusters["cluster_id"].nunique()} cluster ({len(oss_clusters):,} record)')
    print(f'   CEISA : {ceisa_clusters["cluster_id"].nunique()} cluster ({len(ceisa_clusters):,} record)')

    # Langkah 10: Export
    print('\n10. Export hasil...')
    fuzzy_export = pd.DataFrame(columns=['NIB_OSS', 'ID_PERUSAHAAN_CEISA', 'NIB_CEISA', 'match_type', 'similarity_score'])
    if not df_fuzzy.empty:
        fuzzy_export = (
            df_fuzzy[df_fuzzy['match_type'].isin(['FUZZY_MATCH', 'FUZZY_REVIEW'])]
            [['NIB_OSS', 'ID_PERUSAHAAN_CEISA', 'NIB_CEISA', 'match_type', 'composite_score']]
            .rename(columns={'composite_score': 'similarity_score'})
        )

    pair_frames = [df for df in [exact_nib_pairs, exact_npwp_pairs, fuzzy_export] if not df.empty]
    candidate_pairs = pd.concat(pair_frames, ignore_index=True) if pair_frames else fuzzy_export
    candidate_pairs.to_csv('reports/candidate_pairs.csv', index=False)
    print(f'   reports/candidate_pairs.csv  -> {len(candidate_pairs):,} pairs')

    df_clusters.to_csv('reports/duplicate_cluster.csv', index=False)
    print(f'   reports/duplicate_cluster.csv -> {len(df_clusters):,} records')

    # Orphan records (tidak match sama sekali) - kandidat untuk Tahap 4 SOURCE=OSS_ONLY/CEISA_ONLY
    matched_oss_fuzzy = set(fuzzy_export['NIB_OSS'])
    matched_ceisa_fuzzy = set(fuzzy_export['ID_PERUSAHAAN_CEISA'])
    matched_oss_all = matched_oss_nib | matched_oss_npwp | matched_oss_fuzzy
    matched_ceisa_all = matched_ceisa_id | matched_ceisa_npwp | matched_ceisa_fuzzy

    n_orphan_oss = (~df_oss['NIB'].isin(matched_oss_all)).sum()
    n_orphan_ceisa = (~df_ceisa['ID_PERUSAHAAN'].isin(matched_ceisa_all)).sum()

    print('\n' + '=' * 60)
    print('RINGKASAN')
    print('=' * 60)
    print(f'Total candidate pairs   : {len(candidate_pairs):,}')
    print(f'  - EXACT_NIB           : {len(exact_nib_pairs):,}')
    print(f'  - EXACT_NPWP          : {len(exact_npwp_pairs):,}')
    print(f'  - FUZZY (MATCH/REVIEW): {len(fuzzy_export):,}')
    print(f'Orphan records (OSS_ONLY)  : {n_orphan_oss:,}')
    print(f'Orphan records (CEISA_ONLY): {n_orphan_ceisa:,}')
    print(f'Duplicate clusters total   : {df_clusters["cluster_id"].nunique() if not df_clusters.empty else 0}')
    print('=' * 60)


## Tahap 4 — Golden Record & Survivorship

Berdasarkan: `docs/tahapan/tahap4_golden_record.md` & `docs/02_business_rules.md` §3.

Untuk setiap pasangan match dari Tahap 3 (EXACT_NIB, EXACT_NPWP, FUZZY_MATCH/REVIEW), gabungkan record
OSS + CEISA menjadi satu Golden Record menggunakan survivorship rules (§3.A): Legalitas & Status -> OSS
menang (System of Record), Operasional (KODE_KANTOR, NOMOR_TELPON, dll) -> CEISA menang. Record yang
tidak match (orphan) tetap masuk apa adanya dengan SOURCE=OSS_ONLY / CEISA_ONLY.

Output: `data/golden/golden_record.csv`, `reports/provenance_log.csv`, `reports/conflict_log.csv`.

In [ ]:
# [TAHAP 4] GOLDEN RECORD & SURVIVORSHIP
# Berdasarkan: docs/tahapan/tahap4_golden_record.md & docs/02_business_rules.md §3
#
# Untuk setiap pasangan match dari Tahap 3 (EXACT_NIB, EXACT_NPWP, FUZZY_MATCH/REVIEW),
# gabungkan record OSS + CEISA menjadi satu Golden Record menggunakan survivorship
# rules (§3.A): Legalitas & Status -> OSS menang (System of Record), Operasional
# (KODE_KANTOR, NOMOR_TELPON, dll) -> CEISA menang. Record yang tidak match (orphan)
# tetap masuk apa adanya dengan SOURCE=OSS_ONLY / CEISA_ONLY.

import pandas as pd
import numpy as np
import re
from datetime import datetime
from fuzzywuzzy import fuzz

NPWP_PATTERN = re.compile(r'^\d{2}\.\d{3}\.\d{3}\.\d{1}-\d{3}\.\d{3}$')
NIB_PATTERN = re.compile(r'^\d{13}$')


In [ ]:
# ============================================================
# LANGKAH 2: SURVIVORSHIP RULES (business_rules §3.A)
# ============================================================
# field_golden -> (kolom_OSS, kolom_CEISA, sumber_menang)
#   'OSS'        : field ada di kedua sumber -> OSS menang (Legalitas & Status / System of Record)
#   'OSS_ONLY'   : field hanya tersedia di OSS (legalitas/perizinan)
#   'CEISA_ONLY' : field hanya tersedia di CEISA (operasional, termasuk KODE_KANTOR & NOMOR_TELPON)

FIELD_RULES = {
    'NIB':                ('NIB', 'NIB', 'OSS'),
    'NPWP':               ('NPWP_PERSEROAN', 'NPWP', 'OSS'),
    'NAMA':               ('NAMA_PERSEROAN', 'NAMA_PERUSAHAAN', 'OSS'),
    'NAMA_SINGKATAN':     ('NAMA_SINGKATAN', None, 'OSS_ONLY'),
    'JENIS_PERSEROAN':    ('JENIS_PERSEROAN', None, 'OSS_ONLY'),
    'STATUS_BADAN_HUKUM': ('STATUS_BADAN_HUKUM', None, 'OSS_ONLY'),
    'STATUS_PERSEROAN':   ('STATUS_PERSEROAN', None, 'OSS_ONLY'),
    'ALAMAT':             ('ALAMAT_PERSEROAN', 'ALAMAT_PERUSAHAAN', 'OSS'),
    'KELURAHAN':          ('KELURAHAN_PERSEROAN', 'KELURAHAN', 'OSS'),
    'DAERAH_ID':          ('PERSEROAN_DAERAH_ID', 'DAERAH_ID', 'OSS'),
    'KODE_POS':           ('KODE_POS_PERSEROAN', 'KODE_POS', 'OSS'),
    'FLAG_IMPOR':         ('FLAG_IMPOR', None, 'OSS_ONLY'),
    'FLAG_EKSPOR':        ('FLAG_EKSPOR', None, 'OSS_ONLY'),
    'JENIS_API':          ('JENIS_API', None, 'OSS_ONLY'),
    'KODE_KANTOR':        (None, 'KODE_KANTOR', 'CEISA_ONLY'),
    'NOMOR_TELPON':       (None, 'NOMOR_TELPON', 'CEISA_ONLY'),
    'KATEGORI':           (None, 'KATEGORI', 'CEISA_ONLY'),
    'NIPER':              (None, 'NIPER', 'CEISA_ONLY'),
    'NOMOR_API':          (None, 'NOMOR_API', 'CEISA_ONLY'),
    'ID_PERUSAHAAN':      (None, 'ID_PERUSAHAAN', 'CEISA_ONLY'),
    'TGL_PERUBAHAN_NIB':  ('TGL_PERUBAHAN_NIB', None, 'OSS_ONLY'),
    'TGL_TERBIT_NIB':     (None, 'TGL_TERBIT_NIB', 'CEISA_ONLY'),
    'TGL_SYNC_OSS':       (None, 'TGL_SYNC_OSS', 'CEISA_ONLY'),
    'STATUS_NIB':         ('STATUS_NIB', 'STATUS_NIB', 'OSS'),
}

META_COLS = [
    'SOURCE', 'MATCH_TYPE', 'SOURCE_COUNT', 'N_CONFLICTS',
    'IS_OUT_OF_SYNC', 'IS_STALE', 'HIGH_SYNC_LAG', 'IS_LOGICAL_CONFLICT_NIPER',
    'CREATED_AT',
]
GOLDEN_COLUMNS = ['GR_ID'] + list(FIELD_RULES.keys()) + META_COLS


In [ ]:
# ============================================================
# LANGKAH 3: FUNGSI create_golden_record()
# ============================================================

def create_golden_record(oss_row, ceisa_row, field_rules):
    """Gabungkan satu pasangan (oss_row, ceisa_row) menjadi satu golden record
    berdasarkan field_rules. oss_row/ceisa_row boleh None untuk orphan record."""
    golden = {}
    provenance = {}

    for field, (oss_col, ceisa_col, winner) in field_rules.items():
        oss_val = oss_row[oss_col] if (oss_row is not None and oss_col is not None) else None
        ceisa_val = ceisa_row[ceisa_col] if (ceisa_row is not None and ceisa_col is not None) else None

        if winner == 'CEISA_ONLY':
            if pd.notna(ceisa_val) and str(ceisa_val).strip() != '':
                value, source = ceisa_val, 'CEISA'
            else:
                value, source = None, 'N/A'
        elif winner == 'OSS_ONLY':
            if pd.notna(oss_val) and str(oss_val).strip() != '':
                value, source = oss_val, 'OSS'
            else:
                value, source = None, 'N/A'
        else:  # 'OSS' -> field ada di kedua sumber, OSS menang (System of Record)
            if pd.notna(oss_val) and str(oss_val).strip() != '':
                value, source = oss_val, 'OSS'
            elif pd.notna(ceisa_val) and str(ceisa_val).strip() != '':
                value, source = ceisa_val, 'CEISA'
            else:
                value, source = None, 'N/A'

        golden[field] = value
        provenance[field] = source

    return golden, provenance


def detect_field_conflicts(oss_row, ceisa_row):
    """Bandingkan field yang ada di kedua sumber, kembalikan list konflik
    (field, nilai_oss, nilai_ceisa). Dipakai untuk Langkah 6 - Analisis Pola Konflik."""
    conflicts = []

    # STATUS_NIB - target anomali "Sync Conflict" (tahap0 §4)
    if str(oss_row['STATUS_NIB']).strip() != str(ceisa_row['STATUS_NIB']).strip():
        conflicts.append(('STATUS_NIB', oss_row['STATUS_NIB'], ceisa_row['STATUS_NIB']))

    # NPWP - bedakan konflik NILAI (digit beda) vs konflik FORMAT (digit sama, format beda)
    d_oss, d_ceisa = extract_digits(oss_row['NPWP_PERSEROAN']), extract_digits(ceisa_row['NPWP'])
    if d_oss != d_ceisa:
        conflicts.append(('NPWP_VALUE', oss_row['NPWP_PERSEROAN'], ceisa_row['NPWP']))
    elif str(oss_row['NPWP_PERSEROAN']) != str(ceisa_row['NPWP']):
        conflicts.append(('NPWP_FORMAT', oss_row['NPWP_PERSEROAN'], ceisa_row['NPWP']))

    # NAMA - target anomali "Fuzzy Identity"
    if normalize_for_matching(oss_row['NAMA_PERSEROAN']) != normalize_for_matching(ceisa_row['NAMA_PERUSAHAAN']):
        conflicts.append(('NAMA', oss_row['NAMA_PERSEROAN'], ceisa_row['NAMA_PERUSAHAAN']))

    # ALAMAT
    if normalize_for_matching(oss_row['ALAMAT_PERSEROAN']) != normalize_for_matching(ceisa_row['ALAMAT_PERUSAHAAN']):
        conflicts.append(('ALAMAT', oss_row['ALAMAT_PERSEROAN'], ceisa_row['ALAMAT_PERUSAHAAN']))

    # KELURAHAN, DAERAH_ID, KODE_POS - bandingkan jika kedua sisi terisi
    for oss_col, ceisa_col, field_name in [
        ('KELURAHAN_PERSEROAN', 'KELURAHAN', 'KELURAHAN'),
        ('PERSEROAN_DAERAH_ID', 'DAERAH_ID', 'DAERAH_ID'),
        ('KODE_POS_PERSEROAN', 'KODE_POS', 'KODE_POS'),
    ]:
        v_oss, v_ceisa = oss_row[oss_col], ceisa_row[ceisa_col]
        if pd.notna(v_oss) and pd.notna(v_ceisa) and str(v_oss).strip() != str(v_ceisa).strip():
            conflicts.append((field_name, v_oss, v_ceisa))

    return conflicts


def is_logical_conflict_niper(oss_row, ceisa_row):
    """Anomali 'Logical Conflict': FLAG_EKSPOR (OSS) = 'N' tapi NIPER (CEISA) terisi."""
    niper = ceisa_row['NIPER']
    niper_filled = pd.notna(niper) and str(niper).strip() not in ('', 'nan')
    return bool(oss_row['FLAG_EKSPOR'] == 'N' and niper_filled)


In [ ]:
# ============================================================
# LANGKAH 4: PROSES MATCHED PAIRS (batch generation)
# ============================================================

def lookup_oss_row(oss_idx, nib, ceisa_row):
    """Ambil baris OSS berdasarkan NIB. NIB dummy "0000000000000" (anomali NIB Invalid)
    dimiliki banyak baris berbeda -> disambiguasi dengan kemiripan nama ke baris CEISA pasangannya."""
    candidates = oss_idx.loc[[nib]]
    if len(candidates) == 1:
        return candidates.iloc[0]
    # NIB dummy "0000000000000" -> banyak baris dengan index label sama, gunakan posisi (iloc)
    scores = candidates['NAMA_PERSEROAN'].apply(
        lambda nama: fuzz.token_set_ratio(normalize_for_matching(nama), normalize_for_matching(ceisa_row['NAMA_PERUSAHAAN']))
    )
    return candidates.iloc[scores.values.argmax()]


def build_matched_records(df_oss, df_ceisa, candidate_pairs, field_rules):
    # Dedup residu anomali "Duplicate Entry" - 1 NIB OSS hanya 1 baris untuk lookup
    pairs = candidate_pairs.drop_duplicates(subset='NIB_OSS', keep='first')

    oss_idx = df_oss.set_index('NIB', drop=False)
    ceisa_idx = df_ceisa.set_index('ID_PERUSAHAAN', drop=False)

    golden_rows, provenance_rows, conflict_rows = [], [], []

    matched_oss_row_ids = set()

    for _, pair in pairs.iterrows():
        ceisa_row = ceisa_idx.loc[pair['ID_PERUSAHAAN_CEISA']]
        oss_row = lookup_oss_row(oss_idx, pair['NIB_OSS'], ceisa_row)
        matched_oss_row_ids.add(oss_row['_ROW_ID'])

        golden, provenance = create_golden_record(oss_row, ceisa_row, field_rules)

        is_out_of_sync = str(oss_row['STATUS_NIB']).strip() != str(ceisa_row['STATUS_NIB']).strip()
        is_logic_conflict = is_logical_conflict_niper(oss_row, ceisa_row)
        field_conflicts = detect_field_conflicts(oss_row, ceisa_row)
        n_conflicts = len(field_conflicts) + (1 if is_logic_conflict else 0)

        golden.update({
            'SOURCE': 'MATCHED',
            'MATCH_TYPE': pair['match_type'],
            'SOURCE_COUNT': 2,
            'N_CONFLICTS': n_conflicts,
            'IS_OUT_OF_SYNC': is_out_of_sync,          # §3.B - anomali "Sync Conflict"
            'IS_STALE': bool(oss_row['IS_STALE']),     # dibawa dari Tahap 1/2, tidak dihitung ulang
            'HIGH_SYNC_LAG': bool(ceisa_row['HIGH_SYNC_LAG']),
            'IS_LOGICAL_CONFLICT_NIPER': is_logic_conflict,
            'CREATED_AT': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        })
        golden_rows.append(golden)

        for field, source in provenance.items():
            provenance_rows.append({'NIB': golden['NIB'], 'field': field, 'source': source})

        for field, oss_val, ceisa_val in field_conflicts:
            conflict_rows.append({
                'NIB': golden['NIB'], 'field': field,
                'oss_value': oss_val, 'ceisa_value': ceisa_val, 'winner': 'OSS',
            })
        if is_logic_conflict:
            conflict_rows.append({
                'NIB': golden['NIB'], 'field': 'FLAG_EKSPOR_vs_NIPER',
                'oss_value': oss_row['FLAG_EKSPOR'], 'ceisa_value': ceisa_row['NIPER'], 'winner': 'OSS (FLAG_EKSPOR)',
            })

    matched_ceisa_ids = set(pairs['ID_PERUSAHAAN_CEISA'])
    return golden_rows, provenance_rows, conflict_rows, matched_oss_row_ids, matched_ceisa_ids


In [ ]:
# ============================================================
# LANGKAH 5: PROSES ORPHAN RECORDS (OSS_ONLY / CEISA_ONLY)
# ============================================================

def build_orphan_records(df_oss, df_ceisa, matched_oss_row_ids, matched_ceisa_ids, field_rules):
    golden_rows, provenance_rows = [], []
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    df_oss_orphan = df_oss[~df_oss['_ROW_ID'].isin(matched_oss_row_ids)]
    for _, oss_row in df_oss_orphan.iterrows():
        golden, provenance = create_golden_record(oss_row, None, field_rules)
        golden.update({
            'SOURCE': 'OSS_ONLY', 'MATCH_TYPE': 'N/A', 'SOURCE_COUNT': 1, 'N_CONFLICTS': 0,
            'IS_OUT_OF_SYNC': False, 'IS_STALE': bool(oss_row['IS_STALE']),
            'HIGH_SYNC_LAG': None, 'IS_LOGICAL_CONFLICT_NIPER': False, 'CREATED_AT': timestamp,
        })
        golden_rows.append(golden)
        for field, source in provenance.items():
            provenance_rows.append({'NIB': golden['NIB'], 'field': field, 'source': source})

    df_ceisa_orphan = df_ceisa[~df_ceisa['ID_PERUSAHAAN'].isin(matched_ceisa_ids)]
    for _, ceisa_row in df_ceisa_orphan.iterrows():
        golden, provenance = create_golden_record(None, ceisa_row, field_rules)
        golden.update({
            'SOURCE': 'CEISA_ONLY', 'MATCH_TYPE': 'N/A', 'SOURCE_COUNT': 1, 'N_CONFLICTS': 0,
            'IS_OUT_OF_SYNC': False, 'IS_STALE': None,
            'HIGH_SYNC_LAG': bool(ceisa_row['HIGH_SYNC_LAG']), 'IS_LOGICAL_CONFLICT_NIPER': False, 'CREATED_AT': timestamp,
        })
        golden_rows.append(golden)
        for field, source in provenance.items():
            provenance_rows.append({'NIB': golden['NIB'], 'field': field, 'source': source})

    return golden_rows, provenance_rows


In [ ]:
# ============================================================
# LANGKAH 6: ANALISIS POLA KONFLIK
# ============================================================

def analyze_conflict_patterns(df_conflicts, df_golden):
    n_matched = (df_golden['SOURCE'] == 'MATCHED').sum()
    print(f'\n   Total matched pairs       : {n_matched:,}')
    print(f'   Pairs dengan >=1 konflik   : {(df_golden.loc[df_golden["SOURCE"] == "MATCHED", "N_CONFLICTS"] > 0).sum():,}')

    if df_conflicts.empty:
        print('   (tidak ada konflik field terdeteksi)')
        return

    by_field = df_conflicts['field'].value_counts().reset_index()
    by_field.columns = ['field', 'jumlah_konflik']
    by_field['persen_dari_matched'] = (by_field['jumlah_konflik'] / n_matched * 100).round(2)

    print('\n   Frekuensi konflik per field:')
    for _, row in by_field.iterrows():
        print(f'   {row["field"]:<22} {row["jumlah_konflik"]:>6,}  ({row["persen_dari_matched"]:>5.1f}% dari matched pairs)')

    n_out_of_sync = df_golden['IS_OUT_OF_SYNC'].sum()
    n_logic = df_golden['IS_LOGICAL_CONFLICT_NIPER'].sum()
    print(f'\n   IS_OUT_OF_SYNC (STATUS_NIB beda)        : {n_out_of_sync:,} record')
    print(f'   IS_LOGICAL_CONFLICT (FLAG_EKSPOR vs NIPER): {n_logic:,} record')


In [ ]:
# ============================================================
# LANGKAH 7: PROVENANCE ANALYSIS
# ============================================================

def provenance_analysis(df_provenance):
    summary = df_provenance.groupby(['field', 'source']).size().unstack(fill_value=0)
    for src in ['OSS', 'CEISA', 'N/A']:
        if src not in summary.columns:
            summary[src] = 0
    summary = summary[['OSS', 'CEISA', 'N/A']]
    pct = summary.div(summary.sum(axis=1), axis=0) * 100

    print('\n   Distribusi sumber per field (jumlah record):')
    print(summary.to_string())

    overall = df_provenance['source'].value_counts(normalize=True) * 100
    print('\n   Kontribusi keseluruhan per sumber:')
    for src, val in overall.items():
        bar = '#' * int(val / 2)
        print(f'   {src:<6}: {val:5.1f}%  {bar}')

    return summary, pct


In [ ]:
# ============================================================
# LANGKAH 8: QUALITY VALIDATION GOLDEN RECORD
# ============================================================

def validate_golden_record(df_golden):
    print('\n   Quality checks:')
    checks = []

    # NIB dummy "0000000000000" (anomali NIB Invalid - Tahap 1 Validitas) secara sah dimiliki
    # banyak perusahaan berbeda -> dikecualikan dari cek keunikan NIB.
    is_dummy = df_golden['NIB'] == DUMMY_NIB
    n_dup_dummy = is_dummy.sum()
    n_dup_other = df_golden.loc[~is_dummy, 'NIB'].duplicated().sum()
    checks.append(('NIB unik (di luar NIB dummy/invalid)', n_dup_other == 0, f'{n_dup_other:,} NIB duplikat'))

    n_nib_invalid = (~df_golden['NIB'].astype(str).str.match(NIB_PATTERN)).sum()
    checks.append(('Format NIB (13 digit)', n_nib_invalid == 0, f'{n_nib_invalid:,} NIB format tidak valid'))

    n_npwp_invalid = (~df_golden['NPWP'].astype(str).str.match(NPWP_PATTERN)).sum()
    checks.append(('Format NPWP (XX.XXX.XXX.X-XXX.XXX)', n_npwp_invalid == 0, f'{n_npwp_invalid:,} NPWP format tidak valid'))

    n_status_invalid = (~df_golden['STATUS_NIB'].isin(STATUS_NIB_POOL)).sum()
    checks.append(('STATUS_NIB valid (AKTIF/DIBEKUKAN/DICABUT)', n_status_invalid == 0, f'{n_status_invalid:,} STATUS_NIB tidak valid'))

    for name, passed, detail in checks:
        status = 'PASS' if passed else 'INFO'
        print(f'   [{status}] {name:<42} | {detail}')

    if n_dup_other > 0:
        print(f'\n   -> {n_dup_other:,} NIB duplikat (di luar dummy) di-drop (keep first).')
        keep_mask = is_dummy | ~df_golden['NIB'].duplicated()
        df_golden = df_golden[keep_mask].reset_index(drop=True)

    if n_dup_dummy > 0:
        print(f'\n   INFO: {n_dup_dummy:,} golden record memiliki NIB dummy "{DUMMY_NIB}"')
        print('         (anomali NIB Invalid "Dummy/Kosong") - tetap dipertahankan sebagai record')
        print('         terpisah (GR_ID unik) karena merepresentasikan perusahaan berbeda.')

    return df_golden


In [ ]:
# ============================================================
# MAIN PIPELINE (Langkah 1-9)
# ============================================================

if __name__ == "__main__":
    print('=' * 60)
    print('TAHAP 4: GOLDEN RECORD & SURVIVORSHIP')
    print('=' * 60)

    # Langkah 1: Load data
    df_oss = pd.read_csv('data/processed/oss_cleaned.csv', dtype={'NIB': str, 'NPWP_PERSEROAN': str})
    df_ceisa = pd.read_csv('data/processed/ceisa_cleaned.csv', dtype={'NIB': str, 'NPWP': str})
    candidate_pairs = pd.read_csv('reports/candidate_pairs.csv', dtype={'NIB_OSS': str, 'NIB_CEISA': str})
    df_clusters = pd.read_csv('reports/duplicate_cluster.csv')
    print(f'\n1. Load data: OSS={len(df_oss):,} baris, CEISA={len(df_ceisa):,} baris, candidate_pairs={len(candidate_pairs):,} baris')

    # Dedup residu anomali "Duplicate Entry" di OSS menggunakan duplicate_cluster.csv (Tahap 3
    # langkah 9) - hanya buang baris kedua dari setiap cluster OSS_n (NIB sah, terduplikasi persis).
    # NIB dummy "0000000000000" SENGAJA tidak dianggap "Duplicate Entry" (lihat step07: cluster_oss_duplicates
    # mengecualikan DUMMY_NIB) karena ia merepresentasikan banyak perusahaan BERBEDA dengan NIB tidak valid.
    n_before = len(df_oss)
    drop_ids = []
    for _, grp in df_clusters[df_clusters['source'] == 'OSS'].groupby('cluster_id'):
        ids = sorted(grp['record_id'].astype(int).tolist())
        drop_ids.extend(ids[1:])
    df_oss = df_oss.drop(index=drop_ids).reset_index(drop=True)
    df_oss['_ROW_ID'] = df_oss.index  # id unik per baris - NIB dummy "0000000000000" tidak unik
    print(f'   Dedup OSS Duplicate Entry: {n_before:,} -> {len(df_oss):,} baris ({len(drop_ids)} baris duplikat persis dibuang)')

    print('\n2. Survivorship rules (FIELD_RULES) terdefinisi:')
    print(f'   {len(FIELD_RULES)} field di-mapping. OSS=System of Record (legalitas & status),')
    print('   CEISA menang untuk field operasional (KODE_KANTOR, NOMOR_TELPON, KATEGORI, NIPER, NOMOR_API, TGL_SYNC_OSS).')

    # Langkah 4: Proses matched pairs
    print('\n4. Proses matched pairs (EXACT_NIB, EXACT_NPWP, FUZZY_MATCH/REVIEW)...')
    matched_rows, prov_rows_m, conflict_rows, matched_oss_nibs, matched_ceisa_ids = build_matched_records(
        df_oss, df_ceisa, candidate_pairs, FIELD_RULES
    )
    print(f'   {len(matched_rows):,} golden record terbentuk dari matched pairs')

    # Langkah 5: Proses orphan records
    print('\n5. Proses orphan records (OSS_ONLY / CEISA_ONLY)...')
    orphan_rows, prov_rows_o = build_orphan_records(df_oss, df_ceisa, matched_oss_nibs, matched_ceisa_ids, FIELD_RULES)
    n_oss_only = sum(1 for r in orphan_rows if r['SOURCE'] == 'OSS_ONLY')
    n_ceisa_only = sum(1 for r in orphan_rows if r['SOURCE'] == 'CEISA_ONLY')
    print(f'   OSS_ONLY   : {n_oss_only:,} record')
    print(f'   CEISA_ONLY : {n_ceisa_only:,} record')

    # Gabungkan semua golden record + assign GR_ID
    all_rows = matched_rows + orphan_rows
    df_golden = pd.DataFrame(all_rows)
    df_golden.insert(0, 'GR_ID', [f'GR{i:06d}' for i in range(1, len(df_golden) + 1)])
    df_golden = df_golden[GOLDEN_COLUMNS]

    df_provenance = pd.DataFrame(prov_rows_m + prov_rows_o)
    df_conflicts = pd.DataFrame(conflict_rows, columns=['NIB', 'field', 'oss_value', 'ceisa_value', 'winner'])

    # Langkah 6: Analisis pola konflik
    print('\n6. Analisis pola konflik OSS vs CEISA...')
    analyze_conflict_patterns(df_conflicts, df_golden)

    # Langkah 7: Provenance analysis
    print('\n7. Provenance analysis...')
    provenance_analysis(df_provenance)

    # Langkah 8: Quality validation
    print('\n8. Quality validation golden record...')
    df_golden = validate_golden_record(df_golden)

    # Langkah 9: Export
    print('\n9. Export hasil...')
    df_golden.to_csv('data/golden/golden_record.csv', index=False)
    print(f'   data/golden/golden_record.csv -> {len(df_golden):,} golden records')

    df_provenance.to_csv('reports/provenance_log.csv', index=False)
    print(f'   reports/provenance_log.csv    -> {len(df_provenance):,} entri provenance')

    df_conflicts.to_csv('reports/conflict_log.csv', index=False)
    print(f'   reports/conflict_log.csv      -> {len(df_conflicts):,} konflik tercatat')

    print('\n' + '=' * 60)
    print('RINGKASAN')
    print('=' * 60)
    print(f'Total golden record   : {len(df_golden):,}')
    print(f'  - MATCHED           : {(df_golden["SOURCE"] == "MATCHED").sum():,}')
    print(f'  - OSS_ONLY          : {(df_golden["SOURCE"] == "OSS_ONLY").sum():,}')
    print(f'  - CEISA_ONLY        : {(df_golden["SOURCE"] == "CEISA_ONLY").sum():,}')
    print(f'IS_OUT_OF_SYNC        : {df_golden["IS_OUT_OF_SYNC"].sum():,}')
    print(f'IS_STALE              : {(df_golden["IS_STALE"] == True).sum():,}')
    print(f'HIGH_SYNC_LAG         : {(df_golden["HIGH_SYNC_LAG"] == True).sum():,}')
    print(f'IS_LOGICAL_CONFLICT   : {df_golden["IS_LOGICAL_CONFLICT_NIPER"].sum():,}')
    print('=' * 60)


## Tahap 5 — Data Quality Monitoring

Berdasarkan: `docs/tahapan/tahap5_dq_monitoring.md` & `docs/02_business_rules.md` §1, §4.

Quality Rules Engine sederhana (rule_id, dimension, field, check, severity) dijalankan pada 3 dataset:
OSS (before), CEISA (before), Golden Record (after) - lalu dibandingkan dalam satu scorecard per dimensi
DMBOK (Completeness, Validity, Uniqueness, Consistency, Timeliness) untuk menunjukkan peningkatan
kualitas data hasil MDM.

Output: `reports/dq_scorecard.csv`, `reports/dq_rule_results.csv`, `reports/flagged_for_review.csv`,
`reports/quality_gate_report.csv`.

In [ ]:
# [TAHAP 5] DATA QUALITY MONITORING
# Berdasarkan: docs/tahapan/tahap5_dq_monitoring.md & docs/02_business_rules.md §1, §4
#
# Quality Rules Engine sederhana (rule_id, dimension, field, check, severity) dijalankan
# pada 3 dataset: OSS (before), CEISA (before), Golden Record (after) - lalu dibandingkan
# dalam satu scorecard per dimensi DMBOK (Completeness, Validity, Uniqueness, Consistency,
# Timeliness) untuk menunjukkan peningkatan kualitas data hasil MDM.

import pandas as pd
import re
from dataclasses import dataclass
from typing import Callable


NIB_PATTERN = re.compile(r'^\d{13}$')
NPWP_PATTERN = re.compile(r'^\d{2}\.\d{3}\.\d{3}\.\d{1}-\d{3}\.\d{3}$')
KODE_POS_PATTERN = re.compile(r'^\d{5}$')

DIMENSIONS = ['Completeness', 'Validity', 'Uniqueness', 'Consistency', 'Timeliness']


@dataclass
class QualityRule:
    rule_id: str
    dimension: str
    field: str
    description: str
    check: Callable[[pd.DataFrame], pd.Series]  # df -> bool Series (True = pass)
    severity: str  # HIGH / MEDIUM / LOW


@dataclass
class RuleResult:
    dataset: str
    rule_id: str
    dimension: str
    field: str
    description: str
    severity: str
    total: int
    n_pass: int
    n_fail: int
    pass_rate: float


# --- Quality Rules Engine: check builders (Langkah 2) ---

def _to_digit_str(series):
    """KODE_POS terbaca float64 (mis. 1330.0) - konversi ke string digit tanpa '.0'
    agar panjang digit (utk cek format 5-digit) tetap apa adanya (leading zero yang
    hilang akibat tipe numerik akan terdeteksi sebagai format tidak valid)."""
    def conv(x):
        if pd.isna(x):
            return None
        if isinstance(x, float) and x.is_integer():
            return str(int(x))
        return str(x)
    return series.apply(conv)


def check_completeness(field):
    """Field wajib (NIB/NPWP/NAMA/STATUS_NIB) tidak boleh null/kosong."""
    def _check(df):
        col = df[field]
        return col.notna() & (col.astype(str).str.strip() != '')
    return _check


def check_format(field, pattern, numeric=False):
    """Field opsional/format: nilai kosong dianggap PASS (itu isu Completeness, bukan
    Validity), nilai yang terisi harus cocok dengan pattern."""
    def _check(df):
        col = _to_digit_str(df[field]) if numeric else df[field]
        is_null = col.isna()
        match = col.astype(str).str.match(pattern)
        return is_null | match
    return _check


def check_not_dummy_nib(df):
    """NIB tidak boleh nilai dummy/kosong "0000000000000" (anomali NIB Invalid - Dummy/Kosong)."""
    return df['NIB'].astype(str) != DUMMY_NIB


def check_enum(field, pool):
    """Nilai field (jika terisi) harus termasuk dalam pool referensi yang sah."""
    def _check(df):
        col = df[field]
        return col.isna() | col.isin(pool)
    return _check


def check_unique_nib(df):
    """NIB unik secara internal - kecuali NIB dummy "0000000000000" yang secara sah
    dimiliki banyak entitas berbeda (lihat tahap4_golden_record.md langkah 8)."""
    nib = df['NIB'].astype(str)
    is_dummy = nib == DUMMY_NIB
    is_dup = nib.duplicated(keep=False)
    return ~(is_dup & ~is_dummy)


def check_flag_impor_jenis_api(df):
    """FLAG_IMPOR='Y' -> JENIS_API harus terisi; FLAG_IMPOR='N' -> JENIS_API harus kosong."""
    flag = df['FLAG_IMPOR']
    jenis_filled = df['JENIS_API'].notna() & (df['JENIS_API'].astype(str).str.strip() != '')
    return flag.isna() | ((flag == 'Y') & jenis_filled) | ((flag == 'N') & ~jenis_filled)


def check_kategori_niper(df):
    """KATEGORI='IMPORTIR' -> NIPER harus kosong; KATEGORI EKSPORTIR/KEDUA-DUANYA -> NIPER
    harus terisi (anomali "Logical Conflict")."""
    kategori = df['KATEGORI']
    niper_filled = df['NIPER'].notna() & (df['NIPER'].astype(str).str.strip() != '')
    return (
        kategori.isna()
        | ((kategori == 'IMPORTIR') & ~niper_filled)
        | (kategori.isin(['EKSPORTIR', 'KEDUA-DUANYA']) & niper_filled)
    )


def check_flag_is_false(field):
    """Flag boolean (IS_STALE/HIGH_SYNC_LAG/IS_OUT_OF_SYNC/IS_LOGICAL_CONFLICT_NIPER) harus
    False. NaN (flag tidak relevan utk record ybs, mis. OSS_ONLY tanpa HIGH_SYNC_LAG)
    dianggap PASS (tidak berlaku)."""
    def _check(df):
        col = df[field].apply(lambda x: bool(x) if pd.notna(x) else False)
        return ~col
    return _check


# --- Definisi rules per dataset (Langkah 3) ---

OSS_RULES = [
    QualityRule('COMP-NIB', 'Completeness', 'NIB', 'NIB tidak boleh kosong',
                check_completeness('NIB'), 'HIGH'),
    QualityRule('COMP-NPWP', 'Completeness', 'NPWP_PERSEROAN', 'NPWP tidak boleh kosong',
                check_completeness('NPWP_PERSEROAN'), 'HIGH'),
    QualityRule('COMP-NAMA', 'Completeness', 'NAMA_PERSEROAN', 'NAMA tidak boleh kosong',
                check_completeness('NAMA_PERSEROAN'), 'HIGH'),
    QualityRule('COMP-STATUS_NIB', 'Completeness', 'STATUS_NIB', 'STATUS_NIB tidak boleh kosong',
                check_completeness('STATUS_NIB'), 'HIGH'),

    QualityRule('VAL-NIB-FORMAT', 'Validity', 'NIB', 'NIB harus 13 digit numerik',
                check_format('NIB', NIB_PATTERN), 'HIGH'),
    QualityRule('VAL-NIB-DUMMY', 'Validity', 'NIB', 'NIB tidak boleh dummy "0000000000000"',
                check_not_dummy_nib, 'MEDIUM'),
    QualityRule('VAL-NPWP-FORMAT', 'Validity', 'NPWP_PERSEROAN', 'NPWP harus format XX.XXX.XXX.X-XXX.XXX',
                check_format('NPWP_PERSEROAN', NPWP_PATTERN), 'HIGH'),
    QualityRule('VAL-KODE_POS-FORMAT', 'Validity', 'KODE_POS_PERSEROAN', 'KODE_POS (jika terisi) harus 5 digit',
                check_format('KODE_POS_PERSEROAN', KODE_POS_PATTERN, numeric=True), 'LOW'),
    QualityRule('VAL-STATUS_NIB-ENUM', 'Validity', 'STATUS_NIB', f'STATUS_NIB harus salah satu dari {STATUS_NIB_POOL}',
                check_enum('STATUS_NIB', STATUS_NIB_POOL), 'HIGH'),
    QualityRule('VAL-JENIS_PERSEROAN-ENUM', 'Validity', 'JENIS_PERSEROAN', f'JENIS_PERSEROAN harus salah satu dari {JENIS_PERSEROAN_POOL}',
                check_enum('JENIS_PERSEROAN', JENIS_PERSEROAN_POOL), 'MEDIUM'),

    QualityRule('UNIQ-NIB', 'Uniqueness', 'NIB', 'NIB unik (di luar NIB dummy)',
                check_unique_nib, 'HIGH'),

    QualityRule('CONS-FLAG_IMPOR-JENIS_API', 'Consistency', 'FLAG_IMPOR/JENIS_API',
                'FLAG_IMPOR konsisten dengan pengisian JENIS_API',
                check_flag_impor_jenis_api, 'MEDIUM'),

    QualityRule('TIME-IS_STALE', 'Timeliness', 'IS_STALE', 'TGL_PERUBAHAN_NIB tidak boleh > 1 tahun (IS_STALE)',
                check_flag_is_false('IS_STALE'), 'MEDIUM'),
]

CEISA_RULES = [
    QualityRule('COMP-NIB', 'Completeness', 'NIB', 'NIB tidak boleh kosong',
                check_completeness('NIB'), 'HIGH'),
    QualityRule('COMP-NPWP', 'Completeness', 'NPWP', 'NPWP tidak boleh kosong',
                check_completeness('NPWP'), 'HIGH'),
    QualityRule('COMP-NAMA', 'Completeness', 'NAMA_PERUSAHAAN', 'NAMA tidak boleh kosong',
                check_completeness('NAMA_PERUSAHAAN'), 'HIGH'),
    QualityRule('COMP-STATUS_NIB', 'Completeness', 'STATUS_NIB', 'STATUS_NIB tidak boleh kosong',
                check_completeness('STATUS_NIB'), 'HIGH'),

    QualityRule('VAL-NIB-FORMAT', 'Validity', 'NIB', 'NIB harus 13 digit numerik',
                check_format('NIB', NIB_PATTERN), 'HIGH'),
    QualityRule('VAL-NIB-DUMMY', 'Validity', 'NIB', 'NIB tidak boleh dummy "0000000000000"',
                check_not_dummy_nib, 'MEDIUM'),
    QualityRule('VAL-NPWP-FORMAT', 'Validity', 'NPWP', 'NPWP harus format XX.XXX.XXX.X-XXX.XXX (anomali Inconsistent NPWP)',
                check_format('NPWP', NPWP_PATTERN), 'HIGH'),
    QualityRule('VAL-KODE_POS-FORMAT', 'Validity', 'KODE_POS', 'KODE_POS (jika terisi) harus 5 digit',
                check_format('KODE_POS', KODE_POS_PATTERN, numeric=True), 'LOW'),
    QualityRule('VAL-STATUS_NIB-ENUM', 'Validity', 'STATUS_NIB', f'STATUS_NIB harus salah satu dari {STATUS_NIB_POOL}',
                check_enum('STATUS_NIB', STATUS_NIB_POOL), 'HIGH'),
    QualityRule('VAL-KATEGORI-ENUM', 'Validity', 'KATEGORI', f'KATEGORI harus salah satu dari {KATEGORI_CEISA_POOL}',
                check_enum('KATEGORI', KATEGORI_CEISA_POOL), 'MEDIUM'),

    QualityRule('UNIQ-NIB', 'Uniqueness', 'NIB', 'NIB unik (1 baris per NIB - data mart sudah di-dedup Tahap 2)',
                check_unique_nib, 'HIGH'),

    QualityRule('CONS-KATEGORI-NIPER', 'Consistency', 'KATEGORI/NIPER',
                'KATEGORI konsisten dengan pengisian NIPER (anomali Logical Conflict)',
                check_kategori_niper, 'MEDIUM'),

    QualityRule('TIME-HIGH_SYNC_LAG', 'Timeliness', 'HIGH_SYNC_LAG', 'TGL_SYNC_OSS tidak boleh > 30 hari (HIGH_SYNC_LAG)',
                check_flag_is_false('HIGH_SYNC_LAG'), 'MEDIUM'),
]

GOLDEN_RULES = [
    QualityRule('COMP-NIB', 'Completeness', 'NIB', 'NIB tidak boleh kosong',
                check_completeness('NIB'), 'HIGH'),
    QualityRule('COMP-NPWP', 'Completeness', 'NPWP', 'NPWP tidak boleh kosong',
                check_completeness('NPWP'), 'HIGH'),
    QualityRule('COMP-NAMA', 'Completeness', 'NAMA', 'NAMA tidak boleh kosong',
                check_completeness('NAMA'), 'HIGH'),
    QualityRule('COMP-STATUS_NIB', 'Completeness', 'STATUS_NIB', 'STATUS_NIB tidak boleh kosong',
                check_completeness('STATUS_NIB'), 'HIGH'),

    QualityRule('VAL-NIB-FORMAT', 'Validity', 'NIB', 'NIB harus 13 digit numerik',
                check_format('NIB', NIB_PATTERN), 'HIGH'),
    QualityRule('VAL-NIB-DUMMY', 'Validity', 'NIB', 'NIB tidak boleh dummy "0000000000000"',
                check_not_dummy_nib, 'MEDIUM'),
    QualityRule('VAL-NPWP-FORMAT', 'Validity', 'NPWP', 'NPWP harus format XX.XXX.XXX.X-XXX.XXX',
                check_format('NPWP', NPWP_PATTERN), 'HIGH'),
    QualityRule('VAL-KODE_POS-FORMAT', 'Validity', 'KODE_POS', 'KODE_POS (jika terisi) harus 5 digit',
                check_format('KODE_POS', KODE_POS_PATTERN, numeric=True), 'LOW'),
    QualityRule('VAL-STATUS_NIB-ENUM', 'Validity', 'STATUS_NIB', f'STATUS_NIB harus salah satu dari {STATUS_NIB_POOL}',
                check_enum('STATUS_NIB', STATUS_NIB_POOL), 'HIGH'),
    QualityRule('VAL-JENIS_PERSEROAN-ENUM', 'Validity', 'JENIS_PERSEROAN', f'JENIS_PERSEROAN harus salah satu dari {JENIS_PERSEROAN_POOL}',
                check_enum('JENIS_PERSEROAN', JENIS_PERSEROAN_POOL), 'MEDIUM'),
    QualityRule('VAL-KATEGORI-ENUM', 'Validity', 'KATEGORI', f'KATEGORI harus salah satu dari {KATEGORI_CEISA_POOL}',
                check_enum('KATEGORI', KATEGORI_CEISA_POOL), 'MEDIUM'),

    QualityRule('UNIQ-NIB', 'Uniqueness', 'NIB', 'NIB unik (di luar NIB dummy - lihat tahap4 langkah 8)',
                check_unique_nib, 'HIGH'),

    QualityRule('CONS-FLAG_IMPOR-JENIS_API', 'Consistency', 'FLAG_IMPOR/JENIS_API',
                'FLAG_IMPOR konsisten dengan pengisian JENIS_API',
                check_flag_impor_jenis_api, 'MEDIUM'),
    QualityRule('CONS-KATEGORI-NIPER', 'Consistency', 'KATEGORI/NIPER',
                'KATEGORI konsisten dengan pengisian NIPER (anomali Logical Conflict)',
                check_kategori_niper, 'MEDIUM'),
    QualityRule('CONS-STATUS_NIB-SYNC', 'Consistency', 'IS_OUT_OF_SYNC',
                'STATUS_NIB OSS & CEISA harus sinkron (anomali Sync Conflict)',
                check_flag_is_false('IS_OUT_OF_SYNC'), 'HIGH'),
    QualityRule('CONS-FLAG_EKSPOR-NIPER', 'Consistency', 'IS_LOGICAL_CONFLICT_NIPER',
                'FLAG_EKSPOR konsisten dengan NIPER (anomali Logical Conflict)',
                check_flag_is_false('IS_LOGICAL_CONFLICT_NIPER'), 'HIGH'),

    QualityRule('TIME-IS_STALE', 'Timeliness', 'IS_STALE', 'TGL_PERUBAHAN_NIB tidak boleh > 1 tahun (IS_STALE)',
                check_flag_is_false('IS_STALE'), 'MEDIUM'),
    QualityRule('TIME-HIGH_SYNC_LAG', 'Timeliness', 'HIGH_SYNC_LAG', 'TGL_SYNC_OSS tidak boleh > 30 hari (HIGH_SYNC_LAG)',
                check_flag_is_false('HIGH_SYNC_LAG'), 'MEDIUM'),
]


# --- Eksekusi rules (Langkah 4) ---

def run_rules(df, rules, dataset_name):
    results = []
    total = len(df)
    for rule in rules:
        mask = rule.check(df)
        n_pass = int(mask.sum())
        results.append(RuleResult(
            dataset=dataset_name, rule_id=rule.rule_id, dimension=rule.dimension,
            field=rule.field, description=rule.description, severity=rule.severity,
            total=total, n_pass=n_pass, n_fail=total - n_pass,
            pass_rate=(n_pass / total * 100) if total else 0.0,
        ))
    return results


def print_rule_results(results):
    for r in results:
        status = 'PASS' if r.pass_rate == 100 else ('WARN' if r.pass_rate >= 95 else 'FAIL')
        print(f'   [{status}] {r.rule_id:<26} {r.dimension:<13} | {r.pass_rate:6.2f}% '
              f'({r.n_pass:,}/{r.total:,}) | {r.description}')


# --- Scorecard perbandingan before vs after (Langkah 5) ---

def build_scorecard(df_results):
    pivot = df_results.groupby(['dimension', 'dataset'])['pass_rate'].mean().unstack('dataset')
    pivot = pivot.reindex(DIMENSIONS)[['OSS', 'CEISA', 'GOLDEN']]
    pivot.loc['Overall'] = pivot.mean(axis=0)

    scorecard = pivot.reset_index().rename(columns={
        'dimension': 'dimension', 'OSS': 'oss_score', 'CEISA': 'ceisa_score', 'GOLDEN': 'golden_score',
    })
    scorecard['delta_vs_oss'] = scorecard['golden_score'] - scorecard['oss_score']
    scorecard['delta_vs_ceisa'] = scorecard['golden_score'] - scorecard['ceisa_score']
    return scorecard


if __name__ == "__main__":
    print('=' * 60)
    print('TAHAP 5: DATA QUALITY MONITORING')
    print('=' * 60)

    # Langkah 1: Load data
    df_oss = pd.read_csv('data/processed/oss_cleaned.csv', dtype={'NIB': str, 'NPWP_PERSEROAN': str})
    df_ceisa = pd.read_csv('data/processed/ceisa_cleaned.csv', dtype={'NIB': str, 'NPWP': str})
    df_golden = pd.read_csv('data/golden/golden_record.csv', dtype={'NIB': str, 'NPWP': str})
    print(f'\n1. Load data: OSS={len(df_oss):,} baris, CEISA={len(df_ceisa):,} baris, '
          f'Golden Record={len(df_golden):,} baris')

    # Langkah 2-3: Quality Rules Engine + definisi rules per dimensi
    print(f'\n2-3. Quality Rules Engine terdefinisi: '
          f'OSS={len(OSS_RULES)} rules, CEISA={len(CEISA_RULES)} rules, Golden={len(GOLDEN_RULES)} rules')
    print(f'     5 dimensi DMBOK: {", ".join(DIMENSIONS)}')

    # Langkah 4: Eksekusi rules untuk 3 dataset
    print('\n4. Eksekusi rules...')
    results_oss = run_rules(df_oss, OSS_RULES, 'OSS')
    results_ceisa = run_rules(df_ceisa, CEISA_RULES, 'CEISA')
    results_golden = run_rules(df_golden, GOLDEN_RULES, 'GOLDEN')
    df_results = pd.DataFrame([r.__dict__ for r in results_oss + results_ceisa + results_golden])

    print(f'\n   --- OSS (Before, {len(df_oss):,} baris) ---')
    print_rule_results(results_oss)
    print(f'\n   --- CEISA (Before, {len(df_ceisa):,} baris) ---')
    print_rule_results(results_ceisa)
    print(f'\n   --- Golden Record (After, {len(df_golden):,} baris) ---')
    print_rule_results(results_golden)

    # Langkah 5: Scorecard perbandingan before vs after
    print('\n5. Scorecard perbandingan (Before vs After)...')
    df_scorecard = build_scorecard(df_results)
    print('\n' + df_scorecard.to_string(index=False, float_format=lambda x: f'{x:6.2f}'))

    # Langkah 6: Export
    df_scorecard.to_csv('reports/dq_scorecard.csv', index=False)
    df_results.to_csv('reports/dq_rule_results.csv', index=False)
    print('\n6. Export hasil...')
    print(f'   reports/dq_scorecard.csv     -> {len(df_scorecard):,} baris (skor per dimensi)')
    print(f'   reports/dq_rule_results.csv  -> {len(df_results):,} baris (detail per rule)')

    overall = df_scorecard[df_scorecard['dimension'] == 'Overall'].iloc[0]
    print('\n' + '=' * 60)
    print('RINGKASAN OVERALL DQ SCORE')
    print('=' * 60)
    print(f'OSS (Before)          : {overall["oss_score"]:.2f}/100')
    print(f'CEISA (Before)        : {overall["ceisa_score"]:.2f}/100')
    print(f'Golden Record (After) : {overall["golden_score"]:.2f}/100')
    print(f'Peningkatan vs OSS    : {overall["delta_vs_oss"]:+.2f} poin')
    print(f'Peningkatan vs CEISA  : {overall["delta_vs_ceisa"]:+.2f} poin')
    print('=' * 60)


## Tahap 6 — YData Profiling Dashboard

Berdasarkan: `docs/tahapan/tahap6_profiling_dashboard.md`.

Profiling ulang terhadap Golden Record (after MDM) menggunakan `ydata_profiling`, lalu membandingkan
ringkasan profiling_before (OSS+CEISA) vs profiling_after (Golden Record), ditutup dengan insight
bisnis & rekomendasi data governance.

In [ ]:
# [TAHAP 6] YDATA PROFILING DASHBOARD
# Berdasarkan: docs/tahapan/tahap6_profiling_dashboard.md
#
# Profiling ulang terhadap Golden Record (after MDM) menggunakan ydata_profiling,
# lalu membandingkan ringkasan profiling_before (OSS+CEISA) vs profiling_after
# (Golden Record), ditutup dengan insight bisnis & rekomendasi data governance.

import pandas as pd
from ydata_profiling import ProfileReport


DUMMY_NIB = '0' * 13  # NIB dummy/kosong - sama dengan Source/step07_matching.py


def format_validity_pct(df, nib_col, npwp_col, kp_col):
    """Hitung % valid format untuk NIB, NPWP, dan KODE_POS (dari yang terisi)."""
    nib_valid = df[nib_col].astype(str).str.match(NIB_PATTERN).mean() * 100
    npwp_valid = df[npwp_col].astype(str).str.match(NPWP_PATTERN).mean() * 100

    kp_series = _to_digit_str(df[kp_col])
    kp_filled = kp_series.dropna()
    kp_valid = kp_filled.str.match(KODE_POS_PATTERN).mean() * 100 if len(kp_filled) else 0.0

    return nib_valid, npwp_valid, kp_valid


if __name__ == "__main__":
    df_oss = pd.read_csv('data/raw/oss_nib_data.csv')
    df_ceisa = pd.read_csv('data/raw/ceisa_data.csv')
    # NIB & NPWP dipaksa string agar leading zero (mis. NIB dummy "0000000000000")
    # tidak hilang akibat auto-infer ke int64 (lihat Source/step09_quality_monitoring.py)
    df_golden = pd.read_csv('data/golden/golden_record.csv', dtype={'NIB': str, 'NPWP': str})

    # 1-2. ydata_profiling report untuk Golden Record
    print(f"\n{'='*65}\n YDATA PROFILING REPORT — GOLDEN RECORD (AFTER MDM)\n{'='*65}")
    print("Membuat ydata_profiling report untuk Golden Record -> reports/profiling_after.html ...")
    profile = ProfileReport(
        df_golden,
        title='Data Profiling Report — Golden Record (After MDM)',
        explorative=True,
        correlations=YDATA_CORRELATIONS,
        interactions={"continuous": False},
    )
    profile.to_file('reports/profiling_after.html')
    print("Report disimpan: reports/profiling_after.html")

    # 3. Perbandingan profiling_before vs profiling_after
    print(f"\n{'='*65}\n PERBANDINGAN: BEFORE (OSS + CEISA) vs AFTER (GOLDEN RECORD)\n{'='*65}")

    n_oss, n_ceisa, n_golden = len(df_oss), len(df_ceisa), len(df_golden)
    n_before = n_oss + n_ceisa
    print("\n[Jumlah Record]")
    print(f"  OSS    (before) : {n_oss:,}")
    print(f"  CEISA  (before) : {n_ceisa:,}")
    print(f"  Total  (before) : {n_before:,}")
    print(f"  Golden (after)  : {n_golden:,}")
    print(f"  Pengurangan akibat dedup/matching: {n_before - n_golden:,} baris "
          f"({(1 - n_golden / n_before) * 100:.2f}%)")

    # Missing value % - field wajib (NIB, NPWP, NAMA, STATUS_NIB)
    mand_oss = ['NIB', 'NPWP_PERSEROAN', 'NAMA_PERSEROAN', 'STATUS_NIB']
    mand_ceisa = ['NIB', 'NPWP', 'NAMA_PERUSAHAAN', 'STATUS_NIB']
    mand_golden = ['NIB', 'NPWP', 'NAMA', 'STATUS_NIB']

    miss_oss = df_oss[mand_oss].isnull().mean().mean() * 100
    miss_ceisa = df_ceisa[mand_ceisa].isnull().mean().mean() * 100
    miss_golden = df_golden[mand_golden].isnull().mean().mean() * 100

    print("\n[Missing Value % - Field Wajib (NIB, NPWP, NAMA, STATUS_NIB)]")
    print(f"  OSS    (before) : {miss_oss:.2f}%")
    print(f"  CEISA  (before) : {miss_ceisa:.2f}%")
    print(f"  Golden (after)  : {miss_golden:.2f}%")

    # Format validity %
    nib_oss, npwp_oss, kp_oss = format_validity_pct(df_oss, 'NIB', 'NPWP_PERSEROAN', 'KODE_POS_PERSEROAN')
    nib_ceisa, npwp_ceisa, kp_ceisa = format_validity_pct(df_ceisa, 'NIB', 'NPWP', 'KODE_POS')
    nib_golden, npwp_golden, kp_golden = format_validity_pct(df_golden, 'NIB', 'NPWP', 'KODE_POS')

    print("\n[Format Validity %]")
    print(f"  {'Field':<22}{'OSS':>10}{'CEISA':>10}{'Golden':>10}")
    print(f"  {'NIB (13 digit)':<22}{nib_oss:>9.2f}%{nib_ceisa:>9.2f}%{nib_golden:>9.2f}%")
    print(f"  {'NPWP (format baku)':<22}{npwp_oss:>9.2f}%{npwp_ceisa:>9.2f}%{npwp_golden:>9.2f}%")
    print(f"  {'KODE_POS (5 digit)':<22}{kp_oss:>9.2f}%{kp_ceisa:>9.2f}%{kp_golden:>9.2f}%")

    # Duplikasi NIB
    dup_oss = df_oss['NIB'].duplicated().sum()
    dup_ceisa = df_ceisa['NIB'].duplicated().sum()
    dup_golden_real = df_golden[df_golden['NIB'].astype(str) != DUMMY_NIB]['NIB'].duplicated().sum()

    print("\n[Duplikasi NIB]")
    print(f"  OSS    (before) : {dup_oss:,} baris duplikat dari {n_oss:,} ({dup_oss / n_oss * 100:.2f}%)")
    print(f"  CEISA  (before) : {dup_ceisa:,} baris duplikat dari {n_ceisa:,} ({dup_ceisa / n_ceisa * 100:.2f}%) "
          f"(data mart, expected)")
    print(f"  Golden (after)  : {dup_golden_real:,} baris duplikat (NIB valid non-dummy) dari {n_golden:,} "
          f"({dup_golden_real / n_golden * 100:.2f}%)")

    # 4. Insight bisnis & rekomendasi data governance
    n_dummy_nib = (df_golden['NIB'].astype(str) == DUMMY_NIB).sum()
    n_out_of_sync = int(df_golden['IS_OUT_OF_SYNC'].sum()) if 'IS_OUT_OF_SYNC' in df_golden else 0
    n_logical_conflict = int(df_golden['IS_LOGICAL_CONFLICT_NIPER'].sum()) if 'IS_LOGICAL_CONFLICT_NIPER' in df_golden else 0
    n_high_sync_lag = int(df_golden['HIGH_SYNC_LAG'].apply(lambda x: bool(x) if pd.notna(x) else False).sum()) \
        if 'HIGH_SYNC_LAG' in df_golden else 0

    print(f"\n{'='*65}\n INSIGHT BISNIS & REKOMENDASI DATA GOVERNANCE\n{'='*65}")
    print(f"""
1. Konsolidasi data berhasil mengurangi {n_before - n_golden:,} baris ({(1 - n_golden / n_before) * 100:.1f}%)
   dari total record OSS+CEISA menjadi {n_golden:,} Golden Record (Single Importer/Exporter View),
   terutama berkat dedup snapshot CEISA dan exact/fuzzy matching NIB & NPWP (Tahap 3).

2. Uniqueness NIB meningkat signifikan: dari {(1 - dup_oss / n_oss) * 100:.2f}% (OSS) /
   {(1 - dup_ceisa / n_ceisa) * 100:.2f}% (CEISA) menjadi mendekati 100% di Golden Record
   (di luar NIB dummy/invalid yang sengaja dipertahankan sebagai catatan kualitas data).

3. Risiko kualitas data yang masih perlu ditindaklanjuti meski sudah ada Golden Record:
   - {n_dummy_nib:,} record memiliki NIB dummy/tidak valid ('{DUMMY_NIB}') -> perlu verifikasi ulang ke OSS.
   - {n_out_of_sync:,} record berstatus 'Sync Conflict' (STATUS_NIB OSS vs CEISA berbeda, IS_OUT_OF_SYNC=True).
   - {n_logical_conflict:,} record memiliki 'Logical Conflict' antara FLAG_EKSPOR dan NIPER.
   - {n_high_sync_lag:,} record memiliki HIGH_SYNC_LAG (CEISA belum sinkron >30 hari dari OSS).

4. Rekomendasi implementasi Data Governance & MDM:
   - Terapkan validasi format NIB/NPWP/KODE_POS di titik input (OSS & CEISA) agar anomali
     format tidak terbawa ke data mart/operasional.
   - Jadikan OSS sebagai System of Record untuk legalitas (NIB, status badan hukum) dan
     terapkan SLA sinkronisasi CEISA <30 hari untuk menekan HIGH_SYNC_LAG.
   - Bangun proses rekonsiliasi berkala (mis. bulanan) untuk menyelesaikan Sync Conflict
     dan Logical Conflict (FLAG_EKSPOR vs NIPER) yang terdeteksi di Golden Record.
   - Jadwalkan re-running pipeline profiling -> cleansing -> matching -> golden record ->
     DQ monitoring secara periodik agar dq_scorecard.csv selalu mencerminkan kondisi terkini.
""")

    print("✅ Tahap 6 - YData Profiling Dashboard selesai.")


## Laporan Akhir & Bahan Presentasi (Bonus)

Menggabungkan seluruh hasil pipeline (Tahap 0-6) + dokumentasi logika bisnis (`docs/`) menjadi satu
laporan HTML interaktif (Plotly) yang memuat temuan teknis & bisnis, insight, rekomendasi data
governance, serta checklist deliverable sesuai pedoman mini project.

Output: `reports/final_report.html` (single-page) dan `reports/final_report/` (multi-page).

In [ ]:
# [TAHAP FINAL] LAPORAN AKHIR INTERAKTIF & BAHAN PRESENTASI
#
# Menyatukan seluruh hasil pipeline (Tahap 0-6) + dokumentasi logika bisnis
# (docs/03_logika_bisnis_presentasi.md, docs/01_data_dictionary.md,
# docs/02_business_rules.md) menjadi satu laporan HTML interaktif
# (Plotly) yang memuat temuan teknis & bisnis, insight, rekomendasi
# data governance, serta checklist deliverable sesuai pedoman mini project
# (di luar notebook .ipynb). Laporan ini didesain agar bisa berperan
# sebagai bahan presentasi (setiap section = 1 "slide", siap di-print ke PDF).

import base64
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
import markdown as md
import plotly.graph_objects as go
from plotly.offline import get_plotlyjs
from jinja2 import Template

PLOTLY_JS = get_plotlyjs()


ROOT = Path.cwd()  # asumsi: notebook dijalankan dari root project
REPORTS = ROOT / 'reports'
DATA = ROOT / 'data'
DOCS = ROOT / 'docs'

C = dict(oss='#5C6BC0', ceisa='#26A69A', golden='#2E7D32',
         warn='#FB8C00', bad='#E53935', neutral='#90A4AE', accent='#FFB300')

# Lampiran D — Data Viewer: CSV hasil pipeline yang di-embed (base64) sebagai tabel interaktif.
CSV_FILES = {
    'golden_record': ('Golden Record (Hasil Akhir MDM)', 'data/golden/golden_record.csv'),
    'dq_scorecard': ('DQ Scorecard', 'reports/dq_scorecard.csv'),
    'dq_rule_results': ('DQ Rule Results', 'reports/dq_rule_results.csv'),
    'candidate_pairs': ('Candidate Pairs (Hasil Matching)', 'reports/candidate_pairs.csv'),
    'duplicate_cluster': ('Duplicate Cluster', 'reports/duplicate_cluster.csv'),
    'conflict_log': ('Conflict Log (Survivorship)', 'reports/conflict_log.csv'),
    'provenance_log': ('Provenance Log', 'reports/provenance_log.csv'),
    'audit_trail': ('Audit Trail (Cleansing)', 'reports/audit_trail.csv'),
    'quality_gate_report': ('Quality Gate Report', 'reports/quality_gate_report.csv'),
    'flagged_for_review': ('Flagged for Review', 'reports/flagged_for_review.csv'),
}
CSV_PAGE_SIZE = 25




In [ ]:
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def b64_image(rel_path):
    p = ROOT / rel_path
    ext = p.suffix.lstrip('.')
    return f"data:image/{ext};base64,{base64.b64encode(p.read_bytes()).decode()}"


def csv_b64(rel_path):
    return base64.b64encode((ROOT / rel_path).read_bytes()).decode()


def render_md(text):
    return md.markdown(text, extensions=['tables', 'fenced_code', 'sane_lists'])


def to_bool(series):
    return series.apply(lambda x: str(x).strip().lower() in ('true', '1', '1.0', 'yes'))


def fig_html(fig):
    fig.update_layout(template='plotly_white',
                       margin=dict(l=40, r=20, t=50, b=40),
                       font=dict(family='Segoe UI, sans-serif', size=12),
                       legend=dict(orientation='h', y=-0.18))
    return fig.to_html(full_html=False,
                        include_plotlyjs=False,
                        config={'displaylogo': False, 'responsive': True})


def split_doc_sections(rel_path):
    text = (ROOT / rel_path).read_text(encoding='utf-8')
    return text.split('\n---\n')


def dq_rules_table(df):
    sev_cls = {'HIGH': 'sev-high', 'MEDIUM': 'sev-medium', 'LOW': 'sev-low'}
    rows = []
    for _, r in df.iterrows():
        pr = r['pass_rate']
        pr_cls = 'pr-good' if pr >= 99.5 else ('pr-warn' if pr >= 90 else 'pr-bad')
        rows.append(
            f"<tr><td><code>{r['rule_id']}</code></td><td>{r['dimension']}</td>"
            f"<td>{r['field']}</td><td>{r['description']}</td>"
            f"<td><span class='badge {sev_cls.get(r['severity'], '')}'>{r['severity']}</span></td>"
            f"<td>{r['total']:,}</td><td>{r['n_pass']:,}</td><td>{r['n_fail']:,}</td>"
            f"<td><span class='badge {pr_cls}'>{pr:.2f}%</span></td></tr>"
        )
    head = ("<tr><th>Rule ID</th><th>Dimensi</th><th>Field</th><th>Deskripsi</th>"
            "<th>Severity</th><th>Total</th><th>Pass</th><th>Fail</th><th>Pass Rate</th></tr>")
    return f"<div class='table-wrap'><table class='dtable'><thead>{head}</thead><tbody>{''.join(rows)}</tbody></table></div>"


def gate_table(df):
    rows = []
    for _, r in df.iterrows():
        cls = 'status-pass' if r['status'] == 'PASS' else 'status-fail'
        rows.append(
            f"<tr><td>{r['dataset']}</td><td><code>{r['gate']}</code></td><td>{r['description']}</td>"
            f"<td>{r['value']:.1f}</td><td>{r['threshold']:.0f}</td>"
            f"<td><span class='badge {cls}'>{r['status']}</span></td></tr>"
        )
    head = "<tr><th>Dataset</th><th>Gate</th><th>Deskripsi</th><th>Nilai</th><th>Threshold</th><th>Status</th></tr>"
    return f"<div class='table-wrap'><table class='dtable'><thead>{head}</thead><tbody>{''.join(rows)}</tbody></table></div>"


def audit_table(df):
    rows = []
    for _, r in df.iterrows():
        rows.append(
            f"<tr><td>{r['dataset']}</td><td><code>{r['operation']}</code></td><td>{r['field']}</td>"
            f"<td>{r['n_affected']:,}</td><td>{r['pct_affected']:.2f}%</td><td>{r['description']}</td></tr>"
        )
    head = "<tr><th>Dataset</th><th>Operasi</th><th>Field</th><th>N Affected</th><th>% Affected</th><th>Deskripsi</th></tr>"
    return f"<div class='table-wrap'><table class='dtable'><thead>{head}</thead><tbody>{''.join(rows)}</tbody></table></div>"


def simple_table(df, fmt=None):
    return df.to_html(index=False, classes='dtable', border=0, escape=False, formatters=fmt or {})


def kpi_card(value, label, sub=None, color='#0D47A1'):
    sub_html = f"<div class='kpi-sub'>{sub}</div>" if sub else ""
    return (f"<div class='kpi-card'><div class='kpi-value' style='color:{color}'>{value}</div>"
            f"<div class='kpi-label'>{label}</div>{sub_html}</div>")




In [ ]:
# ---------------------------------------------------------------------------
# CSS & HTML TEMPLATE
# ---------------------------------------------------------------------------

CSS = """
:root {
  --primary: #0D47A1;
  --primary-dark: #082c63;
  --accent: #FFB300;
  --bg: #F2F5F9;
  --card: #FFFFFF;
  --text: #1B2530;
  --muted: #64748B;
}
* { box-sizing: border-box; }
body {
  margin: 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
  background: var(--bg); color: var(--text); font-size: 15px; line-height: 1.55;
}
.sidebar {
  position: fixed; top: 0; left: 0; bottom: 0; width: 270px; overflow-y: auto;
  background: var(--primary-dark); color: #E3EAF5; padding: 20px 0 60px 0; z-index: 10;
}
.sidebar .brand { padding: 0 20px 16px 20px; border-bottom: 1px solid rgba(255,255,255,.12); margin-bottom: 10px; }
.sidebar .brand b { font-size: 1.05rem; color: #fff; }
.sidebar .brand div { font-size: .75rem; color: #9FB3D1; margin-top: 4px; }
.sidebar .nav-group { font-size: .72rem; text-transform: uppercase; letter-spacing: .08em;
  color: #FFC44D; padding: 14px 20px 4px 20px; font-weight: 600; }
.sidebar a { display: block; padding: 6px 20px 6px 26px; color: #CBD8EE; text-decoration: none; font-size: .85rem; }
.sidebar a:hover { background: rgba(255,255,255,.08); color: #fff; }
main { margin-left: 270px; padding: 28px 36px 80px 36px; max-width: 1200px; }
.slide {
  background: var(--card); border-radius: 10px; box-shadow: 0 1px 4px rgba(20,40,80,.08);
  padding: 32px 40px; margin-bottom: 26px; scroll-margin-top: 16px;
}
.slide-kicker { font-size: .75rem; text-transform: uppercase; letter-spacing: .12em; color: var(--accent); font-weight: 700; margin-bottom: 6px; }
.slide h1 { font-size: 1.55rem; color: var(--primary); margin: 0 0 16px 0; border-bottom: 3px solid var(--accent); padding-bottom: 10px; }
.slide h2 { font-size: 1.2rem; color: var(--primary); margin-top: 24px; }
.slide h3 { font-size: 1.02rem; color: #2c3e50; }
.slide p, .slide li { color: #33414f; }
.slide blockquote { border-left: 4px solid var(--accent); margin: 12px 0; padding: 4px 16px; background: #FFF8E1; color: #5d4d22; }
.slide pre { background: #0f1b2d; color: #d4e4ff; padding: 14px; border-radius: 6px; overflow-x: auto; font-size: .8rem; }
.slide code { background: #eef2f8; padding: 1px 5px; border-radius: 4px; font-size: .85em; color: #c0392b; }
.slide pre code { background: none; color: inherit; padding: 0; }
.cover { text-align: center; padding: 60px 40px; }
.cover h1 { border: none; font-size: 2.1rem; }
.cover .subtitle { color: var(--muted); font-size: 1.05rem; margin-bottom: 28px; }
.kpi-row { display: flex; flex-wrap: wrap; gap: 16px; justify-content: center; margin: 24px 0; }
.kpi-card { background: #F7FAFF; border: 1px solid #E1E9F5; border-radius: 10px; padding: 16px 22px; min-width: 150px; }
.kpi-value { font-size: 1.7rem; font-weight: 700; }
.kpi-label { font-size: .8rem; color: var(--muted); margin-top: 4px; }
.kpi-sub { font-size: .72rem; color: #94A3B8; margin-top: 2px; }
.table-wrap { overflow-x: auto; margin: 12px 0; }
table.dtable { border-collapse: collapse; width: 100%; font-size: .82rem; }
table.dtable th { background: var(--primary); color: #fff; padding: 8px 10px; text-align: left; position: sticky; top: 0; }
table.dtable td { padding: 6px 10px; border-bottom: 1px solid #E5EAF1; vertical-align: top; }
table.dtable tr:nth-child(even) { background: #F7FAFF; }
.badge { display: inline-block; padding: 2px 9px; border-radius: 12px; font-size: .72rem; font-weight: 700; color: #fff; }
.sev-high { background: #E53935; } .sev-medium { background: #FB8C00; } .sev-low { background: #90A4AE; }
.pr-good { background: #2E7D32; } .pr-warn { background: #FB8C00; } .pr-bad { background: #E53935; }
.status-pass { background: #2E7D32; } .status-fail { background: #E53935; }
.grid-2 { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }
.grid-2 img { width: 100%; border: 1px solid #E5EAF1; border-radius: 6px; }
.callout { background: #EAF3FF; border-left: 4px solid var(--primary); padding: 14px 18px; border-radius: 6px; margin: 12px 0; }
.callout.warn { background: #FFF4E5; border-left-color: var(--warn); }
.callout.good { background: #EAF7EC; border-left-color: var(--golden); }
.reco-list li { margin-bottom: 10px; }
.deliv-status-ok { color: #2E7D32; font-weight: 700; }
.linklist a { color: var(--primary); }
.sidebar a.active { background: rgba(255,255,255,.12); color: #fff; font-weight: 600; border-left: 3px solid var(--accent); }
.toc-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(220px, 1fr)); gap: 18px; margin-top: 16px; }
.toc-group h3 { margin: 0 0 8px 0; color: var(--primary); font-size: 1rem; }
.toc-group ul { margin: 0; padding-left: 18px; }
.toc-group li { margin-bottom: 4px; }
.page-nav { display: flex; justify-content: space-between; gap: 12px; margin: 0 0 40px 0; }
.page-nav-btn { background: var(--card); border: 1px solid #E1E9F5; border-radius: 8px; padding: 12px 18px;
  color: var(--primary); text-decoration: none; font-weight: 600; flex: 1; box-shadow: 0 1px 4px rgba(20,40,80,.08); }
.page-nav-btn.next { text-align: right; }
.page-nav-btn:hover { background: #F7FAFF; }
.csv-tabs { display: flex; flex-wrap: wrap; gap: 8px; margin: 16px 0; }
.csv-tab { background: #F7FAFF; border: 1px solid #E1E9F5; border-radius: 20px; padding: 6px 16px;
  font-size: .8rem; cursor: pointer; color: var(--primary); }
.csv-tab.active { background: var(--primary); color: #fff; border-color: var(--primary); }
.csv-viewer-toolbar { display: flex; flex-wrap: wrap; justify-content: space-between; align-items: center;
  gap: 12px; margin: 12px 0; font-size: .85rem; color: var(--muted); }
.csv-search { padding: 6px 12px; border: 1px solid #E1E9F5; border-radius: 6px; font-size: .85rem; min-width: 240px; }
.csv-pagination { display: flex; justify-content: center; align-items: center; gap: 16px; margin: 12px 0; font-size: .85rem; }
.csv-pagination button { background: var(--primary); color: #fff; border: none; border-radius: 6px;
  padding: 6px 14px; cursor: pointer; font-size: .85rem; }
.csv-pagination button:disabled { background: #CBD5E1; cursor: not-allowed; }
@media print {
  .sidebar { display: none; }
  main { margin-left: 0; }
  .slide { box-shadow: none; page-break-after: always; }
}
"""

# Sidebar nav: shared antara TEMPLATE (single-page, anchor #sid) dan
# INDEX_TEMPLATE/PAGE_TEMPLATE (multi-page, link {sid}.html / index.html).
NAV_SINGLE = """
<nav class="sidebar">
  <div class="brand"><b>MDM DJBC</b><div>Single Importer &amp; Exporter View — Kelompok 5</div></div>
  {% for group in nav_groups %}
  <div class="nav-group">{{ group.label }}</div>
  {% for sid, stitle in group.links %}
  <a href="#{{ sid }}">{{ stitle }}</a>
  {% endfor %}
  {% endfor %}
</nav>
"""

NAV_MULTI = """
<nav class="sidebar">
  <div class="brand"><b>MDM DJBC</b><div>Single Importer &amp; Exporter View — Kelompok 5</div></div>
  {% for group in nav_groups %}
  <div class="nav-group">{{ group.label }}</div>
  {% for sid, stitle in group.links %}
  <a href="{{ 'index.html' if sid == 'cover' else sid + '.html' }}" class="{{ 'active' if sid == active_id else '' }}">{{ stitle }}</a>
  {% endfor %}
  {% endfor %}
</nav>
"""

TEMPLATE = Template("""
<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{{ title }}</title>
<style>{{ css }}</style>
<script>{{ plotlyjs }}</script>
</head>
<body>
""" + NAV_SINGLE + """
<main>
  {% for slide in slides %}
  <section class="slide {{ slide.cls|default('') }}" id="{{ slide.id }}">
    {% if slide.kicker %}<div class="slide-kicker">{{ slide.kicker }}</div>{% endif %}
    {% if slide.title %}<h1>{{ slide.title }}</h1>{% endif %}
    {{ slide.body|safe }}
  </section>
  {% endfor %}
</main>
</body>
</html>
""")

# Halaman landing/TOC multi-page: KPI summary (cover_body) + daftar isi per nav-group
INDEX_TEMPLATE = Template("""
<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{{ title }}</title>
<link rel="stylesheet" href="assets/style.css">
<script src="assets/plotly.min.js"></script>
</head>
<body>
""" + NAV_MULTI + """
<main>
  <section class="slide">
    {{ cover_body|safe }}
  </section>
  <section class="slide">
    <h1>Daftar Isi</h1>
    <div class="toc-grid">
    {% for group in nav_groups %}
    {% if group.label != 'Ringkasan' %}
      <div class="toc-group">
        <h3>{{ group.label }}</h3>
        <ul class="linklist">
        {% for sid, stitle in group.links %}
          <li><a href="{{ sid }}.html">{{ stitle }}</a></li>
        {% endfor %}
        </ul>
      </div>
    {% endif %}
    {% endfor %}
    </div>
  </section>
</main>
</body>
</html>
""")

# Halaman per-section multi-page: 1 slide + tombol prev/next
PAGE_TEMPLATE = Template("""
<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{{ slide.title or title }} — {{ title }}</title>
<link rel="stylesheet" href="assets/style.css">
<script src="assets/plotly.min.js"></script>
</head>
<body>
""" + NAV_MULTI + """
<main>
  <section class="slide {{ slide.cls|default('') }}">
    {% if slide.kicker %}<div class="slide-kicker">{{ slide.kicker }}</div>{% endif %}
    {% if slide.title %}<h1>{{ slide.title }}</h1>{% endif %}
    {{ slide.body|safe }}
  </section>
  <div class="page-nav">
    {% if prev %}<a class="page-nav-btn prev" href="{{ prev.href }}">&larr; {{ prev.title }}</a>{% else %}<span></span>{% endif %}
    {% if next %}<a class="page-nav-btn next" href="{{ next.href }}">{{ next.title }} &rarr;</a>{% else %}<span></span>{% endif %}
  </div>
</main>
</body>
</html>
""")




In [ ]:
# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    print("Memuat seluruh output pipeline (Tahap 0-6)...")

    df_oss = pd.read_csv(DATA / 'raw/oss_nib_data.csv')
    df_ceisa = pd.read_csv(DATA / 'raw/ceisa_data.csv')
    df_golden = pd.read_csv(DATA / 'golden/golden_record.csv', dtype={'NIB': str, 'NPWP': str})
    dataset_clean = pd.read_csv(DATA / 'processed/dataset_clean.csv', dtype=str)

    dq_scorecard = pd.read_csv(REPORTS / 'dq_scorecard.csv')
    dq_rules = pd.read_csv(REPORTS / 'dq_rule_results.csv')
    audit_trail = pd.read_csv(REPORTS / 'audit_trail.csv')
    quality_gate = pd.read_csv(REPORTS / 'quality_gate_report.csv')
    flagged = pd.read_csv(REPORTS / 'flagged_for_review.csv')
    candidate_pairs = pd.read_csv(REPORTS / 'candidate_pairs.csv')
    duplicate_cluster = pd.read_csv(REPORTS / 'duplicate_cluster.csv')
    conflict_log = pd.read_csv(REPORTS / 'conflict_log.csv')
    provenance_log = pd.read_csv(REPORTS / 'provenance_log.csv')
    provenance_log['source'] = provenance_log['source'].fillna('TIDAK ADA (kosong)')

    # -----------------------------------------------------------------
    # Statistik turunan
    # -----------------------------------------------------------------
    n_oss, n_ceisa, n_golden = len(df_oss), len(df_ceisa), len(df_golden)
    n_before = n_oss + n_ceisa
    pct_reduction = (1 - n_golden / n_before) * 100

    dup_oss = int(df_oss['NIB'].duplicated().sum())
    dup_ceisa = int(df_ceisa['NIB'].duplicated().sum())
    dup_golden_real = int(df_golden[df_golden['NIB'].astype(str) != DUMMY_NIB]['NIB'].duplicated().sum())

    mand_oss = ['NIB', 'NPWP_PERSEROAN', 'NAMA_PERSEROAN', 'STATUS_NIB']
    mand_ceisa = ['NIB', 'NPWP', 'NAMA_PERUSAHAAN', 'STATUS_NIB']
    mand_golden = ['NIB', 'NPWP', 'NAMA', 'STATUS_NIB']
    miss_oss = df_oss[mand_oss].isnull().mean().mean() * 100
    miss_ceisa = df_ceisa[mand_ceisa].isnull().mean().mean() * 100
    miss_golden = df_golden[mand_golden].isnull().mean().mean() * 100

    nib_oss, npwp_oss, kp_oss = format_validity_pct(df_oss, 'NIB', 'NPWP_PERSEROAN', 'KODE_POS_PERSEROAN')
    nib_ceisa, npwp_ceisa, kp_ceisa = format_validity_pct(df_ceisa, 'NIB', 'NPWP', 'KODE_POS')
    nib_golden, npwp_golden, kp_golden = format_validity_pct(df_golden, 'NIB', 'NPWP', 'KODE_POS')

    n_dummy_nib = int((df_golden['NIB'].astype(str) == DUMMY_NIB).sum())
    n_out_of_sync = int(to_bool(df_golden['IS_OUT_OF_SYNC']).sum())
    n_stale = int(to_bool(df_golden['IS_STALE']).sum())
    n_high_sync_lag = int(to_bool(df_golden['HIGH_SYNC_LAG']).sum())
    n_logical_conflict = int(to_bool(df_golden['IS_LOGICAL_CONFLICT_NIPER']).sum())

    source_counts = df_golden['SOURCE'].value_counts()
    match_type_counts = df_golden['MATCH_TYPE'].value_counts()
    n_conflicts_dist = df_golden['N_CONFLICTS'].value_counts().sort_index()

    cp_match_counts = candidate_pairs['match_type'].value_counts()
    dc_source_counts = duplicate_cluster['source'].value_counts()
    n_clusters = duplicate_cluster['cluster_id'].nunique()

    cl_field_counts = conflict_log['field'].value_counts()

    optional_fields = ['KELURAHAN', 'KODE_POS', 'NOMOR_TELPON', 'NAMA_SINGKATAN', 'NIPER', 'NOMOR_API']
    optional_missing = {f: df_golden[f].isna().mean() * 100 for f in optional_fields}

    dedup_row = audit_trail[(audit_trail['dataset'] == 'CEISA') & (audit_trail['operation'] == 'DEDUP_SNAPSHOT')].iloc[0]
    n_dedup_removed = int(dedup_row['n_affected'])

    nib_fix_oss = int(audit_trail[(audit_trail['dataset'] == 'OSS') & (audit_trail['operation'] == 'STANDARDIZE_NIB')]['n_affected'].iloc[0])
    nib_fix_ceisa = int(audit_trail[(audit_trail['dataset'] == 'CEISA') & (audit_trail['operation'] == 'STANDARDIZE_NIB')]['n_affected'].iloc[0])

    npwp_invalid_ceisa_before = int(dq_rules.query("dataset=='CEISA' and rule_id=='VAL-NPWP-FORMAT'")['n_fail'].iloc[0])
    npwp_invalid_golden = int(dq_rules.query("dataset=='GOLDEN' and rule_id=='VAL-NPWP-FORMAT'")['n_fail'].iloc[0])

    n_orphan_oss = int(source_counts.get('OSS_ONLY', 0))
    n_orphan_ceisa = int(source_counts.get('CEISA_ONLY', 0))

    stale_oss_before = int(dq_rules.query("dataset=='OSS' and rule_id=='TIME-IS_STALE'")['n_fail'].iloc[0])
    sync_lag_ceisa_before = int(dq_rules.query("dataset=='CEISA' and rule_id=='TIME-HIGH_SYNC_LAG'")['n_fail'].iloc[0])
    kat_niper_fail = int(dq_rules.query("dataset=='CEISA' and rule_id=='CONS-KATEGORI-NIPER'")['n_fail'].iloc[0])

    flag_reason_counts = flagged['FLAG_REASON'].value_counts()
    flag_pct_ceisa = len(flagged) / n_ceisa * 100

    n_ceisa_clean = int(dq_rules.query("dataset=='CEISA' and rule_id=='COMP-NIB'")['total'].iloc[0])

    overall_oss = float(dq_scorecard.loc[dq_scorecard['dimension'] == 'Overall', 'oss_score'].iloc[0])
    overall_ceisa = float(dq_scorecard.loc[dq_scorecard['dimension'] == 'Overall', 'ceisa_score'].iloc[0])
    overall_golden = float(dq_scorecard.loc[dq_scorecard['dimension'] == 'Overall', 'golden_score'].iloc[0])

    print("Membangun grafik interaktif (Plotly)...")

    # -----------------------------------------------------------------
    # Figures
    # -----------------------------------------------------------------
    def add(fig):
        return fig_html(fig)

    # 1. Jumlah record OSS vs CEISA vs Golden
    fig_counts = go.Figure()
    fig_counts.add_bar(x=['OSS', 'CEISA', 'Total Sebelum (OSS+CEISA)', 'Golden Record'],
                        y=[n_oss, n_ceisa, n_before, n_golden],
                        marker_color=[C['oss'], C['ceisa'], C['neutral'], C['golden']],
                        text=[f"{v:,}" for v in [n_oss, n_ceisa, n_before, n_golden]], textposition='outside')
    fig_counts.update_layout(title='Jumlah Record: Sumber vs Golden Record', yaxis_title='Jumlah Record')
    fig_record_counts = add(fig_counts)

    # 2. Match type distribusi (candidate_pairs)
    fig_mt = go.Figure(data=[go.Pie(labels=cp_match_counts.index, values=cp_match_counts.values, hole=.45,
                                     marker_colors=[C['golden'], C['ceisa'], C['warn']])])
    fig_mt.update_layout(title=f'Distribusi Tipe Match — candidate_pairs.csv (total {len(candidate_pairs):,} pasangan)')
    fig_match_type = add(fig_mt)

    # 3. Duplicate cluster per sumber
    fig_dc = go.Figure()
    fig_dc.add_bar(x=dc_source_counts.index, y=dc_source_counts.values,
                    marker_color=[C['oss'], C['ceisa']],
                    text=[f"{v:,}" for v in dc_source_counts.values], textposition='outside')
    fig_dc.update_layout(title=f'Record Terlibat Duplicate Cluster per Sumber ({n_clusters:,} cluster)',
                          yaxis_title='Jumlah Record')
    fig_duplicate_cluster = add(fig_dc)

    # 4. Conflict log per field
    fig_cl = go.Figure()
    fig_cl.add_bar(x=cl_field_counts.index, y=cl_field_counts.values, marker_color=C['warn'],
                    text=[f"{v:,}" for v in cl_field_counts.values], textposition='outside')
    fig_cl.update_layout(title=f'Field yang Paling Sering Konflik OSS vs CEISA (total {len(conflict_log):,} konflik, semua dimenangkan OSS)',
                          yaxis_title='Jumlah Konflik')
    fig_conflict_field = add(fig_cl)

    # 5. SOURCE & MATCH_TYPE golden record
    fig_src = go.Figure()
    fig_src.add_trace(go.Pie(labels=source_counts.index, values=source_counts.values, hole=.45, name='SOURCE',
                              domain={'x': [0, 0.48]}, marker_colors=[C['golden'], C['oss'], C['ceisa']]))
    fig_src.add_trace(go.Pie(labels=match_type_counts.index, values=match_type_counts.values, hole=.45, name='MATCH_TYPE',
                              domain={'x': [0.52, 1]}, marker_colors=[C['golden'], C['warn'], C['ceisa']]))
    fig_src.update_layout(title='Komposisi Golden Record — SOURCE (kiri) vs MATCH_TYPE (kanan)',
                           annotations=[dict(text='SOURCE', x=0.18, y=-0.15, showarrow=False),
                                         dict(text='MATCH_TYPE', x=0.82, y=-0.15, showarrow=False)])
    fig_golden_source = add(fig_src)

    # 6. N_CONFLICTS distribusi
    fig_nc = go.Figure()
    fig_nc.add_bar(x=[str(i) for i in n_conflicts_dist.index], y=n_conflicts_dist.values, marker_color=C['oss'],
                    text=[f"{v:,}" for v in n_conflicts_dist.values], textposition='outside')
    fig_nc.update_layout(title='Distribusi N_CONFLICTS per Record (jumlah field yang konflik antar sumber)',
                          xaxis_title='N_CONFLICTS', yaxis_title='Jumlah Record')
    fig_n_conflicts = add(fig_nc)

    # 7. Provenance — sumber per field (stacked %)
    prov_pct = (provenance_log.groupby('field')['source']
                .value_counts(normalize=True).unstack(fill_value=0) * 100)
    prov_pct = prov_pct.reindex(columns=['OSS', 'CEISA', 'TIDAK ADA (kosong)'], fill_value=0)
    prov_pct = prov_pct.sort_values('OSS', ascending=False)
    fig_prov = go.Figure()
    fig_prov.add_bar(x=prov_pct.index, y=prov_pct['OSS'], name='OSS', marker_color=C['oss'])
    fig_prov.add_bar(x=prov_pct.index, y=prov_pct['CEISA'], name='CEISA', marker_color=C['ceisa'])
    fig_prov.add_bar(x=prov_pct.index, y=prov_pct['TIDAK ADA (kosong)'], name='Tidak ada / kosong', marker_color=C['neutral'])
    fig_prov.update_layout(barmode='stack', title='Provenance — Sumber Nilai per Field di Golden Record (%)',
                            yaxis_title='% Record', xaxis_tickangle=-45)
    fig_provenance = add(fig_prov)

    # 8. Risk flags
    risk_labels = ['IS_OUT_OF_SYNC<br>(Sync Conflict)', 'IS_STALE<br>(Data Usang)',
                    'HIGH_SYNC_LAG<br>(Sync &gt; 30 hari)', 'IS_LOGICAL_CONFLICT_NIPER<br>(Ekspor vs NIPER)',
                    'NIB Dummy<br>(Belum Terverifikasi)']
    risk_values = [n_out_of_sync, n_stale, n_high_sync_lag, n_logical_conflict, n_dummy_nib]
    risk_pct = [v / n_golden * 100 for v in risk_values]
    fig_risk = go.Figure()
    fig_risk.add_bar(x=risk_labels, y=risk_values, marker_color=C['bad'],
                      text=[f"{v:,} ({p:.1f}%)" for v, p in zip(risk_values, risk_pct)], textposition='outside')
    fig_risk.update_layout(title=f'Record Golden Record dengan Flag Risiko (dari {n_golden:,} total)',
                            yaxis_title='Jumlah Record')
    fig_risk_flags = add(fig_risk)

    # 9. Missing optional fields
    fig_om = go.Figure()
    fig_om.add_bar(x=list(optional_missing.keys()), y=list(optional_missing.values()), marker_color=C['neutral'],
                    text=[f"{v:.1f}%" for v in optional_missing.values()], textposition='outside')
    fig_om.update_layout(title='Field Opsional Kosong di Golden Record (%)', yaxis_title='% Record Kosong')
    fig_optional_missing = add(fig_om)

    # 10. DQ Scorecard — full comparison (+ Akurasi placeholder untuk presentasi)
    dqs = dq_scorecard[dq_scorecard['dimension'] != 'Overall']
    accuracy_row = pd.DataFrame([{
        'dimension': 'Akurasi (Accuracy)*',
        'oss_score': 100.0, 'ceisa_score': 100.0, 'golden_score': 100.0,
        'delta_vs_oss': 0.0, 'delta_vs_ceisa': 0.0,
    }])
    dqs_display = pd.concat([dqs, accuracy_row], ignore_index=True)
    overall_row = dq_scorecard[dq_scorecard['dimension'] == 'Overall']
    dqs_table_display = pd.concat([dqs_display, overall_row], ignore_index=True)

    fig_dq = go.Figure()
    fig_dq.add_bar(x=dqs_display['dimension'], y=dqs_display['oss_score'], name='OSS (Before)', marker_color=C['oss'])
    fig_dq.add_bar(x=dqs_display['dimension'], y=dqs_display['ceisa_score'], name='CEISA (Before)', marker_color=C['ceisa'])
    fig_dq.add_bar(x=dqs_display['dimension'], y=dqs_display['golden_score'], name='Golden Record (After)', marker_color=C['golden'])
    fig_dq.update_layout(barmode='group', title='Skor Data Quality per Dimensi DMBOK: Before vs After (+ Akurasi*)',
                          yaxis_title='Skor (%)', yaxis_range=[0, 105])
    fig_dq_scorecard = add(fig_dq)

    # 11. Format validity before vs after
    fig_val = go.Figure()
    fields_v = ['NIB (13 digit)', 'NPWP (format baku)', 'KODE_POS (5 digit)']
    fig_val.add_bar(x=fields_v, y=[nib_oss, npwp_oss, kp_oss], name='OSS (Before)', marker_color=C['oss'])
    fig_val.add_bar(x=fields_v, y=[nib_ceisa, npwp_ceisa, kp_ceisa], name='CEISA (Before)', marker_color=C['ceisa'])
    fig_val.add_bar(x=fields_v, y=[nib_golden, npwp_golden, kp_golden], name='Golden Record (After)', marker_color=C['golden'])
    fig_val.update_layout(barmode='group', title='Format Validity (%): Before vs After', yaxis_title='% Valid', yaxis_range=[0, 105])
    fig_format_validity = add(fig_val)

    print("Merender dokumentasi (docs/) & menyusun konten slide...")

    # -----------------------------------------------------------------
    # Render docs
    # -----------------------------------------------------------------
    doc03 = split_doc_sections('docs/03_logika_bisnis_presentasi.md')
    # doc03[0] = intro, doc03[1..20] = sections 1..20
    sec = {i: render_md(doc03[i]) for i in range(1, 21)}

    doc00 = render_md((DOCS / '00_overview.md').read_text(encoding='utf-8'))
    doc01 = render_md((DOCS / '01_data_dictionary.md').read_text(encoding='utf-8'))
    doc02 = render_md((DOCS / '02_business_rules.md').read_text(encoding='utf-8'))

    # -----------------------------------------------------------------
    # Slides
    # -----------------------------------------------------------------
    slides = []

    # --- Cover -----------------------------------------------------------
    cover_body = f"""
    <div class="cover">
      <div class="slide-kicker">Laporan Akhir &amp; Bahan Presentasi — Mini Project MDM 2026</div>
      <h1>Master Data Management — Importir &amp; Eksportir Nasional (DJBC)</h1>
      <div class="subtitle">Single Importer and Exporter View · Kelompok 5 ·
        Dibuat otomatis dari hasil pipeline Tahap 0&ndash;6 pada {datetime.now().strftime('%d %B %Y, %H:%M')}</div>
      <div class="kpi-row">
        {kpi_card(f"{n_before:,}", "Total Record Sumber", f"OSS {n_oss:,} + CEISA {n_ceisa:,}", C['neutral'])}
        {kpi_card(f"{n_golden:,}", "Golden Record", f"Reduksi {pct_reduction:.1f}%", C['golden'])}
        {kpi_card(f"{overall_oss:.1f} / {overall_ceisa:.1f}", "DQ Overall Before", "OSS / CEISA (skala 0-100)", C['oss'])}
        {kpi_card(f"{overall_golden:.1f}", "DQ Overall After", "Golden Record (skala 0-100)", C['golden'])}
        {kpi_card(f"{n_out_of_sync:,}", "Sync Conflict", "IS_OUT_OF_SYNC = True", C['bad'])}
        {kpi_card(f"{n_high_sync_lag:,}", "High Sync Lag", "TGL_SYNC_OSS &gt; 30 hari", C['warn'])}
      </div>
      <p style="max-width:780px;margin:0 auto;color:#64748B;">
        Laporan ini merangkum seluruh hasil pipeline MDM (Tahap 0&ndash;6: Simulasi, Profiling, Cleansing &amp;
        Standardization, Duplicate Detection &amp; Matching, Golden Record &amp; Survivorship, Data Quality
        Monitoring, dan Profiling Dashboard), digabungkan dengan dokumentasi logika bisnis di
        <code>docs/</code>. Setiap section di bawah ini didesain sebagai satu "slide" sehingga laporan ini
        dapat langsung digunakan sebagai bahan presentasi (cetak ke PDF untuk mode slide).
      </p>
      <p style="max-width:780px;margin:8px auto 0 auto;color:#64748B;">
        Laporan ini juga berfungsi sebagai <b>prototipe/proof-of-concept</b>: secara teknis, MDM untuk data
        importir &amp; eksportir DJBC sangat memungkinkan untuk dibangun (lihat juga §21, integrasi API) &mdash;
        tantangan utamanya justru pada <b>tata kelola (governance)</b>, lihat §22.
      </p>
      <p style="max-width:780px;margin:18px auto 0 auto;color:#94A3B8;font-size:.85rem;">
        <b>Anggota Kelompok 5:</b> Yola Dafwita Chandra &middot; Farkhan Wisnu Wardhono &middot;
        Vivin Nur Aziza &middot; Beryl Cholif Arrahman Rahardjo &middot; Dananjaya Ricky Setiaji
      </p>
    </div>
    """
    slides.append(dict(id='cover', cls='', kicker='', title='', body=cover_body))

    # --- §1, §2 (konteks bisnis, langsung dari docs) ----------------------
    slides.append(dict(id='s01', kicker='Konteks Bisnis', title='1. Judul &amp; Konteks Project', body=sec[1]))
    s02_body = sec[2] + """
    <p>Framing "OSS = System of Record untuk field legalitas, CEISA = data mart dengan snapshot historis" di atas
    konsisten dengan latar belakang bisnis project (lihat <code>docs/00_overview.md</code> &sect; Latar Belakang
    Bisnis, direproduksi lengkap di Lampiran C).</p>
    """
    slides.append(dict(id='s02', kicker='Konteks Bisnis', title='2. Latar Belakang Bisnis', body=s02_body))

    # --- §3 + data record counts -------------------------------------------
    s03_body = sec[3] + f"""
    <h2>📊 Data Aktual</h2>
    {fig_record_counts}
    <p>CEISA bersifat <i>data mart</i>: dari {n_ceisa:,} baris raw, terdapat <b>{n_dedup_removed:,}</b> baris
    snapshot duplikat (lihat audit trail Tahap 2) yang dibuang sebelum proses matching, sehingga jumlah CEISA
    yang dipakai untuk matching adalah <b>{n_ceisa_clean:,}</b> baris (1 baris/NIB).</p>
    """
    slides.append(dict(id='s03', kicker='Sumber Data', title='3. Dua Sumber Data: Peran &amp; Karakteristik', body=s03_body))

    slides.append(dict(id='s04', kicker='Sumber Data', title='4. Skema Data — OSS (16 Kolom)', body=sec[4]))
    slides.append(dict(id='s05', kicker='Sumber Data', title='5. Skema Data — CEISA (16 Kolom)', body=sec[5]))
    slides.append(dict(id='s06', kicker='Sumber Data', title='6. Pemetaan Kolom OSS ↔ CEISA', body=sec[6]))

    # --- §7 ------------------------------------------------------------------
    s07_body = sec[7] + """
    <div class="callout">
      <p>📊 <b>Catatan Implementasi</b></p>
      <p>Pipeline Tahap 5 (<code>dq_scorecard.csv</code>) secara aktual mengukur <b>5 dimensi</b>: 4 dimensi inti
      di atas (Completeness, Validity, Uniqueness, Timeliness) <b>+ Consistency</b> &mdash; yaitu kesesuaian nilai
      antar-field/antar-sumber (mis. <code>STATUS_NIB</code>, <code>KATEGORI</code>, <code>FLAG_EKSPOR</code> antara
      OSS dan CEISA), yang menjadi dasar flag <code>IS_OUT_OF_SYNC</code> dan
      <code>IS_LOGICAL_CONFLICT_NIPER</code> pada Golden Record (lihat §10).</p>
      <p>Laporan ini menambahkan dimensi ke-6, <b>Akurasi</b>, sebagai <i>placeholder</i> (lihat §17) agar gambaran
      6 dimensi DMBOK lebih lengkap secara presentasi. Penambahan ini murni di level laporan &mdash;
      <code>dq_scorecard.csv</code> dan <code>docs/02_business_rules.md</code> tidak diubah.</p>
    </div>
    """
    slides.append(dict(id='s07', kicker='Strategi &amp; Aturan MDM', title='7. Fondasi Logika Bisnis: 4 Dimensi Kualitas Data (DMBOK)', body=s07_body))

    # --- §8 + matching data ----------------------------------------------------
    n_fuzzy = int(cp_match_counts.get('FUZZY_REVIEW', 0))
    fuzzy_examples = candidate_pairs[candidate_pairs['match_type'] == 'FUZZY_REVIEW'].copy()
    nama_oss_map = df_oss.drop_duplicates('NIB').set_index(df_oss.drop_duplicates('NIB')['NIB'].astype(str))['NAMA_PERSEROAN']
    nama_ceisa_map = df_ceisa.drop_duplicates('ID_PERUSAHAAN').set_index('ID_PERUSAHAAN')['NAMA_PERUSAHAAN']
    fuzzy_examples['NAMA_OSS'] = fuzzy_examples['NIB_OSS'].astype(str).map(nama_oss_map)
    fuzzy_examples['NAMA_CEISA'] = fuzzy_examples['ID_PERUSAHAAN_CEISA'].map(nama_ceisa_map)
    fuzzy_display = fuzzy_examples[['NIB_OSS', 'NAMA_OSS', 'ID_PERUSAHAAN_CEISA', 'NAMA_CEISA', 'similarity_score']].rename(
        columns={'NIB_OSS': 'NIB (OSS)', 'NAMA_OSS': 'Nama Perusahaan (OSS)',
                 'ID_PERUSAHAAN_CEISA': 'ID Perusahaan (CEISA)', 'NAMA_CEISA': 'Nama Perusahaan (CEISA)',
                 'similarity_score': 'Skor Komposit'})
    fuzzy_table = simple_table(fuzzy_display, fmt={'Skor Komposit': '{:.2f}'.format})
    s08_body = sec[8] + f"""
    <h2>📊 Data Aktual</h2>
    <div class="grid-2">
      <div>{fig_match_type}</div>
      <div>{fig_duplicate_cluster}</div>
    </div>
    <p>Dari {len(candidate_pairs):,} kandidat pasangan, mayoritas ({cp_match_counts.get('EXACT_NIB', 0):,})
    cocok via <b>EXACT_NIB</b>, {cp_match_counts.get('EXACT_NPWP', 0):,} via <b>EXACT_NPWP</b> (skenario "NIB Typo" —
    lihat anomali #4), dan {n_fuzzy:,} via <b>FUZZY_REVIEW</b> (skenario "Fuzzy Identity / Typo Nama" — anomali #3).</p>
    <p>Skor komposit fuzzy = 0.20&times;kemiripan NPWP + 0.50&times;kemiripan Nama (Jaro-Winkler &amp; token-set-ratio)
    + 0.30&times;kemiripan Alamat. Klasifikasi: skor &ge; 0.85 &rarr; <code>FUZZY_MATCH</code> (otomatis valid),
    skor &lt; 0.70 &rarr; <code>NON_MATCH</code> (dibuang), dan <b>0.70 &le; skor &lt; 0.85 &rarr; <code>FUZZY_REVIEW</code></b>
    (perlu review manual — {n_fuzzy:,} contoh berikut). Kolom nama perusahaan ditampilkan untuk memperjelas seberapa
    mirip/berbeda kandidat di "zona abu-abu" ini:</p>
    {fuzzy_table}
    <p>Selain matching antar sumber, ditemukan <b>{n_clusters:,} cluster duplikat internal</b>
    ({dc_source_counts.get('OSS', 0):,} record OSS + {dc_source_counts.get('CEISA', 0):,} record CEISA) —
    skenario "Duplicate Entry" (anomali #9).</p>
    """
    slides.append(dict(id='s08', kicker='Strategi &amp; Aturan MDM', title='8. Strategi Matching Bertingkat (Tahap 3)', body=s08_body))

    # --- §9 + survivorship data --------------------------------------------------
    n_status_conflict = int(cl_field_counts.get('STATUS_NIB', 0))
    n_npwp_conflict = int(cl_field_counts.get('NPWP_VALUE', 0))
    n_nama_conflict = int(cl_field_counts.get('NAMA', 0))
    n_eksp_niper_conflict = int(cl_field_counts.get('FLAG_EKSPOR_vs_NIPER', 0))
    s09_body = sec[9] + f"""
    <h2>📊 Data Aktual</h2>
    <div class="grid-2">
      <div>{fig_golden_source}</div>
      <div>{fig_n_conflicts}</div>
    </div>
    {fig_provenance}
    <p>Grafik provenance di atas <b>membuktikan aturan survivorship benar-benar diterapkan</b>: field legalitas
    (NIB, NPWP, NAMA, ALAMAT, STATUS_NIB, JENIS_PERSEROAN, FLAG_IMPOR/EKSPOR) didominasi sumber <b>OSS</b>,
    sementara field operasional (KODE_KANTOR, NOMOR_TELPON, KATEGORI, NIPER, NOMOR_API, TGL_SYNC_OSS,
    TGL_TERBIT_NIB, ID_PERUSAHAAN) didominasi sumber <b>CEISA</b> — persis sesuai prinsip §9.A.</p>
    {fig_conflict_field}
    <p>Total <b>{len(conflict_log):,} konflik nilai</b> tercatat di <code>conflict_log.csv</code>, seluruhnya
    dimenangkan OSS sesuai aturan survivorship: <b>{n_npwp_conflict:,}</b> konflik format NPWP,
    <b>{n_nama_conflict:,}</b> konflik penulisan NAMA, <b>{n_status_conflict:,}</b> konflik STATUS_NIB
    (Sync Conflict), dan <b>{n_eksp_niper_conflict:,}</b> Logical Conflict FLAG_EKSPOR vs NIPER.</p>
    """
    slides.append(dict(id='s09', kicker='Strategi &amp; Aturan MDM', title='9. Aturan Survivorship — "Siapa yang Menang?" (Tahap 4)', body=s09_body))

    # --- §10 + risk flags ---------------------------------------------------------
    s10_body = sec[10] + f"""
    <h2>📊 Data Aktual</h2>
    {fig_risk_flags}
    <ul>
      <li><b>IS_OUT_OF_SYNC</b>: {n_out_of_sync:,} record ({n_out_of_sync/n_golden*100:.2f}%) — STATUS_NIB OSS
        berbeda dari CEISA, berisiko transaksi kepabeanan dengan entitas yang status legalnya sudah berubah.</li>
      <li><b>IS_STALE</b>: {n_stale:,} record ({n_stale/n_golden*100:.2f}%) — TGL_PERUBAHAN_NIB OSS sudah &gt;1 tahun.</li>
      <li><b>HIGH_SYNC_LAG</b>: {n_high_sync_lag:,} record ({n_high_sync_lag/n_golden*100:.2f}%) — TGL_SYNC_OSS
        CEISA sudah &gt;30 hari, melampaui SLA sinkronisasi bulanan.</li>
    </ul>
    """
    slides.append(dict(id='s10', kicker='Strategi &amp; Aturan MDM', title='10. Penandaan Konflik &amp; Risiko (Flag Bisnis Utama)', body=s10_body))

    # --- §11 + tabel anomali aktual --------------------------------------------------
    anomaly_rows = [
        ("1", "Stale Data", "TGL_PERUBAHAN_NIB OSS sangat lama (&gt;1 tahun)", "Tahap 1/5 — IS_STALE",
         f"OSS (before): {stale_oss_before:,}/{n_oss:,} ({stale_oss_before/n_oss*100:.1f}%) gagal rule TIME-IS_STALE.<br>"
         f"Golden (after): {n_stale:,}/{n_golden:,} ({n_stale/n_golden*100:.1f}%) record IS_STALE=True."),
        ("2", "Inconsistent NPWP", "CEISA mengirim NPWP tanpa titik/strip", "Tahap 2 — Standardisasi format",
         f"CEISA (before): {npwp_invalid_ceisa_before:,}/{n_ceisa:,} ({npwp_invalid_ceisa_before/n_ceisa*100:.1f}%) gagal VAL-NPWP-FORMAT "
         f"({npwp_ceisa:.1f}% valid).<br>Golden (after): {npwp_invalid_golden:,}/{n_golden:,} "
         f"({npwp_golden:.2f}% valid) — {len(flagged):,} record CEISA ({flag_pct_ceisa:.1f}%) ditandai "
         f"<code>FORMAT_NPWP_TIDAK_BAKU</code> di flagged_for_review.csv."),
        ("3", "Fuzzy Identity / Typo Nama", 'Nama perusahaan mirip, ada "(TYPO)" / beda spasi', "Tahap 3 — Fuzzy Matching",
         f"{n_fuzzy:,} pasangan candidate_pairs masuk zona <code>FUZZY_REVIEW</code> (skor komposit 0.70&ndash;0.85, "
         f"lihat §8), dan {int(match_type_counts.get('FUZZY_REVIEW', 0)):,} di antaranya jadi Golden Record."),
        ("4", "NIB Typo", "Satu digit NIB beda antar sistem", "Tahap 2/3 — Standardisasi &amp; Secondary Matching via NPWP",
         f"Tahap 2: {nib_fix_oss:,} NIB OSS ({nib_fix_oss/n_oss*100:.1f}%) &amp; {nib_fix_ceisa:,} NIB CEISA "
         f"({nib_fix_ceisa/n_ceisa*100:.1f}%) di-standardisasi (pad/trim 13 digit).<br>"
         f"Tahap 3: {int(cp_match_counts.get('EXACT_NPWP', 0)):,} pasangan ter-resolve via EXACT_NPWP "
         f"({int(match_type_counts.get('EXACT_NPWP', 0)):,} masuk Golden Record)."),
        ("5", "Sync Conflict", "STATUS_NIB OSS = DICABUT, CEISA = AKTIF (atau sebaliknya)", "Tahap 4 — Survivorship (IS_OUT_OF_SYNC)",
         f"{n_out_of_sync:,}/{n_golden:,} record ({n_out_of_sync/n_golden*100:.2f}%) IS_OUT_OF_SYNC=True — "
         f"semua di-resolve dengan STATUS_NIB OSS (latest-date-wins)."),
        ("6", "Logical Conflict", "FLAG_EKSPOR=N tapi NIPER terisi di CEISA", "Tahap 5 — Consistency Validation",
         f"Golden (after): {n_logical_conflict:,}/{n_golden:,} ({n_logical_conflict/n_golden*100:.2f}%) "
         f"IS_LOGICAL_CONFLICT_NIPER=True (rule CONS-FLAG_EKSPOR-NIPER).<br>"
         f"CEISA (before): {kat_niper_fail:,}/{n_ceisa:,} ({kat_niper_fail/n_ceisa*100:.1f}%) gagal "
         f"rule CONS-KATEGORI-NIPER."),
        ("7", "Orphan Records", "Record hanya ada di OSS atau hanya di CEISA", "Tahap 4 — Golden Record SOURCE tracking",
         f"SOURCE=OSS_ONLY: {n_orphan_oss:,} record ({n_orphan_oss/n_golden*100:.1f}%); "
         f"SOURCE=CEISA_ONLY: {n_orphan_ceisa:,} record ({n_orphan_ceisa/n_golden*100:.1f}%). "
         f"Total orphan {n_orphan_oss+n_orphan_ceisa:,} dari {n_golden:,} Golden Record."),
        ("8", "Missing Optional Field", "KELURAHAN, KODE_POS, NOMOR_TELPON, dll kosong", "Tahap 1/5 — Completeness Profiling",
         "; ".join(f"{f}: {v:.1f}%" for f, v in optional_missing.items()) + " (semua field opsional, dibiarkan kosong sesuai aturan §14)."),
        ("9", "Duplicate Entry", "Satu perusahaan muncul 2x di sistem yang sama", "Tahap 3 — Duplicate Clustering",
         f"{n_clusters:,} cluster duplikat internal: {dc_source_counts.get('OSS', 0):,} record OSS "
         f"(rule UNIQ-NIB OSS n_fail={dup_oss:,}) + {dc_source_counts.get('CEISA', 0):,} record CEISA."),
        ("10", "Data Mart Snapshot Duplicate", "NIB sama, &gt;1 baris CEISA dengan TGL_SYNC_OSS berbeda", "Tahap 2 — Pre-matching Dedup (DEDUP_SNAPSHOT)",
         f"{n_dedup_removed:,} baris snapshot CEISA dibuang ({dedup_row['pct_affected']:.2f}% dari raw CEISA), "
         f"snapshot dengan TGL_SYNC_OSS terbaru dipertahankan."),
    ]
    rows_html = "".join(
        f"<tr><td>{n}</td><td><b>{name}</b></td><td>{ex}</td><td>{tahap}</td><td>{found}</td></tr>"
        for n, name, ex, tahap, found in anomaly_rows
    )
    s11_body = sec[11] + f"""
    <h2>📊 Hasil Aktual per Anomali</h2>
    <div class="table-wrap"><table class="dtable">
    <thead><tr><th>#</th><th>Anomali</th><th>Contoh Skenario</th><th>Diuji di</th><th>Temuan Aktual dari Pipeline</th></tr></thead>
    <tbody>{rows_html}</tbody></table></div>
    """
    slides.append(dict(id='s11', kicker='Strategi &amp; Aturan MDM', title='11. Anomali Bisnis yang Disimulasikan &amp; Diuji', body=s11_body))

    # --- §12 Pipeline overview -----------------------------------------------------
    slides.append(dict(id='s12', kicker='Pipeline &amp; Hasil per Tahap', title='12. Pipeline End-to-End — Tahap 0 s.d. 6', body=sec[12]))

    # --- §13 Tahap 1 Profiling ----------------------------------------------------
    s13_body = sec[13] + f"""
    <h2>📊 Hasil Profiling Awal (Before)</h2>
    <h3>OSS</h3>{dq_rules_table(dq_rules[dq_rules['dataset'] == 'OSS'])}
    <h3>CEISA</h3>{dq_rules_table(dq_rules[dq_rules['dataset'] == 'CEISA'])}
    <h3>Visualisasi Missing Value &amp; DQ Scorecard Awal</h3>
    <div class="grid-2">
      <img src="{b64_image('reports/missing_value_oss.png')}" alt="Missing value OSS">
      <img src="{b64_image('reports/missing_value_ceisa.png')}" alt="Missing value CEISA">
    </div>
    <div class="grid-2" style="margin-top:14px;">
      <img src="{b64_image('reports/dq_scorecard_oss_master.png')}" alt="DQ Scorecard OSS">
      <img src="{b64_image('reports/dq_scorecard_ceisa_operational.png')}" alt="DQ Scorecard CEISA">
    </div>
    <p class="linklist">Laporan profiling lengkap (ydata-profiling):
      <a href="profiling_before_oss.html" target="_blank">profiling_before_oss.html</a> ·
      <a href="profiling_before_ceisa.html" target="_blank">profiling_before_ceisa.html</a> ·
      <a href="profiling_before.html" target="_blank">profiling_before.html</a></p>
    """
    slides.append(dict(id='s13', kicker='Pipeline &amp; Hasil per Tahap', title='13. Tahap 1 — Profiling (Before)', body=s13_body))

    # --- §14 Tahap 2 Cleansing -----------------------------------------------------
    s14_body = sec[14] + f"""
    <h2>📊 Audit Trail Cleansing (audit_trail.csv)</h2>
    <h3>OSS</h3>{audit_table(audit_trail[audit_trail['dataset'] == 'OSS'])}
    <h3>CEISA</h3>{audit_table(audit_trail[audit_trail['dataset'] == 'CEISA'])}
    <h2>Flagged for Review (flagged_for_review.csv)</h2>
    <p>{len(flagged):,} record ({flag_pct_ceisa:.1f}% dari CEISA) ditandai untuk review manual,
    seluruhnya karena <code>FORMAT_NPWP_TIDAK_BAKU</code> — NPWP tidak diformat ulang paksa karena
    formatnya tidak baku (anomali "Inconsistent NPWP" sengaja dipertahankan sebagai catatan kualitas data,
    sesuai <code>docs/02_business_rules.md</code> §1).</p>
    <p>Hasil akhir cleansing: <code>oss_cleaned.csv</code> ({n_oss:,} baris), <code>ceisa_cleaned.csv</code>
    ({n_ceisa_clean:,} baris setelah dedup snapshot, dari {n_ceisa:,} baris raw), digabung menjadi
    <code>dataset_clean.csv</code> ({len(dataset_clean):,} baris) sebagai input Tahap 3.</p>
    """
    slides.append(dict(id='s14', kicker='Pipeline &amp; Hasil per Tahap', title='14. Tahap 2 — Cleansing &amp; Standardization', body=s14_body))

    # --- §15 Tahap 3 Matching --------------------------------------------------------
    s15_body = sec[15] + f"""
    <h2>📊 Hasil Matching &amp; Duplicate Detection</h2>
    <div class="grid-2">
      <div>{fig_match_type}</div>
      <img src="{b64_image('reports/composite_score_distribution.png')}" alt="Distribusi composite similarity score">
    </div>
    <p>Dari {n_oss:,} record OSS dan {n_ceisa_clean:,} record CEISA (setelah dedup snapshot), terbentuk
    <b>{int(match_type_counts.get('EXACT_NIB',0)) + int(match_type_counts.get('EXACT_NPWP',0)) + int(match_type_counts.get('FUZZY_REVIEW',0)):,}
    pasangan matched</b> menjadi {len(df_golden[df_golden['SOURCE']=='MATCHED']):,} Golden Record SOURCE=MATCHED,
    sisanya {n_orphan_oss + n_orphan_ceisa:,} menjadi orphan (OSS_ONLY/CEISA_ONLY).
    Selain itu, {n_clusters:,} cluster duplikat internal terdeteksi via clustering graph
    (<code>duplicate_cluster.csv</code>, {len(duplicate_cluster):,} baris).</p>
    """
    slides.append(dict(id='s15', kicker='Pipeline &amp; Hasil per Tahap', title='15. Tahap 3 — Duplicate Detection &amp; Matching', body=s15_body))

    # --- §16 Tahap 4 Golden Record ----------------------------------------------------
    s16_body = sec[16] + f"""
    <h2>📊 Komposisi &amp; Provenance Golden Record</h2>
    <div class="grid-2">
      <div>{fig_golden_source}</div>
      <div>{fig_n_conflicts}</div>
    </div>
    {fig_provenance}
    <p><code>golden_record.csv</code> berisi {n_golden:,} baris x {df_golden.shape[1]} kolom.
    Setiap keputusan sumber per field tercatat di <code>provenance_log.csv</code>
    ({len(provenance_log):,} baris). {n_conflicts_dist.get(0,0):,} record ({n_conflicts_dist.get(0,0)/n_golden*100:.1f}%)
    tidak punya konflik nilai sama sekali (N_CONFLICTS=0), sedangkan
    {(n_conflicts_dist.drop(0, errors='ignore')).sum():,} record memiliki minimal 1 konflik yang
    diresolusi otomatis sesuai aturan survivorship.</p>
    """
    slides.append(dict(id='s16', kicker='Pipeline &amp; Hasil per Tahap', title='16. Tahap 4 — Golden Record &amp; Survivorship', body=s16_body))

    # --- §17 Tahap 5 DQ Monitoring -----------------------------------------------------
    s17_body = sec[17] + f"""
    <h2>📊 DQ Scorecard — OSS vs CEISA vs Golden Record</h2>
    {fig_dq_scorecard}
    <div class="table-wrap">{simple_table(dqs_table_display, fmt={c: '{:.2f}'.format for c in dqs_table_display.columns if c != 'dimension'})}</div>
    <div class="callout">
      <p><b>Catatan atas DQ Scorecard di atas:</b></p>
      <ul>
        <li><b>* Akurasi (Accuracy)</b> adalah baris <b>placeholder</b> (nilai dummy 100 untuk OSS/CEISA/Golden, delta=0).
        DMBOK mendefinisikan Akurasi sebagai kesesuaian data dengan <i>sumber kebenaran eksternal</i>
        (mis. data resmi instansi terkait), yang tidak tersedia sebagai dataset rujukan dalam project ini —
        lihat juga §7. Baris <b>"Overall"</b> tetap dihitung dari 5 dimensi nyata
        (Completeness, Validity, Uniqueness, Consistency, Timeliness); Akurasi tidak memengaruhi skor Overall.</li>
        <li><b>Mengapa skor Consistency Golden Record
        ({dq_scorecard.loc[dq_scorecard['dimension']=='Consistency','golden_score'].iloc[0]:.2f}) lebih rendah dari OSS
        ({dq_scorecard.loc[dq_scorecard['dimension']=='Consistency','oss_score'].iloc[0]:.2f}) / CEISA
        ({dq_scorecard.loc[dq_scorecard['dimension']=='Consistency','ceisa_score'].iloc[0]:.2f})?</b>
        Ini <b>bukan regresi kualitas</b>. Skor Consistency Golden adalah rata-rata dari <b>4 rule</b>, sedangkan
        OSS dan CEISA masing-masing hanya dievaluasi dengan <b>1 rule</b>:
          <ul>
            <li>OSS &mdash; <code>CONS-FLAG_IMPOR-JENIS_API</code>: 100.00%</li>
            <li>CEISA &mdash; <code>CONS-KATEGORI-NIPER</code>: 94.76%</li>
            <li>Golden &mdash; <code>CONS-FLAG_IMPOR-JENIS_API</code>: 100.00%, <code>CONS-KATEGORI-NIPER</code>: 95.45%,
            <code>CONS-STATUS_NIB-SYNC</code>: 96.66%, <code>CONS-FLAG_EKSPOR-NIPER</code>: 63.83%</li>
          </ul>
        Dua rule terakhir (<code>CONS-STATUS_NIB-SYNC</code> dan <code>CONS-FLAG_EKSPOR-NIPER</code>) adalah rule
        <b>cross-source</b> yang baru bisa dihitung <i>setelah</i> OSS dan CEISA digabung menjadi Golden Record —
        keduanya menjadi dasar flag <code>IS_OUT_OF_SYNC</code> dan <code>IS_LOGICAL_CONFLICT_NIPER</code>
        (lihat anomali #6 &mdash; "Logical Conflict" di §11). Skor Consistency Golden yang lebih rendah berarti MDM
        <b>berhasil mengungkap konflik antar-sumber</b> yang sebelumnya tersembunyi saat OSS dan CEISA dievaluasi
        terpisah &mdash; ini justru salah satu nilai tambah utama proses MDM.</li>
        <li><b>Mengapa masih ada {npwp_invalid_golden:,} record Golden yang gagal <code>VAL-NPWP-FORMAT</code>
        (dan {int(dq_rules.query("dataset=='GOLDEN' and rule_id=='VAL-NIB-DUMMY'")['n_fail'].iloc[0]):,} gagal
        <code>VAL-NIB-DUMMY</code>)?</b>
        Seluruh {npwp_invalid_golden:,} kegagalan <code>VAL-NPWP-FORMAT</code> berasal dari record
        <code>SOURCE='CEISA_ONLY'</code> (orphan, <code>NIB</code> dummy <code>'{DUMMY_NIB}'</code>) dengan nilai
        NPWP yang <b>jumlah digitnya salah</b> (bukan sekadar kurang titik/strip), terdeteksi via flag
        <code>IS_NPWP_INVALID</code> pada <code>ceisa_cleaned.csv</code>. Demikian pula, kegagalan
        <code>VAL-NIB-DUMMY</code> didominasi oleh record OSS/CEISA bernilai <code>NIB</code> dummy yang tidak
        memiliki pasangan match. Keduanya adalah <b>masalah data sumber</b> yang tidak bisa diperbaiki lewat
        transformasi/standardisasi &mdash; perlu verifikasi ke sistem asal (terkait dengan keterbatasan Akurasi
        di atas).</li>
      </ul>
    </div>
    <h2>Quality Gate Report</h2>
    {gate_table(quality_gate)}
    <h2>DQ Rule Results — Golden Record</h2>
    {dq_rules_table(dq_rules[dq_rules['dataset'] == 'GOLDEN'])}
    """
    slides.append(dict(id='s17', kicker='Pipeline &amp; Hasil per Tahap', title='17. Tahap 5 — Data Quality Monitoring (DQ Scorecard)', body=s17_body))

    # --- §18 Tahap 6 Profiling Dashboard -------------------------------------------------
    comparison_rows = pd.DataFrame([
        {'Metrik': 'Jumlah Record', 'OSS (Before)': f"{n_oss:,}", 'CEISA (Before)': f"{n_ceisa:,}", 'Golden Record (After)': f"{n_golden:,}"},
        {'Metrik': 'Missing Value % (field wajib)', 'OSS (Before)': f"{miss_oss:.2f}%", 'CEISA (Before)': f"{miss_ceisa:.2f}%", 'Golden Record (After)': f"{miss_golden:.2f}%"},
        {'Metrik': 'Validity NIB (13 digit)', 'OSS (Before)': f"{nib_oss:.2f}%", 'CEISA (Before)': f"{nib_ceisa:.2f}%", 'Golden Record (After)': f"{nib_golden:.2f}%"},
        {'Metrik': 'Validity NPWP (format baku)', 'OSS (Before)': f"{npwp_oss:.2f}%", 'CEISA (Before)': f"{npwp_ceisa:.2f}%", 'Golden Record (After)': f"{npwp_golden:.2f}%"},
        {'Metrik': 'Validity KODE_POS (5 digit)', 'OSS (Before)': f"{kp_oss:.2f}%", 'CEISA (Before)': f"{kp_ceisa:.2f}%", 'Golden Record (After)': f"{kp_golden:.2f}%"},
        {'Metrik': 'Duplikasi NIB (baris duplikat)', 'OSS (Before)': f"{dup_oss:,} ({dup_oss/n_oss*100:.2f}%)",
         'CEISA (Before)': f"{dup_ceisa:,} ({dup_ceisa/n_ceisa*100:.2f}%, data mart - expected)",
         'Golden Record (After)': f"{dup_golden_real:,} ({dup_golden_real/n_golden*100:.2f}%, di luar NIB dummy)"},
    ])
    s18_body = sec[18] + f"""
    <h2>📊 Perbandingan Before vs After</h2>
    {simple_table(comparison_rows)}
    {fig_format_validity}
    <div class="grid-2">
      <img src="{b64_image('reports/dq_comparison_oss.png')}" alt="DQ Comparison OSS">
      <img src="{b64_image('reports/dq_comparison_ceisa.png')}" alt="DQ Comparison CEISA">
    </div>
    <p class="linklist">Laporan profiling lengkap Golden Record (ydata-profiling):
      <a href="profiling_after.html" target="_blank">profiling_after.html</a></p>
    """
    slides.append(dict(id='s18', kicker='Pipeline &amp; Hasil per Tahap', title='18. Tahap 6 — Profiling Dashboard (After)', body=s18_body))

    # --- §19 Insight & Rekomendasi ---------------------------------------------------------
    s19_body = f"""
    <div class="callout good">
      <b>1. Konsolidasi data</b> berhasil mengurangi <b>{n_before - n_golden:,} baris ({pct_reduction:.1f}%)</b>
      dari total record OSS+CEISA ({n_before:,}) menjadi <b>{n_golden:,} Golden Record</b>
      (Single Importer/Exporter View), terutama berkat dedup snapshot CEISA ({n_dedup_removed:,} baris)
      dan exact/fuzzy matching NIB &amp; NPWP (Tahap 3).
    </div>
    <div class="callout good">
      <b>2. Uniqueness NIB meningkat signifikan</b>: dari {(1 - dup_oss/n_oss)*100:.2f}% (OSS) /
      {(1 - dup_ceisa/n_ceisa)*100:.2f}% (CEISA) menjadi <b>{(1 - dup_golden_real/n_golden)*100:.2f}%</b>
      di Golden Record (di luar {n_dummy_nib:,} NIB dummy/invalid yang sengaja dipertahankan sebagai
      catatan kualitas data untuk verifikasi ulang).
    </div>
    <div class="callout warn">
      <b>3. Risiko kualitas data yang masih perlu ditindaklanjuti</b> meski sudah ada Golden Record:
      <ul>
        <li><b>{n_dummy_nib:,}</b> record memiliki NIB dummy/tidak valid (<code>{DUMMY_NIB}</code>) &rarr;
            perlu verifikasi ulang ke OSS.</li>
        <li><b>{n_out_of_sync:,}</b> record berstatus <i>Sync Conflict</i> (STATUS_NIB OSS vs CEISA berbeda,
            IS_OUT_OF_SYNC=True).</li>
        <li><b>{n_logical_conflict:,}</b> record memiliki <i>Logical Conflict</i> antara FLAG_EKSPOR dan NIPER.</li>
        <li><b>{n_high_sync_lag:,}</b> record memiliki HIGH_SYNC_LAG (CEISA belum sinkron &gt;30 hari dari OSS).</li>
        <li><b>{n_stale:,}</b> record IS_STALE (TGL_PERUBAHAN_NIB OSS &gt;1 tahun) &rarr; kemungkinan
            perusahaan tidak aktif/data usang.</li>
        <li><b>{n_orphan_oss + n_orphan_ceisa:,}</b> record orphan (hanya ada di salah satu sistem) &rarr;
            {n_orphan_oss:,} OSS_ONLY, {n_orphan_ceisa:,} CEISA_ONLY.</li>
      </ul>
    </div>
    <h2>Rekomendasi Implementasi Data Governance &amp; MDM</h2>
    <ol class="reco-list">
      <li><b>Auto-update CEISA via trigger dari OSS.</b> Saat ini {n_high_sync_lag:,}
        ({n_high_sync_lag/n_golden*100:.1f}%) record melampaui SLA sync 30 hari — begitu ada perubahan di OSS
        (status NIB, alamat, dll.), langsung dorong ke CEISA agar sync lag minimal.</li>
      <li><b>Validasi format di titik input</b> (OSS &amp; CEISA) untuk NIB/NPWP/KODE_POS, agar anomali format
        (mis. {len(flagged):,} record NPWP tidak baku di CEISA) tidak terbawa ke data mart/operasional.</li>
      <li><b>Jadikan OSS sebagai System of Record</b> untuk legalitas (NIB, status badan hukum, status NIB) —
        sudah diterapkan di survivorship Golden Record (terbukti dari grafik provenance §9), perlu
        diformalkan sebagai kebijakan data governance resmi DJBC.</li>
      <li><b>Bangun proses rekonsiliasi berkala</b> (mis. bulanan) untuk menyelesaikan
        {n_out_of_sync:,} Sync Conflict dan {n_logical_conflict:,} Logical Conflict (FLAG_EKSPOR vs NIPER)
        yang terdeteksi di Golden Record, serta verifikasi ulang {n_dummy_nib:,} record dengan NIB dummy.</li>
      <li><b>Jadwalkan re-running pipeline</b> profiling &rarr; cleansing &rarr; matching &rarr; golden record
        &rarr; DQ monitoring secara periodik agar <code>dq_scorecard.csv</code> dan laporan ini selalu
        mencerminkan kondisi data terkini.</li>
    </ol>
    <div class="callout">
      Rekomendasi di atas bersifat <b>operasional/teknis</b> dalam scope pipeline ini. Untuk pembahasan
      <b>tata kelola (governance)</b> yang lebih strategis &mdash; termasuk pertanyaan "siapa yang
      bertanggung jawab atas data master ini secara berkelanjutan?" di level Kemenkeu maupun nasional &mdash;
      lihat §22 (Tata Kelola &amp; Rekomendasi Strategis).
    </div>
    """
    slides.append(dict(id='s19', kicker='Insight &amp; Penutup', title='19. Insight Bisnis &amp; Rekomendasi Data Governance', body=s19_body))

    # --- §20 Ringkasan + Deliverables checklist ------------------------------------------------
    deliverables = [
        ("Dataset Bersih (oss_cleaned, ceisa_cleaned, dataset_clean)", "data/processed/*.csv",
         f"{len(dataset_clean):,} baris (OSS {n_oss:,} + CEISA {n_ceisa:,})"),
        ("Audit Trail", "reports/audit_trail.csv", f"{len(audit_trail):,} entri operasi cleansing"),
        ("Candidate Pairs (hasil matching)", "reports/candidate_pairs.csv", f"{len(candidate_pairs):,} pasangan kandidat"),
        ("Duplicate Cluster", "reports/duplicate_cluster.csv", f"{len(duplicate_cluster):,} baris, {n_clusters:,} cluster"),
        ("Golden Record", "data/golden/golden_record.csv", f"{n_golden:,} baris x {df_golden.shape[1]} kolom"),
        ("Provenance Log", "reports/provenance_log.csv", f"{len(provenance_log):,} baris"),
        ("Conflict Log", "reports/conflict_log.csv", f"{len(conflict_log):,} baris"),
        ("DQ Scorecard (Tahap 5)", "reports/dq_scorecard.csv", "5 dimensi DMBOK x 3 dataset (OSS/CEISA/Golden)"),
        ("DQ Rule Results &amp; Quality Gate", "reports/dq_rule_results.csv, reports/quality_gate_report.csv",
         f"{len(dq_rules):,} rule, {len(quality_gate):,} gate"),
        ("Profiling Before (ydata-profiling)", "reports/profiling_before_oss.html, profiling_before_ceisa.html", "Tahap 1"),
        ("Profiling After (ydata-profiling)", "reports/profiling_after.html", "Tahap 6"),
        ("Bahan Presentasi (PPT/PDF, maks. 10 slide)",
         "reports/final_report.html, reports/final_report/index.html",
         "Dua format: reports/final_report.html (single-page, cetak ke PDF/PPT untuk presentasi) dan "
         "reports/final_report/index.html (multi-page, untuk browsing interaktif per section)"),
    ]
    deliv_rows = "".join(
        f"<tr><td>{name}</td><td><code>{path}</code></td><td class='deliv-status-ok'>✅ {info}</td></tr>"
        for name, path, info in deliverables
    )
    s20_body = sec[20] + f"""
    <h2>📋 Checklist Deliverable (sesuai Pedoman Mini Project, di luar notebook .ipynb)</h2>
    <div class="table-wrap"><table class="dtable">
    <thead><tr><th>Deliverable</th><th>File</th><th>Status &amp; Ringkasan</th></tr></thead>
    <tbody>{deliv_rows}</tbody></table></div>
    <p style="margin-top:18px;color:#94A3B8;font-size:.85rem;">
      Laporan dibuat otomatis oleh <code>Source/step11_final_report.py</code> &mdash;
      {datetime.now().strftime('%d %B %Y, %H:%M:%S')}.
    </p>
    """
    slides.append(dict(id='s20', kicker='Insight &amp; Penutup', title='20. Ringkasan Penutup &amp; Deliverables', body=s20_body))

    # --- §21 (Bonus) Integrasi API ---------------------------------------------------------
    s21_body = f"""
    <div class="callout warn">
      <b>Bagian opsional/bonus.</b> Section ini <b>di luar Tahap 1&ndash;6</b> PEDOMAN Mini Project
      (lihat <code>docs/00_overview.md</code> baris 39, Lampiran C) dan <b>tidak termasuk deliverable wajib</b>.
      Konten di bawah ini bersifat <b>konseptual/ilustratif</b>, diadaptasi dari Lab 5
      &mdash; "API &amp; Data Integration" (<code>reference/Data Profiling (1).ipynb</code>), untuk
      menggambarkan bagaimana <code>golden_record.csv</code> ({n_golden:,} baris) dapat dikonsumsi
      sebagai layanan oleh sistem lain.
    </div>

    <h2>Arsitektur: MDM Hub sebagai Service</h2>
    <pre><code>data/golden/golden_record.csv ──▶ MDM Hub API (FastAPI) ──▶ ngrok tunnel (publik, demo)
                                            │
                ┌───────────────┬───────────┼───────────────┬───────────────┐
                ▼               ▼           ▼               ▼               ▼
            GET /        GET /api/mdm/  GET /api/mdm/  GET /api/mdm/   POST /api/mdm/
          (health check)   {{nib}}        search?nama=   list?page=      validate
                                            │
                                            ▼
                                     GET /api/mdm/stats</code></pre>
    <p>Pola arsitektur ini sejalan dengan Lab 5 (FastAPI + Pydantic schema + ngrok tunnel + nest_asyncio),
    hanya domainnya diganti dari "Wajib Pajak / NPWP" menjadi "Importir-Eksportir / NIB" sesuai
    <code>golden_record.csv</code> hasil Tahap 4.</p>

    <h2>Endpoint MDM Hub API (konseptual)</h2>
    <div class="table-wrap"><table class="dtable">
    <thead><tr><th>Endpoint</th><th>Method</th><th>Deskripsi</th></tr></thead>
    <tbody>
      <tr><td><code>/</code></td><td>GET</td><td>Health check &mdash; status service &amp; jumlah total record.</td></tr>
      <tr><td><code>/api/mdm/{{nib}}</code></td><td>GET</td><td>Ambil 1 Golden Record berdasarkan <code>NIB</code>.</td></tr>
      <tr><td><code>/api/mdm/search?nama=</code></td><td>GET</td><td>Cari Golden Record berdasarkan kemiripan <code>NAMA</code> (fuzzy search).</td></tr>
      <tr><td><code>/api/mdm/list?page=</code></td><td>GET</td><td>Daftar Golden Record dengan pagination.</td></tr>
      <tr><td><code>/api/mdm/validate</code></td><td>POST</td><td>Validasi data NIB/NPWP/Nama dari sistem eksternal terhadap MDM (mis. cek <code>VAL-NIB-DUMMY</code>/<code>VAL-NPWP-FORMAT</code>) sebelum diproses.</td></tr>
      <tr><td><code>/api/mdm/stats</code></td><td>GET</td><td>Statistik ringkas MDM Hub (jumlah record, distribusi <code>SOURCE</code>, skor DQ Scorecard &mdash; lihat §17).</td></tr>
    </tbody></table></div>

    <h2>Sinkronisasi Event-Driven (Webhook)</h2>
    <p><code>docs/00_overview.md</code> baris 24 mencatat sebagai <i>next step</i> di luar scope:
    "idealnya ada mekanisme <b>auto-update CEISA via trigger</b> dari OSS agar sync lag ke depan minim"
    &mdash; terkait langsung dengan <code>HIGH_SYNC_LAG</code>/<code>IS_OUT_OF_SYNC</code> yang dibahas di
    §19 (Insight &amp; Rekomendasi). Lab 5 mengilustrasikan mekanisme ini lewat pola
    <code>WebhookEvent</code> &amp; <code>WebhookBroker</code>:</p>
    <pre><code>WebhookEvent(
    event_id      = "evt-00123",
    event_type    = "DATA_UPDATED",   # DATA_CREATED | DATA_UPDATED | DATA_DELETED
    entity_type   = "GOLDEN_RECORD",
    nib           = "1234567890123",
    changed_fields= ["STATUS_NIB"],
    old_values    = {{"STATUS_NIB": "AKTIF"}},
    new_values    = {{"STATUS_NIB": "NONAKTIF"}},
    timestamp     = "2026-06-12T10:00:00",
    source        = "OSS",
)</code></pre>
    <p>Konsepnya: setiap kali <code>STATUS_NIB</code>/<code>KATEGORI</code>/<code>FLAG_EKSPOR</code> berubah
    di OSS, MDM Hub <b>publish</b> event <code>WebhookEvent</code> ke subscriber (mis. CEISA, sistem internal
    DJBC lain) sehingga update OSS ter-propagate tanpa menunggu sync batch &mdash; mengurangi
    <code>HIGH_SYNC_LAG</code> ({n_high_sync_lag:,} record saat ini, lihat §10/§19) dan
    <code>IS_OUT_OF_SYNC</code> ({n_out_of_sync:,} record).</p>

    <div class="callout warn">
      <b>Disclaimer:</b> Seluruh API, endpoint, dan webhook di atas adalah <b>desain konseptual</b> untuk
      ilustrasi &mdash; tidak diimplementasikan/dijalankan sebagai bagian dari deliverable Tahap 0&ndash;6
      project ini.
    </div>
    """
    slides.append(dict(id='s21', kicker='Insight &amp; Penutup', title='21. (Bonus) Integrasi API — MDM Hub sebagai Service', body=s21_body))

    # --- §22 Tata Kelola & Rekomendasi Strategis -------------------------------------------------
    s22_body = """
    <div class="callout">
      <p>Pipeline Tahap 0&ndash;6 dan integrasi API konseptual di §21 menunjukkan bahwa secara <b>teknis</b>,
      MDM untuk data importir/eksportir DJBC <b>sangat memungkinkan</b> untuk dibangun. Laporan ini pada
      dasarnya adalah <b>prototipe / proof-of-concept</b> &mdash; tantangan sesungguhnya bukan pada teknologi,
      melainkan pada <b>tata kelola (governance)</b>.</p>
      <p>Dan semakin <b>makro</b> skala penerapannya, semakin besar dampaknya: pada level unit setingkat
      <b>Eselon I</b> saja, MDM yang dapat dipercaya sudah berpengaruh signifikan terhadap kualitas
      keputusan &mdash; apalagi bila diterapkan lintas Kementerian/Lembaga atau di level <b>nasional</b>.</p>
    </div>

    <h2>Temuan Utama</h2>
    <p>Masalah data di DJBC <b>bukan semata teknikal</b> &mdash; akarnya adalah <b>tidak ada pihak yang secara
    eksplisit bertanggung jawab</b> memastikan data master tetap akurat sepanjang waktu.</p>

    <h2>Level Makro &mdash; Kajian Master Data Nasional</h2>
    <ul>
      <li>Untuk entitas <b>individu</b>, <b>NIK (Dukcapil)</b> sudah berfungsi sebagai <i>de facto</i> master
      identifier penduduk.</li>
      <li>Untuk entitas <b>badan usaha</b>, belum ada master yang jelas &mdash; ada tiga kandidat yang perlu
      dikaji lebih lanjut:
        <ul>
          <li><b>NIB</b> (OSS-BKPM, domain perizinan)</li>
          <li><b>NPWP</b> (DJP, domain perpajakan)</li>
          <li>Data <b>Ditjen AHU</b> (domain legalitas badan hukum)</li>
        </ul>
      </li>
      <li>Masing-masing kandidat punya kekuatan di domainnya masing-masing, tapi juga keterbatasan &mdash;
      kajian lebih lanjut diperlukan untuk menentukan mana yang paling layak menjadi sumber master
      lintas K/L.</li>
      <li><b>Satu Data Indonesia</b> (Perpres 39/2019) idealnya diperkuat mandatnya ke layer ini.</li>
    </ul>

    <h2>Level Mikro &mdash; Gap Konkret di DJBC</h2>
    <ul>
      <li>Data badan usaha dari OSS dapat diduga sudah divalidasi saat <i>onboarding</i>, tetapi tidak ada
      kewajiban pembaruan berkala &mdash; data berpotensi <i>stale</i> tanpa ada yang mendeteksi (lihat
      <code>HIGH_SYNC_LAG</code>/<code>IS_OUT_OF_SYNC</code> di §10/§19).</li>
      <li>Kementerian Keuangan perlu membangun <b>MDM internal</b> yang mengambil data dari sumber master
      nasional dan memperkayanya dengan atribut domain Kemenkeu (mis. status kepabeanan, NIPER, dsb).</li>
    </ul>

    <h2>Prinsip Tata Kelola (DMBOK Ch. 10)</h2>
    <ul>
      <li>Setiap atribut di MDM harus punya <b>Data Steward</b> yang bertanggung jawab atas definisi,
      standar kualitas, dan eskalasi anomali &mdash; bukan cukup ada sistemnya, harus ada <b>orangnya</b>.</li>
      <li><b>MDM bukan soal tools-nya; MDM adalah disiplin.</b> Tanpa governance yang jelas, sistem
      secanggih apapun tidak menjamin kebenaran datanya.</li>
    </ul>
    """
    slides.append(dict(id='s22', kicker='Insight &amp; Penutup', title='22. Tata Kelola &amp; Rekomendasi Strategis', body=s22_body))

    # --- Lampiran -----------------------------------------------------------------------------
    slides.append(dict(id='appendix-dict', kicker='Lampiran', title='Lampiran A — Data Dictionary (OSS vs CEISA)', body=doc01))
    slides.append(dict(id='appendix-rules', kicker='Lampiran', title='Lampiran B — Business Rules &amp; DMBOK', body=doc02))
    slides.append(dict(id='appendix-overview', kicker='Lampiran', title='Lampiran C — Overview &amp; Roadmap Project', body=doc00))

    # --- Lampiran D — Data Viewer (CSV) -------------------------------------------------------
    csv_data_json = json.dumps({key: csv_b64(path) for key, (label, path) in CSV_FILES.items()})
    csv_meta_json = json.dumps({key: dict(label=label, path=path) for key, (label, path) in CSV_FILES.items()})
    csv_tabs_html = "".join(
        f'<button class="csv-tab{" active" if key == "golden_record" else ""}" data-key="{key}" '
        f'onclick="switchCsv(\'{key}\')">{label}</button>'
        for key, (label, path) in CSV_FILES.items()
    )

    appendix_data_head = f"""
    <div class="callout">
      <p>Tabel di bawah ini menampilkan <b>seluruh isi CSV hasil pipeline</b> (Tahap 0&ndash;6) secara interaktif
      &mdash; data di-<i>embed</i> langsung di dalam file HTML ini (base64), sehingga bisa dibuka dan diperiksa
      tanpa software tambahan. Gunakan tab di bawah untuk memilih dataset, kotak pencarian untuk memfilter
      seluruh kolom, dan tombol navigasi untuk pindah halaman ({CSV_PAGE_SIZE} baris/halaman).</p>
    </div>
    <div class="csv-tabs">{csv_tabs_html}</div>
    <div class="csv-viewer-toolbar">
      <div id="csv-viewer-info"></div>
      <input type="text" id="csv-search-input" class="csv-search" placeholder="Cari di semua kolom..." oninput="csvSearch(this.value)">
    </div>
    <div id="csv-viewer-table"></div>
    """

    appendix_data_js = """
    <script>
    const CSV_DATA = __CSV_DATA__;
    const CSV_META = __CSV_META__;
    const CSV_PAGE_SIZE = __CSV_PAGE_SIZE__;
    const csvCache = {};
    let csvActive = null;

    function b64ToUtf8(b64) {
      const bin = atob(b64);
      const bytes = new Uint8Array(bin.length);
      for (let i = 0; i < bin.length; i++) bytes[i] = bin.charCodeAt(i);
      return new TextDecoder('utf-8').decode(bytes);
    }

    function parseCsv(text) {
      const rows = [];
      let row = [], field = '', inQuotes = false;
      for (let i = 0; i < text.length; i++) {
        const c = text[i];
        if (inQuotes) {
          if (c === '"') {
            if (text[i + 1] === '"') { field += '"'; i++; }
            else inQuotes = false;
          } else field += c;
        } else if (c === '"') inQuotes = true;
        else if (c === ',') { row.push(field); field = ''; }
        else if (c === '\\n') { row.push(field); rows.push(row); row = []; field = ''; }
        else if (c === '\\r') { /* ignore */ }
        else field += c;
      }
      if (field.length || row.length) { row.push(field); rows.push(row); }
      while (rows.length && rows[rows.length - 1].length === 1 && rows[rows.length - 1][0] === '') rows.pop();
      return rows;
    }

    function escapeHtml(s) {
      return String(s).replace(/&/g, '&amp;').replace(/</g, '&lt;').replace(/>/g, '&gt;');
    }

    function loadCsv(key) {
      if (!csvCache[key]) {
        const rows = parseCsv(b64ToUtf8(CSV_DATA[key]));
        csvCache[key] = { header: rows[0] || [], data: rows.slice(1), filtered: rows.slice(1), page: 0, search: '' };
      }
      return csvCache[key];
    }

    function switchCsv(key) {
      csvActive = key;
      document.querySelectorAll('.csv-tab').forEach(function (b) {
        b.classList.toggle('active', b.dataset.key === key);
      });
      const state = loadCsv(key);
      document.getElementById('csv-search-input').value = state.search || '';
      renderCsvTable();
    }

    function renderCsvTable() {
      const key = csvActive;
      const state = csvCache[key];
      const meta = CSV_META[key];
      const totalPages = Math.max(1, Math.ceil(state.filtered.length / CSV_PAGE_SIZE));
      state.page = Math.min(state.page, totalPages - 1);
      const start = state.page * CSV_PAGE_SIZE;
      const pageRows = state.filtered.slice(start, start + CSV_PAGE_SIZE);

      document.getElementById('csv-viewer-info').innerHTML =
        '<b>' + escapeHtml(meta.label) + '</b> &mdash; <code>' + escapeHtml(meta.path) + '</code> (' +
        state.data.length.toLocaleString('id-ID') + ' baris x ' + state.header.length + ' kolom)';

      let html = '<div class="table-wrap"><table class="dtable"><thead><tr>';
      state.header.forEach(function (h) { html += '<th>' + escapeHtml(h) + '</th>'; });
      html += '</tr></thead><tbody>';
      pageRows.forEach(function (r) {
        html += '<tr>';
        r.forEach(function (c) { html += '<td>' + escapeHtml(c) + '</td>'; });
        html += '</tr>';
      });
      html += '</tbody></table></div>';
      html += '<div class="csv-pagination">' +
        '<button onclick="csvPage(-1)"' + (state.page <= 0 ? ' disabled' : '') + '>&larr; Sebelumnya</button>' +
        '<span>Halaman ' + (state.page + 1) + ' / ' + totalPages + ' &middot; ' +
        state.filtered.length.toLocaleString('id-ID') + ' baris' + (state.search ? ' (hasil pencarian)' : '') + '</span>' +
        '<button onclick="csvPage(1)"' + (state.page >= totalPages - 1 ? ' disabled' : '') + '>Berikutnya &rarr;</button>' +
        '</div>';
      document.getElementById('csv-viewer-table').innerHTML = html;
    }

    function csvSearch(term) {
      const state = csvCache[csvActive];
      state.search = term;
      const q = term.trim().toLowerCase();
      state.filtered = q ? state.data.filter(function (r) {
        return r.some(function (c) { return c.toLowerCase().includes(q); });
      }) : state.data;
      state.page = 0;
      renderCsvTable();
    }

    function csvPage(delta) {
      const state = csvCache[csvActive];
      state.page += delta;
      renderCsvTable();
    }

    document.addEventListener('DOMContentLoaded', function () { switchCsv('golden_record'); });
    </script>
    """
    appendix_data_js = (appendix_data_js
                         .replace('__CSV_DATA__', csv_data_json)
                         .replace('__CSV_META__', csv_meta_json)
                         .replace('__CSV_PAGE_SIZE__', str(CSV_PAGE_SIZE)))

    appendix_data_body = appendix_data_head + appendix_data_js
    slides.append(dict(id='appendix-data', kicker='Lampiran', title='Lampiran D — Data Viewer (CSV Hasil)', body=appendix_data_body))

    # -----------------------------------------------------------------
    # Navigasi
    # -----------------------------------------------------------------
    nav_groups = [
        dict(label='Ringkasan', links=[('cover', 'Cover & KPI')]),
        dict(label='Konteks Bisnis', links=[('s01', '1. Judul & Konteks'), ('s02', '2. Latar Belakang')]),
        dict(label='Sumber Data', links=[('s03', '3. OSS vs CEISA'), ('s04', '4. Skema OSS'),
                                          ('s05', '5. Skema CEISA'), ('s06', '6. Pemetaan Kolom')]),
        dict(label='Strategi & Aturan MDM', links=[('s07', '7. Dimensi DMBOK'), ('s08', '8. Strategi Matching'),
                                                     ('s09', '9. Survivorship'), ('s10', '10. Konflik & Risiko'),
                                                     ('s11', '11. Anomali Bisnis')]),
        dict(label='Pipeline & Hasil', links=[('s12', '12. Pipeline E2E'), ('s13', '13. Tahap 1 Profiling'),
                                               ('s14', '14. Tahap 2 Cleansing'), ('s15', '15. Tahap 3 Matching'),
                                               ('s16', '16. Tahap 4 Golden Record'), ('s17', '17. Tahap 5 DQ Monitoring'),
                                               ('s18', '18. Tahap 6 Profiling Dashboard')]),
        dict(label='Insight & Penutup', links=[('s19', '19. Insight & Rekomendasi'), ('s20', '20. Penutup & Deliverables'),
                                                ('s21', '21. (Bonus) Integrasi API'), ('s22', '22. Tata Kelola & Rekomendasi')]),
        dict(label='Lampiran', links=[('appendix-dict', 'A. Data Dictionary'), ('appendix-rules', 'B. Business Rules'),
                                       ('appendix-overview', 'C. Overview & Roadmap'), ('appendix-data', 'D. Data Viewer (CSV)')]),
    ]

    report_title = 'Laporan Akhir MDM DJBC — Single Importer & Exporter View'

    print("Menulis reports/final_report.html (single-page)...")
    html = TEMPLATE.render(title=report_title, css=CSS, plotlyjs=PLOTLY_JS,
                            nav_groups=nav_groups, slides=slides)
    out_path = REPORTS / 'final_report.html'
    out_path.write_text(html, encoding='utf-8')
    print(f"✅ Laporan akhir (single-page) berhasil dibuat: {out_path} ({out_path.stat().st_size/1024:.1f} KB)")

    # -----------------------------------------------------------------
    # Output multi-page: reports/final_report/
    # -----------------------------------------------------------------
    print("Menulis reports/final_report/ (multi-page)...")
    site_dir = REPORTS / 'final_report'
    assets_dir = site_dir / 'assets'
    assets_dir.mkdir(parents=True, exist_ok=True)
    (assets_dir / 'style.css').write_text(CSS, encoding='utf-8')
    (assets_dir / 'plotly.min.js').write_text(PLOTLY_JS, encoding='utf-8')

    flat_nav = [(sid, stitle) for group in nav_groups for sid, stitle in group['links']]

    def nav_href(sid):
        return 'index.html' if sid == 'cover' else f'{sid}.html'

    index_html = INDEX_TEMPLATE.render(title=report_title, nav_groups=nav_groups, cover_body=cover_body, active_id='cover')
    (site_dir / 'index.html').write_text(index_html, encoding='utf-8')

    for slide in slides:
        if slide['id'] == 'cover':
            continue
        idx = next(i for i, (sid, _) in enumerate(flat_nav) if sid == slide['id'])
        prev_link = flat_nav[idx - 1] if idx > 0 else None
        next_link = flat_nav[idx + 1] if idx < len(flat_nav) - 1 else None
        prev = dict(href=nav_href(prev_link[0]), title=prev_link[1]) if prev_link else None
        next_ = dict(href=nav_href(next_link[0]), title=next_link[1]) if next_link else None
        page_html = PAGE_TEMPLATE.render(title=report_title, nav_groups=nav_groups, slide=slide,
                                          active_id=slide['id'], prev=prev, next=next_)
        (site_dir / f"{slide['id']}.html").write_text(page_html, encoding='utf-8')

    n_pages = len(slides)  # cover -> index.html + (n_pages - 1) section pages
    print(f"✅ Laporan akhir (multi-page) berhasil dibuat: {site_dir} ({n_pages} halaman)")
